# Come nasce un outfit — la pipeline intera, in un notebook

Dalla pagina del negozio all'outfit finito: **chiamate al sito → valutazione del capo →
vettori → clusterizzazione → punteggi → accoppiamento → pool**.

Il notebook **contiene tutto il codice**, non lo importa: ogni funzione della pipeline è qui,
verbatim, nella cella della fase a cui appartiene, con sopra l'intestazione del file da cui
viene (`# ═══ nuvolari_generation_pipeline/scoring.py ═══`). Eseguendolo dall'alto si
definisce l'intera pipeline; niente `import` di moduli del progetto.

Per ogni funzione c'è lo stesso blocco:

- **Input** — che cosa riceve, e da dove arriva
- **Processo** — che cosa fa, e perché così
- **Output** — che cosa restituisce, e chi lo consuma dopo

## Niente parte da sola

Le tre cose che **costano** — soldi o traffico verso nuvolari.biz — sono qui dentro, complete e
pronte, ma dietro un interruttore che sta a `False`. Nessuna cella le fa partire per conto suo:
si accende quella che si vuole, e si riesegue la sua cella.

| interruttore | che cosa fa partire | costo |
|---|---|---|
| `ESEGUI_SCRAPING` | le chiamate a nuvolari.biz: pagine prodotto, foto, descrizioni | traffico sul sito del negozio |
| `ESEGUI_REVISIONE_VISIVA` | la classificazione delle foto con Gemini | a chiamata, sul tuo account |
| `ESEGUI_POOL` | la generazione dell'intero pool di outfit | ore di CPU |

Le valutazioni **ci sono già**: `vision_review` sta dentro ogni `metadata.json` del catalogo, e
i parquet e il pool sono sul disco. Il notebook li **legge**; rilanciare le chiamate serve solo
se il catalogo cambia o se si vuole rifare un giudizio.

## I percorsi

Tutti relativi alla radice del repository `Enrico2209/MyStylist` — nessun percorso assoluto:

```
nuvolari_generation_pipeline/     il codice, le due guide .md, questo notebook
nuvolari_db/                      i dati prodotti: parquet, pool, immagini degli outfit
nuvolari_full_organizzato/        il catalogo: una cartella per prodotto, foto + metadata.json
```

Da un clone fresco arrivano il codice e **il catalogo con dentro le revisioni visive già
pagate** (foto e `metadata.json`, via git-lfs). Non arrivano `features.parquet`,
`features_clustered.parquet` e `outfits_pool.jsonl`, che si rifanno dal catalogo con le fasi
3, 4 e 6 — senza rete e senza spendere niente.

## Le fasi

| # | fase | file | funzione d'ingresso | produce |
|---|---|---|---|---|
| 1 | chiamate al sito | `scrape_with_attributes.py` | `run_scrape_with_attributes` | foto + `metadata.json` per 2901 prodotti |
| 2 | valutazione testuale | `scrape_with_attributes.py` | `derive_attributes` | attributi + il flag `needs_vision_review` |
| 2 | valutazione visiva | `vision_review.py` + `guida_classificazione.md` | `revisiona` | il blocco `vision_review` in `metadata.json` |
| 3 | vettori del capo | `feature_engineering.py` | `build_feature_table` | `features.parquet` — colore Lab + vettore di stile |
| 4 | clusterizzazione | `clustering.py` | `run_clustering` | `features_clustered.parquet` — `style_cluster` |
| 5 | punteggio fra due capi | `scoring.py` | `score_pair` | `score(A,B)` ∈ [0,1] |
| 6 | accoppiamento | `outfit_generation.py` | `build_outfits` / `run_outfit_pipeline` | `outfits_pool.jsonl` |

La fase 7 (la foto dell'outfit indossato, `generate_outfit_images.py` con `generation_rules.md`)
sta a valle: è la resa dell'outfit, non la sua creazione, e resta fuori da qui.

> I numeri di questo notebook sono **calcolati eseguendo**, mai trascritti.
> `COME_NASCE_UN_OUTFIT.md` racconta lo stesso processo in prosa ma è rimasto indietro:
> descrive dieci tag di stile (oggi cinque), 44 cluster con il 47% di outlier (oggi 7 e l'1%)
> e 1762 outfit nel pool (oggi 67.564). Dove i due divergono vale il codice, che è questo.

In [1]:
# Le librerie di tutti i moduli della pipeline, unite.
import argparse, collections, hashlib, io, itertools, json, math, os, random, re, sys, time
import unicodedata
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from PIL import Image
from skimage.color import rgb2lab
from skimage.segmentation import flood
from sklearn.cluster import HDBSCAN, KMeans

from IPython.display import HTML, display

# google-genai serve solo alle chiamate a Gemini (Fase 2): se manca, tutto il
# resto del notebook funziona lo stesso.
try:
    from google import genai
    from google.genai import types
    GENAI = True
except ImportError:
    GENAI = False
    print("[i] google-genai non installato: la Fase 2 visiva resta leggibile ma non eseguibile")

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

In [2]:
# ── I percorsi, relativi alla radice del repository ──────────────────────────
# Si risale dalla cartella corrente finché non si trova la radice del repo, così
# il notebook funziona sia aperto da nuvolari_generation_pipeline/ sia dalla radice.
# La cartella del codice si chiama nuvolari_generation_pipeline/ nel repo del
# backend (Enrico2209/MyStylist) e pipeline/ in quello online
# (Enrico2209/mystylist_online): si accettano entrambi i nomi.
NOMI_CODICE = ("nuvolari_generation_pipeline", "pipeline")

def trova_radice(partenza: Path):
    for c in [partenza, *partenza.parents]:
        for nome in NOMI_CODICE:
            if (c / nome).is_dir():
                return c, c / nome
    return partenza, partenza

RADICE, CODICE = trova_radice(Path.cwd())
BASE     = RADICE / "nuvolari_db"                    # i dati prodotti (nei moduli si chiama BASE)
DATI     = BASE
CATALOGO = RADICE / "nuvolari_full_organizzato"      # le schede prodotto con le foto

FEATURES  = DATI / "features.parquet"
CLUSTERED = DATI / "features_clustered.parquet"
POOL      = DATI / "outfits_pool.jsonl"

print("radice del repo:", RADICE.name)
print("cartella del codice:", CODICE.name + "/")
for p in (CODICE, BASE, CATALOGO):
    print(f"  {p.relative_to(RADICE).as_posix() + '/':34s} {'presente' if p.exists() else 'manca'}")
for p in (FEATURES, CLUSTERED, POOL):
    stato = f"{p.stat().st_size/1e6:7.1f} MB" if p.exists() else "  manca — si rifà eseguendo la sua fase"
    print(f"  {p.relative_to(RADICE).as_posix():34s} {stato}")

if not CATALOGO.exists():
    print()
    print("Il catalogo non c'è in questo repository: sta in Enrico2209/MyStylist, versionato")
    print("con git-lfs (foto + metadata.json, revisioni visive comprese). Senza di lui le celle")
    print("di dimostrazione non hanno dati; il codice resta tutto leggibile ed eseguibile.")

radice del repo: nuvolari backend
cartella del codice: nuvolari_generation_pipeline/
  nuvolari_generation_pipeline/      presente
  nuvolari_db/                       presente
  nuvolari_full_organizzato/         presente
  nuvolari_db/features.parquet           1.2 MB
  nuvolari_db/features_clustered.parquet     1.3 MB
  nuvolari_db/outfits_pool.jsonl       246.5 MB


In [3]:
# ── Gli interruttori: tutto ciò che costa parte solo da qui ──────────────────
ESEGUI_SCRAPING          = False   # chiamate a nuvolari.biz (Fase 1)
ESEGUI_REVISIONE_VISIVA  = False   # chiamate a Gemini, a pagamento (Fase 2)
ESEGUI_FEATURES          = False   # ricalcolo dei vettori da tutte le foto (Fase 3, ~30 min, gratis)
ESEGUI_POOL              = False   # generazione dell'intero pool (Fase 6, ore, gratis)

for nome, valore in [("ESEGUI_SCRAPING", ESEGUI_SCRAPING),
                     ("ESEGUI_REVISIONE_VISIVA", ESEGUI_REVISIONE_VISIVA),
                     ("ESEGUI_FEATURES", ESEGUI_FEATURES),
                     ("ESEGUI_POOL", ESEGUI_POOL)]:
    print(f"  {nome:26s} {'ACCESO' if valore else 'spento'}")

  ESEGUI_SCRAPING            spento
  ESEGUI_REVISIONE_VISIVA    spento
  ESEGUI_FEATURES            spento
  ESEGUI_POOL                spento


In [4]:
# Un capo di esempio, lo stesso per tutto il notebook: lo si segue dalla scheda
# grezza fino agli outfit in cui finisce. Si legge dal catalogo, non dal parquet,
# così questa cella funziona anche su un clone appena scaricato.
ESEMPIO = "abbigliamento/camicie/2-pkt-camp-shirt-es3-112363920-soy-bean-soy-bean"

cartella = CATALOGO / ESEMPIO
meta     = json.loads((cartella / "metadata.json").read_text(encoding="utf-8"))

print(meta["title"])
print("  cartella :", ESEMPIO)
print("  foto     :", len([p for p in cartella.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")]))
print("  chiavi del metadata.json:", ", ".join(sorted(meta)))

CAMICIA UOMO LEE CAMP SHIRT SOY BEAN
  cartella : abbigliamento/camicie/2-pkt-camp-shirt-es3-112363920-soy-bean-soy-bean
  foto     : 3
  chiavi del metadata.json: abbinamenti_suggeriti, brand, brand_slug, color_hints, composition, currency, description_text, descrizione_completa, fit, formality_score, image_count, needs_vision_review, pattern, price, relpath, scheda_tecnica, season, sku, style_tags, title, url, vision_review


---
# 1. Le chiamate al sito — Fase 1

Il catalogo non esiste finché non lo si va a prendere. Questa fase apre nuvolari.biz, scopre
le categorie dal menu, gira le pagine di listino, apre **una volta sola** ogni pagina prodotto
e ne porta via foto, titolo, prezzo, composizione, vestibilità, descrizione e abbinamenti
consigliati. Il risultato è la struttura su cui vive tutto il resto:

```
nuvolari_full_organizzato/<categoria>/<sottocategoria>/<slug-prodotto>/
    2000003579068_1.jpg      ← le foto della galleria
    2000003579068_2.jpg
    metadata.json            ← la scheda + gli attributi della Fase 2
```

Il codice è tutto qui sotto ed è pronto a partire, ma **nessuna cella lo lancia**: le chiamate
vere stanno dietro `ESEGUI_SCRAPING`, in fondo alla sezione. Le 2901 schede sono già scaricate
e versionate nel repo.

## 1.1 Le costanti e la sessione

### `slugify(text)`

- **Input** — una stringa qualsiasi (nome di categoria, titolo).
- **Processo** — normalizza in ASCII, minuscolizza e sostituisce tutto ciò che non è
  alfanumerico con un trattino. È così che un percorso del sito diventa un percorso di cartella.
- **Output** — lo slug.

### `get_session()`

- **Input** — niente.
- **Processo** — apre una `requests.Session` con lo User-Agent dichiarato e
  `Accept-Language: it-IT`. Una sessione sola per tutto il giro: tiene le connessioni aperte e
  i cookie, invece di ripresentarsi da zero a ogni pagina.
- **Output** — la sessione da passare a tutte le funzioni che chiamano il sito.

`EXCLUDED_NAV_HINTS` e `EXCLUDED_PATH_HINTS` sono i due filtri che tengono fuori quello che non
è un prodotto: le voci di servizio del menu e le immagini di sistema del sito
(`/media/wysiwyg/`, `/media/logo`, `placeholder`, `/static/`).

In [5]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

 
BASE_URL = "https://www.nuvolari.biz"
USER_AGENT = "NuvolariPartnerBot/1.0 (+contatto: enricociaralli@gmail.com)"

BRAND_LIST_FILE = BASE / "brand_list.json"
 
CATEGORY_SEEDS_FALLBACK = [
    "/abbigliamento.html",
    "/abbigliamento-donna.html",
    "/scarpe.html",
    "/accessori.html",
]
 
EXCLUDED_NAV_HINTS = [
    "/customer/", "/checkout/", "/wishlist", "/stores", "#",
    "/privacy-policy", "/cookie-policy", "/pagamenti", "/condizioni-generali",
    "/taglie", "/spedizioni", "/resi_rimborsi", "/gift-card", "/faq",
    "/chi-siamo", "/sostenibilita", "/work-with-us", "/fidelity-card",
    ".svg", ".png", ".jpg", "javascript:",
]
 
EXCLUDED_PATH_HINTS = [
    "/media/wysiwyg/", "/media/logo", "/media/PopUp", "/static/", "placeholder",
]
 
 
def slugify(text: str) -> str:
    text = re.sub(r"[^\w\-]+", "-", text.strip().lower())
    return re.sub(r"-{2,}", "-", text).strip("-")[:120] or "prodotto"
 
 
def get_session() -> requests.Session:
    s = requests.Session()
    s.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "it-IT,it;q=0.9"})
    return s

## 1.2 Il marchio

Il link "brand" dentro la pagina è quasi sempre sbagliato — è la prima `<a href="/brands/...">`
dell'HTML, cioè una voce del menu, identica su ogni prodotto. Il marchio si riconosce quindi dal
**titolo** e, se non basta, dall'**URL**.

### `discover_brand_list(session)` · `load_brand_list()` · `save_brand_list(brand_list)`

- **Input** — la sessione; poi il file `nuvolari_db/brand_list.json`.
- **Processo** — scarica **una volta** l'elenco ufficiale dei marchi dalla pagina `/brands`
  (nome visualizzato → slug) e lo mette in cache su disco.
- **Output** — il dizionario dei marchi; `BRAND_LIST` lo carica all'avvio.

### `senza_accenti(s)` · `match_brand_from_title(title)` · `match_brand_from_url(url)` · `match_brand(title, url)`

- **Input** — titolo e URL del prodotto.
- **Processo** — confronto con l'elenco **senza accenti** (la normalizzazione Unicode NFD
  recupera i 33 prodotti `GAËLLE PARIS` che il confronto letterale perdeva); se il titolo non
  basta, si cerca lo slug nell'URL con i confini `[/-]slug[-/.]` — i confini evitano che il
  dominio faccia scattare il marchio "Nuvolari" su ogni prodotto del sito. Recupera 182 marchi.
- **Output** — `(nome, slug)` del marchio, o `(None, None)`. Copertura: 2671 su 2901 (92,1%).

In [6]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

# =====================================================================
# Lista brand ufficiale (per riconoscere il brand reale dal titolo)
# =====================================================================
#
# Il link "brand" dentro la pagina prodotto NON è affidabile: è il primo
# <a href="/brands/..."> trovato nell'HTML, che nella grande maggioranza
# dei casi è un link del menu di navigazione (sempre lo stesso, es.
# "NUVOLARI BRAND") e non il brand del prodotto specifico. Il titolo del
# prodotto invece segue quasi sempre il pattern
# "<CATEGORIA> <GENERE> <BRAND> <NOME PRODOTTO>", quindi cerchiamo quale
# brand della lista ufficiale (scaricata una volta da /brands) compare nel
# titolo.

def discover_brand_list(session: requests.Session) -> dict:
    """Scarica la lista ufficiale dei brand dal menu /brands del sito
    (nome visualizzato -> slug)."""
    try:
        r = session.get(f"{BASE_URL}/brands", timeout=30)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"  [!] impossibile scaricare la lista brand: {e}", file=sys.stderr, flush=True)
        return {}
    soup = BeautifulSoup(r.text, "lxml")
    brand_list = {}
    for a in soup.find_all("a", href=re.compile(r"/brands/[^/?#]+/?$")):
        text = a.get_text(strip=True)
        href = a.get("href")
        if text and href:
            brand_list[text] = href.rstrip("/").split("/")[-1]
    return brand_list


def load_brand_list() -> dict:
    if BRAND_LIST_FILE.exists():
        return json.loads(BRAND_LIST_FILE.read_text(encoding="utf-8"))
    return {}


def save_brand_list(brand_list: dict) -> None:
    BRAND_LIST_FILE.write_text(json.dumps(brand_list, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  [i] lista brand salvata in {BRAND_LIST_FILE} ({len(brand_list)} brand)", flush=True)


BRAND_LIST = load_brand_list()


def senza_accenti(s: str) -> str:
    """Toglie i segni diacritici, lasciando la lettera di base.

    Serve perché la lista ufficiale e i titoli dei prodotti non concordano
    sugli accenti: la lista dice "GAËLLE PARIS", i titoli scrivono "GAELLE
    PARIS". Confrontandoli così com'erano si perdevano 33 prodotti.
    """
    return "".join(c for c in unicodedata.normalize("NFD", s or "")
                   if unicodedata.category(c) != "Mn")


def match_brand_from_title(title: str) -> tuple:
    """Trova quale brand ufficiale compare nel titolo del prodotto.

    Prova prima i nomi più lunghi/specifici (es. "FRED PERRY x RAF SIMONS"
    prima di "FRED PERRY") per evitare match parziali sbagliati, e richiede
    un confine di parola per non scambiare parole generiche di senso
    compiuto per brand (es. alcuni brand nel catalogo si chiamano proprio
    "ICON"/"SUIT"/"ONLY" — il confine di parola limita, ma non azzera, il
    rischio di falsi positivi su questi nomi ambigui).
    """
    if not title or not BRAND_LIST:
        return None, None
    title_piatto = senza_accenti(title)
    for display_name, slug in sorted(BRAND_LIST.items(), key=lambda kv: -len(kv[0])):
        if re.search(r"\b" + re.escape(senza_accenti(display_name)) + r"\b",
                     title_piatto, re.IGNORECASE):
            return display_name, slug
    return None, None


def match_brand_from_url(url: str) -> tuple:
    """Ripiego sull'URL quando il titolo non nomina il brand.

    Il titolo del catalogo spesso salta il brand e tiene solo modello e colore
    ("PANTALONE TUTA 38233 NAVY/BLACK"), mentre lo slug resta nel path
    (".../pj-tuta-sergio-tacchini-sc-38233-navy-black-231.html"). Cercando lì
    si recuperano ~180 prodotti che altrimenti restano senza brand.

    Il confine richiesto è `[/-]slug[-/.]`: il dominio nuvolari.biz è preceduto
    da un punto e quindi non produce falsi positivi sul brand proprio del
    negozio, che invece viene riconosciuto quando compare davvero nel path
    (".../pantalone-nuvolari-sc-fury-tortora.html").
    """
    if not url or not BRAND_LIST:
        return None, None
    url = url.lower()
    for display_name, slug in sorted(BRAND_LIST.items(), key=lambda kv: -len(kv[1])):
        if re.search(r"[/-]" + re.escape(slug) + r"[-/.]", url):
            return display_name, slug
    return None, None


def match_brand(title: str, url: str) -> tuple:
    """Brand dal titolo se c'è, altrimenti dallo slug nell'URL."""
    nome, slug = match_brand_from_title(title)
    if slug:
        return nome, slug
    return match_brand_from_url(url)

## 1.3 La mappa del sito

### `discover_category_seeds(session)`

- **Input** — la sessione.
- **Processo** — scarica la homepage e legge il menu di navigazione, tenendo le voci che portano
  a categorie e scartando quelle di servizio. Se la homepage non risponde, ricade su
  `CATEGORY_SEEDS_FALLBACK`.
- **Output** — la lista degli URL di partenza.

### `crawl_categories(session, max_pages_per_category=60)`

- **Input** — la sessione e il tetto di pagine per categoria.
- **Processo** — per ogni categoria gira il listino pagina per pagina, raccoglie gli URL dei
  prodotti e calcola per ognuno il **percorso di cartella** in cui finirà, ricavato dal percorso
  della categoria. Si ferma quando le pagine smettono di dare prodotti nuovi.
- **Output** — la mappa `url del prodotto → percorso relativo`, cioè l'indice di tutto il lavoro
  da fare.

### `save_cache` · `load_cache` · `load_progress` · `append_progress`

- **Input/Output** — la mappa e l'elenco degli URL già fatti, su disco.
- **Processo** — servono a **riprendere**: il crawl si salva in `category_cache.json` e ogni
  prodotto completato viene appeso al file di progresso. Un giro interrotto riparte da dove era.

In [7]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

# =====================================================================
# FASE 1: scoperta categorie (invariata rispetto alla v2)
# =====================================================================
 
def discover_category_seeds(session: requests.Session) -> list:
    print("  [*] scarico la homepage per leggere il menu...", flush=True)
    seeds = set()
    try:
        r = session.get(BASE_URL, timeout=30)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"  [!] impossibile leggere la homepage: {e}", file=sys.stderr, flush=True)
        return CATEGORY_SEEDS_FALLBACK
 
    soup = BeautifulSoup(r.text, "lxml")
    for nav in soup.find_all("nav"):
        for a in nav.find_all("a", href=True):
            href = a["href"]
            full = urljoin(BASE_URL, href).split("?")[0].split("#")[0]
            if not full.startswith(BASE_URL):
                continue
            path = urlparse(full).path
            if not path or path == "/":
                continue
            if any(x in full.lower() for x in EXCLUDED_NAV_HINTS):
                continue
            seeds.add(full)
 
    return sorted(seeds) if seeds else CATEGORY_SEEDS_FALLBACK
 
 
def crawl_categories(session: requests.Session, max_pages_per_category: int = 60) -> dict:
    seeds = discover_category_seeds(session)
    print(f"  [*] {len(seeds)} categorie/brand scoperti dal menu del sito", flush=True)
 
    product_to_relpath = {}
 
    for seed in seeds:
        seed_path = urlparse(seed).path
        seed_path = seed_path[:-5] if seed_path.endswith(".html") else seed_path
        seed_parts = [slugify(p) for p in seed_path.split("/") if p]
 
        page = 1
        empty_streak = 0
        while page <= max_pages_per_category and empty_streak < 2:
            sep = "&" if "?" in seed else "?"
            url = f"{seed}{sep}p={page}"
            try:
                r = session.get(url, timeout=30)
                r.raise_for_status()
            except requests.RequestException as e:
                print(f"  [!] errore crawling {url}: {e}", file=sys.stderr, flush=True)
                break
 
            soup = BeautifulSoup(r.text, "lxml")
            links = soup.select("a.product-item-link[href]") or soup.select(".product-item-info a[href]")
 
            found = 0
            for a in links:
                href = a.get("href")
                if not (href and href.startswith("http") and "/media/" not in href):
                    continue
                href = href.split("?")[0]
                product_slug = slugify(Path(urlparse(href).path).stem)
                candidate_relpath = Path(*seed_parts, product_slug) if seed_parts else Path(product_slug)
 
                existing = product_to_relpath.get(href)
                if existing is None:
                    found += 1
                    product_to_relpath[href] = candidate_relpath
                elif len(candidate_relpath.parts) > len(existing.parts):
                    product_to_relpath[href] = candidate_relpath
 
            label = seed_path or seed
            print(f"  [*] {label} pagina {page}: {found} nuovi prodotti (totale finora: {len(product_to_relpath)})", flush=True)
            empty_streak = 0 if found else empty_streak + 1
            page += 1
            time.sleep(0.4)
 
    return product_to_relpath
 
 
def save_cache(product_to_relpath: dict, cache_file: str) -> None:
    data = {url: str(relpath) for url, relpath in product_to_relpath.items()}
    Path(cache_file).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"  [i] cache salvata in {cache_file} ({len(data)} prodotti)", flush=True)
 
 
def load_cache(cache_file: str) -> dict:
    data = json.loads(Path(cache_file).read_text(encoding="utf-8"))
    return {url: Path(relpath) for url, relpath in data.items()}
 
 
def load_progress(progress_file: str) -> set:
    p = Path(progress_file)
    if not p.exists():
        return set()
    return set(line.strip() for line in p.read_text(encoding="utf-8").splitlines() if line.strip())
 
 
def append_progress(progress_file: str, product_url: str) -> None:
    with open(progress_file, "a", encoding="utf-8") as f:
        f.write(product_url + "\n")
        f.flush()

## 1.4 La pagina prodotto

### `fetch_product_page(session, product_url)`

- **Input** — la sessione e l'URL del prodotto.
- **Processo** — scarica la pagina **una volta sola**: foto e metadati si estraggono entrambi da
  questa stessa risposta, invece di aprire la pagina due volte.
- **Output** — `(soup, None)` oppure `(None, errore)`.

### `extract_gallery_images_from_soup(soup, product_url)`

- **Input** — il DOM della pagina.
- **Processo** — la galleria di Magento sta dentro uno `<script type="text/x-magento-init">`:
  si legge quel JSON e si prende per ogni scatto la versione `full`. Se manca, si ripiega
  sull'attributo `data-gallery`. Infine si scartano le immagini di sistema del sito.
- **Output** — la lista degli URL delle foto vere del capo.

### `extract_metadata_from_soup(soup, product_url)`

- **Input** — il DOM della pagina.
- **Processo** — legge i meta OpenGraph (titolo, descrizione, prezzo, valuta), lo SKU, la
  composizione, la vestibilità e la sezione degli abbinamenti consigliati.
- **Output** — il dizionario grezzo della scheda: è l'ingresso della Fase 2.

### `download_image(session, img_url, dest_dir)`

- **Input** — l'URL di una foto e la cartella del prodotto.
- **Processo** — scarica a blocchi e **salta se il file esiste già**, così una ripresa non
  riscarica niente. Gli errori si stampano e si tira dritto: una foto persa non fa perdere il capo.
- **Output** — il file su disco.

In [8]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

 
 
# =====================================================================
# FASE 2: apertura pagina prodotto UNA VOLTA -> foto + metadata grezzi
# =====================================================================
 
def fetch_product_page(session: requests.Session, product_url: str):
    """Scarica la pagina prodotto una sola volta. Ritorna (soup, None) o (None, errore)."""
    try:
        r = session.get(product_url, timeout=30)
        r.raise_for_status()
    except requests.RequestException as e:
        return None, str(e)
    return BeautifulSoup(r.text, "lxml"), None
 
 
def extract_gallery_images_from_soup(soup: BeautifulSoup, product_url: str) -> list:
    image_urls = set()
 
    for script in soup.find_all("script", {"type": "text/x-magento-init"}):
        if not script.string or "mage/gallery/gallery" not in script.string:
            continue
        try:
            data = json.loads(script.string)
        except json.JSONDecodeError:
            continue
        for _selector, config in data.items():
            gallery_cfg = config.get("mage/gallery/gallery") if isinstance(config, dict) else None
            if not gallery_cfg:
                continue
            for item in gallery_cfg.get("data", []):
                candidate = item.get("full") or item.get("img") or item.get("thumb")
                if candidate:
                    image_urls.add(urljoin(product_url, candidate))
 
    if not image_urls:
        gallery_div = soup.find(attrs={"data-gallery-role": "gallery-placeholder"})
        if gallery_div and gallery_div.get("data-gallery"):
            try:
                items = json.loads(gallery_div["data-gallery"])
                for item in items:
                    candidate = item.get("full") or item.get("img")
                    if candidate:
                        image_urls.add(urljoin(product_url, candidate))
            except json.JSONDecodeError:
                pass
 
    return [
        u for u in image_urls
        if not any(bad in urlparse(u).path.lower() for bad in EXCLUDED_PATH_HINTS)
    ]
 
 
def extract_metadata_from_soup(soup: BeautifulSoup, product_url: str) -> dict:
    def get_meta(prop):
        tag = soup.find("meta", {"property": prop})
        return tag["content"].strip() if tag and tag.get("content") else None
 
    meta = {"url": product_url}
    meta["title"] = get_meta("og:title") or (soup.title.get_text(strip=True) if soup.title else "")
    meta["description_text"] = get_meta("og:description") or ""
 
    price_raw = get_meta("product:price:amount")
    try:
        meta["price"] = float(price_raw) if price_raw else None
    except ValueError:
        meta["price"] = None
    meta["currency"] = get_meta("product:price:currency")
 
    meta["brand"], meta["brand_slug"] = match_brand(meta["title"], meta.get("url"))
 
    full_text = soup.get_text("\n", strip=True)
 
    # "Vestibilità" compare spesso anche nel testo discorsivo della
    # descrizione (es. "vestibilità rilassata") PRIMA della vera scheda
    # tecnica più in basso in pagina (es. "Vestibilità regular"): prendiamo
    # l'ULTIMO match, non il primo, per centrare quello tecnico.
    fit_matches = list(re.finditer(r"Vestibilit[àa]\s*:?\s*(\S+)", full_text, re.IGNORECASE))
    meta["fit"] = fit_matches[-1].group(1).lower().strip(":.,") if fit_matches else None

    # stesso problema del fit: "composizione" compare a volte anche nel
    # testo di marketing (es. "la sua ricercata composizione materica...")
    # prima della vera scheda tecnica ("COMPOSIZIONE\n50% poliestere, ...",
    # senza ":") più in basso in pagina — prendiamo l'ultimo match.
    comp_matches = list(re.finditer(r"Composizione\s*:?\s*([^\n]+)", full_text, re.IGNORECASE))
    meta["composition"] = comp_matches[-1].group(1).strip() if comp_matches else None
 
    m = re.search(r"Codice Prodotto:\s*([^\n]+)", full_text, re.IGNORECASE)
    meta["sku"] = m.group(1).strip() if m else None
 
    # colore euristico dallo slug URL
    slug = Path(urlparse(product_url).path).stem
    tokens = re.split(r"[-_]", slug.lower())
    meta["color_hints"] = [t for t in tokens if t in COLOR_WORDS] or None
 
    return meta
 
 
def download_image(session: requests.Session, img_url: str, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    filename = Path(urlparse(img_url).path).name or "image.jpg"
    dest_path = dest_dir / filename
    if dest_path.exists():
        return
    try:
        r = session.get(img_url, timeout=30, stream=True)
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(8192):
                f.write(chunk)
    except requests.RequestException as e:
        print(f"    [!] errore scaricando {img_url}: {e}", file=sys.stderr, flush=True)

## 1.5 La descrizione lunga

`description_text` è l'`og:description`: corta e incoerente. La descrizione vera — quella con la
sezione "Come abbinarla?" che poi la Fase 2 manda a Gemini — sta nel corpo della pagina e si
prende con una seconda passata.

### `_testo_da(soup, selettori)` · `estrai_descrizione_completa(soup, product_url)`

- **Input** — il DOM della pagina.
- **Processo** — prova in ordine i selettori noti del tema Magento e tiene il primo blocco non
  vuoto; separa la prosa dalla scheda tecnica.
- **Output** — `{"descrizione_completa": ..., "scheda_tecnica": ...}`.

### `scarica_descrizioni(output, delay=0.7, progress_file=None, max_products=None, riparti=False)`

- **Input** — la cartella del catalogo già scaricato.
- **Processo** — rivisita ogni pagina prodotto e allega la descrizione completa al suo
  `metadata.json`, **senza toccare foto né attributi**. Riprendibile: chi ha già
  `descrizione_completa` si salta. I campi vecchi restano dove sono, perché sovrascriverli
  renderebbe impossibile confrontare le due versioni.
- **Output** — i `metadata.json` arricchiti.

In [9]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

# =====================================================================
# Descrizione completa dalla pagina prodotto
# =====================================================================
#
# og:description — l'unica cosa che salvavamo — e' una versione accorciata e
# incoerente: su 2901 schede solo il 21% conservava la sezione "Come abbinarla?"
# e il 32% il blocco tecnico. La pagina invece ha entrambi, in due contenitori
# distinti, e nella parte discorsiva ci sono LINK alle categorie dei capi da
# abbinare — che non sono prosa da interpretare ma puntatori mappabili sugli
# slot dell'outfit.

SELETTORI_DESCRIZIONE = ("#product-info-short-description", ".wrapper-short-description")
SELETTORI_SCHEDA = ("#description", ".product.attribute.description")


def _testo_da(soup, selettori) -> str:
    for sel in selettori:
        el = soup.select_one(sel)
        if el:
            testo = el.get_text("\n", strip=True)
            if testo:
                return testo
    return ""


def estrai_descrizione_completa(soup, product_url: str) -> dict:
    """Descrizione discorsiva, scheda tecnica e link di abbinamento."""
    contenitore = None
    for sel in SELETTORI_DESCRIZIONE:
        contenitore = soup.select_one(sel)
        if contenitore:
            break

    abbinamenti = []
    if contenitore:
        for a in contenitore.find_all("a", href=True):
            href = urljoin(product_url, a["href"])
            if "/coordinati" in href:
                continue  # rimando generico agli outfit del sito, non un capo
            abbinamenti.append({"testo": a.get_text(strip=True), "categoria_url": href})

    return {
        "descrizione_completa": _testo_da(soup, SELETTORI_DESCRIZIONE),
        "scheda_tecnica": _testo_da(soup, SELETTORI_SCHEDA),
        "abbinamenti_suggeriti": abbinamenti,
    }


def scarica_descrizioni(output: str, delay: float = 0.7, progress_file: str = None,
                        max_products: int = None, riparti: bool = False) -> dict:
    """Rivisita ogni pagina prodotto e allega la descrizione completa al suo
    metadata.json, senza toccare foto ne' attributi.

    Riprendibile: i capi che hanno gia' 'descrizione_completa' si saltano, a
    meno di riparti=True. I campi vecchi (description_text) restano dove sono:
    servono ancora a chi legge la scheda accorciata, e sovrascriverli
    renderebbe impossibile confrontare le due versioni.
    """
    output_dir = Path(output)
    meta_paths = sorted(output_dir.rglob("metadata.json"))
    da_fare = []
    for mp in meta_paths:
        m = json.loads(mp.read_text(encoding="utf-8"))
        if not m.get("url"):
            continue
        if riparti or "descrizione_completa" not in m:
            da_fare.append(mp)
    if max_products:
        da_fare = da_fare[:max_products]
    print(f"[*] {len(da_fare)} schede da scaricare (su {len(meta_paths)} prodotti)", flush=True)

    session = get_session()
    fatti = errori = con_abbinamenti = 0
    for i, mp in enumerate(da_fare, 1):
        m = json.loads(mp.read_text(encoding="utf-8"))
        soup, err = fetch_product_page(session, m["url"])
        if soup is None:
            errori += 1
            print(f"  [{i}/{len(da_fare)}] ERRORE {m.get('relpath','?')[:46]}: {err}", flush=True)
            time.sleep(delay)
            continue
        m.update(estrai_descrizione_completa(soup, m["url"]))
        mp.write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding="utf-8")
        fatti += 1
        if m["abbinamenti_suggeriti"]:
            con_abbinamenti += 1
        if i % 100 == 0 or i == len(da_fare):
            print(f"  [{i}/{len(da_fare)}] {fatti} scaricate, {con_abbinamenti} con abbinamenti, "
                  f"{errori} errori", flush=True)
        time.sleep(delay)

    print(f"\n[OK] {fatti} descrizioni allegate, {con_abbinamenti} con link di abbinamento, "
          f"{errori} errori", flush=True)
    return {"scaricate": fatti, "con_abbinamenti": con_abbinamenti, "errori": errori}

## 1.6 I giri completi

### `run_scrape_with_attributes(output, delay=1.0, max_products=None, max_pages_per_category=60, cache_file=None, force_recrawl=False, progress_file=None)`

- **Input** — dove scrivere il catalogo, più i parametri di cortesia (`delay` fra una pagina e
  l'altra) e di ripresa.
- **Processo** — il giro intero: crawl delle categorie (o cache), poi per ogni prodotto una sola
  apertura della pagina, download delle foto, estrazione della scheda, **e subito la Fase 2
  testuale** (`derive_attributes`), scrivendo `metadata.json` accanto alle foto.
- **Output** — il catalogo su disco e un riepilogo di quanti prodotti sono stati fatti.

### `refresh_metadata(output, delay=1.0, ...)`

- **Input** — il catalogo già scaricato.
- **Processo** — rivisita ogni pagina e **ri-estrae solo i metadati testuali**, senza toccare le
  foto. Serve dopo una correzione a `extract_metadata_from_soup` o a `derive_attributes`.
- **Output** — i `metadata.json` riscritti.

### `refix_brand_and_attributes(output)`

- **Input** — il catalogo già scaricato.
- **Processo** — **senza rete**: ricorregge marchio e slug da titolo e URL già salvati, e
  ricalcola di conseguenza gli attributi. Non è un extra: `brand_slug` entra nei punteggi di
  stile tramite `BRAND_AFFINITY`, quindi correggere il marchio senza rifare `derive_attributes`
  lascerebbe i tag coerenti con il marchio vecchio.
- **Output** — i `metadata.json` corretti.

### `rebuild_catalog(output)`

- **Input** — il catalogo.
- **Processo** — ricompone l'indice `catalogo.jsonl`, una riga per prodotto.
- **Output** — il file d'indice usato a valle.

In [10]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

 
 
# =====================================================================
# FUNZIONE PRINCIPALE
# =====================================================================
 
def run_scrape_with_attributes(
    output: str = "output",
    delay: float = 1.0,
    max_products: int = None,
    max_pages_per_category: int = 60,
    cache_file: str = None,
    force_recrawl: bool = False,
    progress_file: str = None,
):
    """
    Scarica foto + metadata + attributi stilistici per ogni prodotto,
    salvando nella struttura organizzata per categoria.
 
    Genera, dentro `output`:
      <categoria>/<sottocategoria>/<slug>/foto*.jpg
      <categoria>/<sottocategoria>/<slug>/metadata.json
      catalogo.jsonl   <- un JSON per riga, un prodotto per riga, per tutto il catalogo
 
    Parametri: vedi versioni precedenti (output, delay, max_products,
    max_pages_per_category, cache_file, force_recrawl, progress_file).
    """
    print(f"[*] Avvio scraping con attributi in {output} ...", flush=True)
    session = get_session()
    output_dir = Path(output)
    output_dir.mkdir(parents=True, exist_ok=True)
    catalog_path = output_dir / "catalogo.jsonl"
 
    if cache_file and Path(cache_file).exists() and not force_recrawl:
        print(f"[*] Trovata cache in {cache_file}, la uso invece di riscansionare...", flush=True)
        product_to_relpath = load_cache(cache_file)
        print(f"    {len(product_to_relpath)} prodotti caricati dalla cache", flush=True)
    else:
        print("[*] Scopro le categorie dal menu del sito e le scansiono...", flush=True)
        product_to_relpath = crawl_categories(session, max_pages_per_category=max_pages_per_category)
        print(f"\n[*] {len(product_to_relpath)} prodotti totali trovati nel sito", flush=True)
        if cache_file:
            save_cache(product_to_relpath, cache_file)
 
    items = list(product_to_relpath.items())
 
    already_done = set()
    if progress_file:
        already_done = load_progress(progress_file)
        if already_done:
            print(f"[*] Trovato progress_file: {len(already_done)} prodotti gia' completati, li salto.", flush=True)
        Path(progress_file).touch(exist_ok=True)
        items = [(url, relpath) for url, relpath in items if url not in already_done]
 
    if max_products:
        items = items[:max_products]
 
    print(f"\n[*] Elaboro {len(items)} prodotti (restanti da fare)...\n", flush=True)
 
    total_images = 0
    total_products_ok = 0
    total_needs_review = 0
 
    with open(catalog_path, "a", encoding="utf-8") as catalog_f:
        for i, (url, relpath) in enumerate(items, 1):
            print(f"[{i}/{len(items)}] {relpath}", flush=True)
 
            soup, err = fetch_product_page(session, url)
            if soup is None:
                print(f"    [!] errore pagina: {err}", flush=True)
                if progress_file:
                    append_progress(progress_file, url)
                time.sleep(delay)
                continue
 
            images = extract_gallery_images_from_soup(soup, url)
            raw_metadata = extract_metadata_from_soup(soup, url)
            attributes = derive_attributes(raw_metadata, relpath)
 
            if not images:
                print("    (nessuna galleria trovata, salto le foto ma salvo comunque i metadata)", flush=True)
            else:
                dest_dir = output_dir / relpath
                for img_url in images:
                    download_image(session, img_url, dest_dir)
                dest_dir.mkdir(parents=True, exist_ok=True)
                (dest_dir / "metadata.json").write_text(
                    json.dumps({**raw_metadata, **attributes, "relpath": str(relpath), "image_count": len(images)},
                               ensure_ascii=False, indent=2),
                    encoding="utf-8",
                )
                total_images += len(images)
 
            record = {**raw_metadata, **attributes, "relpath": str(relpath)}
            catalog_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            catalog_f.flush()
 
            total_products_ok += 1
            if attributes["needs_vision_review"]:
                total_needs_review += 1
 
            tags_str = ", ".join(attributes["style_tags"].keys()) or "(nessuno — needs_vision_review)"
            print(f"    -> {len(images)} foto | stile: {tags_str} | formalita': {attributes['formality_score']}", flush=True)
 
            if progress_file:
                append_progress(progress_file, url)
 
            time.sleep(delay)
 
    stats = {
        "prodotti_processati": total_products_ok,
        "foto_totali": total_images,
        "prodotti_da_rivedere_vision_api": total_needs_review,
        "catalogo": str(catalog_path.resolve()),
        "output_dir": str(output_dir.resolve()),
    }
 
    print("\n[OK] Fatto (per questa sessione).", flush=True)
    print(f"     Prodotti processati:              {stats['prodotti_processati']}", flush=True)
    print(f"     Foto totali scaricate:             {stats['foto_totali']}", flush=True)
    print(f"     Da rivedere con vision API:         {stats['prodotti_da_rivedere_vision_api']}", flush=True)
    print(f"     Catalogo consolidato:               {stats['catalogo']}", flush=True)
    if progress_file:
        print(f"     Per riprendere in futuro: stesso comando, stesso progress_file='{progress_file}'", flush=True)

    return stats


# =====================================================================
# REFRESH: ri-estrae solo i metadata testuali (no foto) per correggere
# i regex di composizione/vestibilità senza rifare tutto lo scraping
# =====================================================================

def refresh_metadata(output: str, delay: float = 1.0, progress_file: str = None, max_products: int = None) -> dict:
    """Rivisita ogni pagina prodotto già scaricata (stesso URL salvato nel
    metadata.json esistente), ri-estrae SOLO i metadata testuali (niente
    foto, già presenti su disco) e riscrive metadata.json con i valori
    corretti di composition/fit/style_tags/formality_score/season/pattern.

    Usare dopo una correzione ai regex/alla logica di extract_metadata_from_soup
    o derive_attributes, per propagare la correzione senza dover riscaricare
    le foto (già buone).
    """
    print(f"[*] Avvio refresh metadata (solo pagina, no foto) in {output} ...", flush=True)
    session = get_session()
    output_dir = Path(output)

    meta_paths = sorted(output_dir.rglob("metadata.json"))
    print(f"[*] {len(meta_paths)} metadata.json trovati", flush=True)

    already_done = set()
    if progress_file:
        already_done = load_progress(progress_file)
        if already_done:
            print(f"[*] Trovato progress_file: {len(already_done)} gia' aggiornati, li salto.", flush=True)
        Path(progress_file).touch(exist_ok=True)

    if max_products:
        meta_paths = meta_paths[:max_products]

    total_ok = 0
    total_errors = 0
    total_needs_review = 0

    for i, meta_path in enumerate(meta_paths, 1):
        old_metadata = json.loads(meta_path.read_text(encoding="utf-8"))
        url = old_metadata.get("url")
        relpath = Path(old_metadata.get("relpath", str(meta_path.parent.relative_to(output_dir))))

        if not url or url in already_done:
            continue

        soup, err = fetch_product_page(session, url)
        if soup is None:
            print(f"[{i}/{len(meta_paths)}] {relpath} -> errore pagina: {err}", flush=True)
            total_errors += 1
            if progress_file:
                append_progress(progress_file, url)
            time.sleep(delay)
            continue

        raw_metadata = extract_metadata_from_soup(soup, url)
        attributes = derive_attributes(raw_metadata, relpath)

        new_metadata = {
            **raw_metadata,
            **attributes,
            "relpath": str(relpath),
            "image_count": old_metadata.get("image_count"),
        }
        meta_path.write_text(json.dumps(new_metadata, ensure_ascii=False, indent=2), encoding="utf-8")

        total_ok += 1
        if attributes["needs_vision_review"]:
            total_needs_review += 1

        tags_str = ", ".join(attributes["style_tags"].keys()) or "(nessuno)"
        comp = raw_metadata.get("composition") or "-"
        fit = raw_metadata.get("fit") or "-"
        print(f"[{i}/{len(meta_paths)}] {relpath} -> fit={fit} | comp={comp} | stile: {tags_str}", flush=True)

        if progress_file:
            append_progress(progress_file, url)

        time.sleep(delay)

    print("\n[OK] Refresh completato.", flush=True)
    print(f"     Aggiornati:          {total_ok}", flush=True)
    print(f"     Errori pagina:       {total_errors}", flush=True)
    print(f"     Needs vision review: {total_needs_review}", flush=True)

    return {"aggiornati": total_ok, "errori": total_errors, "needs_vision_review": total_needs_review}


def refix_brand_and_attributes(output: str) -> dict:
    """Ricorregge SOLO brand/brand_slug (da titolo e URL già salvati, tramite
    match_brand) e ricalcola di conseguenza style_tags/formality_score/
    needs_vision_review — tutto in locale, senza rete, perché
    title/url/description/composition/fit sono già in metadata.json.

    Serve dopo aver corretto match_brand (in precedenza il link brand estratto
    era quasi sempre sbagliato — vedi extract_metadata_from_soup — e poi il
    match guardava solo il titolo, che spesso non nomina il brand).

    Il ricalcolo degli attributi non è un extra: brand_slug entra nei punteggi
    di stile tramite BRAND_AFFINITY, quindi correggere il brand senza rifare
    derive_attributes lascerebbe style_tags coerenti col brand vecchio.
    """
    output_dir = Path(output)
    meta_paths = sorted(output_dir.rglob("metadata.json"))
    print(f"[*] Ricorreggo brand + attributi per {len(meta_paths)} prodotti (nessuna rete)...", flush=True)

    changed_brand = 0
    for i, meta_path in enumerate(meta_paths, 1):
        metadata = json.loads(meta_path.read_text(encoding="utf-8"))
        relpath = Path(metadata.get("relpath", str(meta_path.parent.relative_to(output_dir))))

        old_brand_slug = metadata.get("brand_slug")
        brand, brand_slug = match_brand(metadata.get("title"), metadata.get("url"))
        metadata["brand"] = brand
        metadata["brand_slug"] = brand_slug
        if brand_slug != old_brand_slug:
            changed_brand += 1

        attributes = derive_attributes(metadata, relpath)
        metadata.update(attributes)

        meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

        if i % 200 == 0 or i == len(meta_paths):
            print(f"    [{i}/{len(meta_paths)}] elaborati...", flush=True)

    print(f"\n[OK] Brand corretto per {changed_brand}/{len(meta_paths)} prodotti.", flush=True)
    return {"totale": len(meta_paths), "brand_corretti": changed_brand}


def rebuild_catalog(output: str) -> str:
    """Ricostruisce catalogo.jsonl da tutti i metadata.json presenti su disco
    (dopo un refresh_metadata, per tenerlo allineato)."""
    output_dir = Path(output)
    catalog_path = output_dir / "catalogo.jsonl"
    meta_paths = sorted(output_dir.rglob("metadata.json"))
    with open(catalog_path, "w", encoding="utf-8") as f:
        for meta_path in meta_paths:
            record = json.loads(meta_path.read_text(encoding="utf-8"))
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"[OK] catalogo.jsonl ricostruito con {len(meta_paths)} prodotti in {catalog_path}", flush=True)
    return str(catalog_path)

### Le chiamate vere

La cella qui sotto è l'unica di questa sezione che tocca il sito. Parte solo se
`ESEGUI_SCRAPING` è `True`, ed è tarata su **3 prodotti**: serve a vedere il meccanismo, non a
rifare il catalogo. Per il giro intero si toglie `max_products`.

Richiede che le celle di definizione del notebook siano già state eseguite: usa
`derive_attributes` della Fase 2.

In [11]:
if ESEGUI_SCRAPING:
    riassunto = run_scrape_with_attributes(
        output=str(CATALOGO),
        cache_file=str(BASE / "category_cache.json"),
        progress_file=str(BASE / "progress.txt"),
        delay=1.0,
        max_products=3,          # ← togliere per il catalogo intero (2901 prodotti)
    )
    print(riassunto)
else:
    print("ESEGUI_SCRAPING = False: nessuna chiamata a nuvolari.biz.")
    print("Il catalogo è già sul disco:")
    n = sum(1 for _ in CATALOGO.rglob("metadata.json"))
    print(f"   {n} schede prodotto in {CATALOGO.relative_to(RADICE).as_posix()}/")

ESEGUI_SCRAPING = False: nessuna chiamata a nuvolari.biz.
Il catalogo è già sul disco:


   2901 schede prodotto in nuvolari_full_organizzato/


In [12]:
# Che cosa ha lasciato la Fase 1 sul disco, per il capo di esempio.
print(ESEMPIO)
for p in sorted(cartella.iterdir()):
    print(f"   {p.name:34s} {p.stat().st_size//1024:5d} KB")
print()
print("i campi che vengono dalla pagina (il resto lo aggiunge la Fase 2):")
for campo in ["url", "title", "price", "currency", "sku", "composition", "fit", "brand"]:
    valore = meta.get(campo)
    print(f"   {campo:14s} {str(valore)[:78]}")
print(f"   {'descrizione':14s} {len(meta.get('descrizione_completa') or ''):d} caratteri"
      f" (og:description: {len(meta.get('description_text') or ''):d})")
print(f"   {'abbinamenti':14s} {len(meta.get('abbinamenti_suggeriti') or [])} suggeriti dalla scheda")

abbigliamento/camicie/2-pkt-camp-shirt-es3-112363920-soy-bean-soy-bean
   2000003579068_1.jpg                  140 KB
   2000003579068_2.jpg                  277 KB
   2000003579068_3.jpg                  123 KB
   metadata.json                          7 KB

i campi che vengono dalla pagina (il resto lo aggiunge la Fase 2):
   url            https://www.nuvolari.biz/2-pkt-camp-shirt-es3-112363920-soy-bean-soy-bean.html
   title          CAMICIA UOMO LEE CAMP SHIRT SOY BEAN
   price          30.0
   currency       EUR
   sku            None
   composition    54% lino, 46% viscosa
   fit            regular
   brand          LEE
   descrizione    2964 caratteri (og:description: 2957)
   abbinamenti    1 suggeriti dalla scheda


---
# 2. La valutazione del capo — Fase 2

Prima che esista qualsiasi numero confrontabile, il capo va **giudicato**. Succede due volte,
e le due valutazioni non sono alternative: la prima decide chi ha bisogno della seconda.

1. **La valutazione testuale**, dentro `scrape_with_attributes.py`: dal titolo, dalla
   descrizione, dalla composizione, dalla vestibilità e dal marchio si ricavano dei punteggi di
   stile. Quando quei punteggi non hanno **riscontro nel testo** — arrivano solo da segnali
   generici come "98% cotone" — il capo viene messo in quarantena con `needs_vision_review`.
2. **La valutazione visiva**, dentro `vision_review.py`: la foto del capo va a Gemini insieme
   alla scheda del negozio, con `guida_classificazione.md` come istruzione di sistema. Torna un
   JSON con la ripartizione di stile, la formalità (e il perché), la stagione e la fantasia.

La quarantena nasce da una diffidenza precisa: un capo con la scheda muta otterrebbe comunque
`casual` sopra soglia, **non perché sappiamo com'è fatto ma perché non sappiamo nulla** e i
segnali generici hanno riempito il vuoto. Trattarlo come "taggato con sicurezza" lo renderebbe
abbinabile a tutto.

Oggi la revisione visiva è stata estesa a **tutto** il catalogo (`vision_review.py --tutti`),
non solo alla quarantena: così un capo ben descritto e uno muto stanno sulla stessa scala.

## 2.1 La valutazione testuale

### `compute_style_scores(metadata)`

- **Input** — il `metadata.json` del capo (titolo, descrizione, composizione, vestibilità, marchio).
- **Processo** — accumula punti su ogni registro da quattro sorgenti con pesi diversi: parola nel
  **titolo** +0,8 (lì la parola *nomina* il capo), **materiale** +0,5, **vestibilità** +0,3,
  **marchio** +0,4. Dalla prosa non prende più niente: le uniche parole che ci si fidava a
  leggere alimentavano tag che non esistono più, e l'elenco è rimasto vuoto di proposito.
  Tiene anche traccia di **quali tag hanno un riscontro nel testo**, che è l'informazione su cui
  si decide la quarantena.
- **Output** — `(punteggi_grezzi, tag_con_riscontro_testuale)`. I punteggi non sono sogliati:
  servono continui.

- **Attenzione al senso del punteggio** — un punteggio alto non vuol dire "so di più su questo
  capo": vuol dire solo che più indizi *generici* (materiale, marchio, vestibilità) puntano
  nella stessa direzione. Il capo di esempio qui sotto arriva a 1,7 su `casual` senza un solo
  riscontro nel testo, e infatti va comunque in revisione visiva.

### `derive_attributes(metadata, category_path)`

- **Input** — lo stesso `metadata.json` e il percorso di categoria.
- **Processo** — applica la soglia **0,5** ai punteggi per ottenere i `style_tags` "confermati",
  e alza `needs_vision_review` quando nessun tag sopra soglia ha riscontro nel testo. Nello
  stesso passaggio derivano formalità testuale, stagione, fantasia e vestibilità
  (`formalita_da_tipo`, `season_da_tipo`: il tipo di capo letto nel titolo, dal livello più
  formale al meno, con i termini specifici — cargo, k-way, trackpant — controllati **prima** di
  quelli generici).
- **Output** — il blocco di attributi che viene scritto dentro `metadata.json`.

In [13]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

 
# =====================================================================
# REGOLE DI TAGGING (da style_tagging_rules.md)
# =====================================================================
 
STYLE_KEYWORDS = {
    "sportivo": [
        "sportivo", "sport", "running", "training", "palestra", "fitness",
        "tuta", "jogging", "tecnico da gara",
    ],
    "elegante": [
        "elegante", "formale", "cerimonia", "sartoriale", "raffinato",
        "classico", "business",
    ],
    "streetwear": [
        "streetwear", "street", "urban", "skate", "hip hop", "oversize",
        "graphic", "logo all over",
    ],
    "da_mare": [
        "mare", "spiaggia", "beach", "costume", "boxer da bagno", "pareo",
        "copricostume", "telo", "infradito", "ciabatte",
    ],
    "casual": [
        "casual", "quotidiano", "informale", "everyday", "relax",
    ],
}

MATERIAL_SIGNALS = {
    "lana":          {"style": ["elegante"], "season": "inverno"},
    "cashmere":      {"style": ["elegante"], "season": "inverno"},
    "piumino":       {"style": ["casual"], "season": "inverno"},
    "nylon":         {"style": ["streetwear"], "season": "tutte"},
    "pelle":         {"style": ["streetwear", "elegante"], "season": "mezza_stagione"},
    "denim":         {"style": ["casual"], "season": "tutte"},
    "lino":          {"style": ["elegante", "casual"], "season": "estate"},
    "cotone":        {"style": ["casual"], "season": "tutte"},
    "poliestere":    {"style": ["sportivo"], "season": "tutte"},
    "viscosa":       {"style": ["elegante", "casual"], "season": "mezza_stagione"},
    "elastan":       {"style": ["sportivo"], "season": "tutte"},
}
 
# Stagionalità intrinseca del TIPO di capo, indipendente da materiale e testo.
#
# È il segnale che mancava. La stagione veniva dedotta solo dalle parole della
# scheda ("estivo", "invernale") e dalla composizione, ma alcuni capi sono
# stagionali per natura: una sciarpa di cotone resta un capo invernale, un
# costume da bagno resta estivo. Senza questa tabella finivano in "tutte", che
# nell'abbinamento è un jolly compatibile con qualsiasi stagione — ed è così che
# una sciarpa si ritrovava sopra una t-shirt estiva.
#
# Ha la precedenza su materiale e testo proprio perché è il segnale più
# affidabile: il tipo di capo non è ambiguo, la composizione e la descrizione sì.
# Le chiavi si cercano nel titolo con confine di parola.
# Sciarpe e tessuti leggeri: servono alla deroga in derive_attributes.
SCIARPA = re.compile(r"\b(sciarp\w*|scaldacollo|foulard|stola)\b")
TESSUTO_LEGGERO = re.compile(r"\b(lino|cotone|seta|linen|cotton|silk)\b")

GARMENT_SEASON = {
    "inverno": [
        "sciarpa", "sciarpe", "scaldacollo", "guanti", "muffole",
        "cappotto", "piumino", "montone", "parka", "pile",
        "maglione", "dolcevita", "lupetto",
        "colbacco", "passamontagna", "berretto di lana",
        # volutamente fuori: cardigan, felpa, giacca. Sono capi trasversali
        # (un cardigan leggero sta su una t-shirt in primavera) e marcarli
        # "inverno" escluderebbe abbinamenti corretti invece di correggerne.
    ],
    "estate": [
        "costume", "bikini", "boxer mare", "pareo",
        "canotta", "canottiera", "top a fascia",
        "infradito", "ciabatte", "sandalo", "sandali",
        "bermuda", "pantaloncino", "pantaloncini", "shorts",
        "cappello di paglia",
    ],
}


# La stagione dal TESTO vale solo se l'aggettivo stagionale descrive il capo
# o il suo tessuto, non l'occasione d'uso. Misurato sulle schede: 405 capi su
# 671 diventavano "estate" per frasi come "un aperitivo estivo", "la tua
# estate", "un look estivo" — marketing sull'occasione, non sul capo — e
# camicie trasversali venivano cosi' escluse da ogni outfit invernale. I
# segnali sostanziali osservati ("fibra estiva", "filato estivo", "classica
# maglietta estiva") hanno tutti la stessa forma: sostantivo di capo/tessuto
# seguito dall'aggettivo.
_NOME_DI_CAPO_O_TESSUTO = (
    r"(?:capo|capi|tessut\w+|fibr\w+|filat\w+|magli\w+|camici\w+|"
    r"pantalon\w+|giacc\w+|giubbott\w+|felp\w+|modell\w+|version\w+|variant\w+)"
)
STAGIONE_ESTIVA_DEL_CAPO = re.compile(_NOME_DI_CAPO_O_TESSUTO + r"\s+estiv")
STAGIONE_INVERNALE_DEL_CAPO = re.compile(_NOME_DI_CAPO_O_TESSUTO + r"\s+invernal")


def season_da_tipo(title: str):
    """Stagione dedotta dal tipo di capo, o None se il capo non è stagionale.

    Cerca con confine di parola per non far scattare "sandalo" dentro altre
    parole. Il primo riscontro vince: le liste non si sovrappongono.
    """
    t = senza_accenti(title or "").lower()
    for stagione, parole in GARMENT_SEASON.items():
        for parola in parole:
            if re.search(r"\b" + re.escape(parola) + r"\b", t):
                return stagione
    return None

In [14]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

FIT_SIGNALS = {
    "slim":     {"style": ["elegante"], "formality_delta": 1},
    "skinny":   {"style": ["streetwear"], "formality_delta": 0},
    "regular":  {"style": ["casual"], "formality_delta": 0},
    "oversize": {"style": ["streetwear"], "formality_delta": -1},
    "baggy":    {"style": ["streetwear"], "formality_delta": -1},
    "relaxed":  {"style": ["casual"], "formality_delta": 0},
}
 
PATTERN_KEYWORDS = {
    "righe":      ["a righe", "rigato", "righine"],
    "quadri":     ["a quadri", "check", "tartan"],
    "camouflage": ["camouflage", "mimetico"],
    "stampato":   ["stampa", "stampato", "grafica"],
    "animalier":  ["animalier", "leopardato", "zebrato"],
}
 
CATEGORY_FORMALITY_PRIOR = {
    "completi": 5, "giacche": 4, "camicie": 3, "polo": 3,
    "maglieria": 3, "maglie": 3, "pantaloni": 3, "jeans": 2, "t-shirt": 2,
    "felpe": 2, "tute": 1, "costumi": 1, "sneakers": 2, "giubbotti": 2,
    "cappotti": 4, "bomber": 2, "gilet": 3,
}
 
# Formalità intrinseca del TIPO di capo, riconosciuta dal TITOLO.
#
# CATEGORY_FORMALITY_PRIOR sopra guarda solo il percorso di categoria, che nel
# catalogo è spesso generico ("abbigliamento", "accessori"): 1018 prodotti su
# 2901 non trovavano nessuna corrispondenza e ricadevano sul neutro 3, e con
# essi TUTTI gli accessori, perché "accessori" non è una chiave di quella
# tabella. Risultato: il 67% del catalogo finiva a formalità 0,50 normalizzata,
# cioè la formalità non distingueva più una camicia da uno zaino — ed è così
# che una borsa street si ritrovava abbinata a camicia e mocassini.
#
# Il titolo è il segnale più ricco: dice sempre di che capo si tratta.
GARMENT_FORMALITY = {
    1: ["zaino", "zainetto", "marsupio", "tuta", "costume", "bikini",
        "infradito", "ciabatte", "canotta", "canottiera", "pareo",
        # zaini, marsupi e pochette a tracolla sono capi street: portati sopra
        # un maglione e un chino stonano, e il vincolo di dispersione li tiene
        # fuori dagli outfit smart solo se partono davvero dal minimo
        "pochette"],
    2: ["t-shirt", "tshirt", "felpa", "jeans", "sneakers", "bermuda",
        "pantaloncino", "pantaloncini", "shorts", "bomber", "giubbotto",
        "cappellino", "cappello", "tracolla", "sandalo", "sandali"],
    3: ["camicia", "polo", "maglia", "maglione", "cardigan", "pantalone",
        "chino", "gilet", "borsa", "sciarpa", "dolcevita", "gonna", "vestito"],
    4: ["giacca", "blazer", "cappotto", "mocassini", "mocassino", "parka",
        "trench", "stringate", "derby", "oxford"],
    5: ["completo", "smoking", "frac", "cravatta", "papillon"],
}


# Tetto di formalità per brand.
#
# Le parole di GARMENT_FORMALITY descrivono la FORMA del capo, non il registro:
# "borsa" vale sia per una borsa in pelle da donna sia per la REDBOX CARRY BAG
# di The North Face, che è una sacca da montagna verde lime; "giacca" vale sia
# per un blazer sia per il K-Way LE VRAI CLAUDE, che è un antivento. La forma
# non basta a distinguerli, il marchio sì: ci sono brand la cui produzione sta
# tutta in un registro, e per loro il tipo di capo non deve poter alzare il
# punteggio oltre un certo livello.
#
# Il tetto conta perché la formalità entra nel vincolo di dispersione (0,3):
# tetto 1 (= 0,00 normalizzato) ammette accanto solo capi di livello 1-2 —
# t-shirt, felpe, tute, sneakers, mai camicie o mocassini. Tetto 2 (= 0,25)
# arriva fino al livello 3, quindi un piumino The North Face può ancora stare
# con un maglione, ma non con un completo.
#
# È un tetto e non un valore fisso: un costume K-Way è già 1 e ci resta.
BRAND_FORMALITY_TETTO = {
    # street puro: qualunque capo è un capo trap
    "sprayground": 1, "bape": 1, "a-bathing-ape": 1, "off-white": 1,
    "stussy": 1, "supreme": 1,
    # K-Way fa antivento: il titolo li chiama "giacca" e 28 capi su 51
    # finivano a formalità 3, cioè alla pari di una camicia. Tetto 2 e non 1:
    # l'antivento è casual sportivo, non street. A 1 (= 0,00) la dispersione
    # massima di 0,3 lo avrebbe ammesso solo accanto a livelli 1-2, cioè
    # tute, costumi e marsupi — e avrebbe escluso l'abbinamento con un
    # pantalone cargo, che è livello 3 ed è invece un outfit giusto.
    "k-way": 2,
    # sportivi e outdoor: il capospalla tecnico e la sacca da montagna non
    # sono formali, ma restano abbinabili a un maglione — da qui il tetto 2
    "the-north-face": 2, "napapijri": 2, "nike": 2, "adidas": 2,
    "adidas-originals": 2, "new-balance": 2, "asics": 2, "saucony": 2,
    "hoka": 2, "puma": 2, "under-armour": 2,
}


# Termini che nominano un modello preciso e battono la parola generica che li
# accompagna. Servono perché la scansione sotto va dal più formale al meno
# formale, e quindi la parola generica — che è quasi sempre la più formale —
# vince sempre: in "PANTALONE CARGO" si fermava su "pantalone" (3, come una
# camicia) senza mai guardare "cargo", che è workwear; in "GIACCA K-WAY LE
# VRAI" si fermava su "giacca" (4, come un blazer) invece di riconoscere un
# antivento. La regola "vince il più formale" nasceva per un caso solo —
# "GIACCA CAMICIA" è un capospalla — e lì continua a valere, perché queste
# eccezioni si controllano prima e riguardano altri termini.
GARMENT_FORMALITY_SPECIFICO = {
    "cargo": 2,       # 68 capi su 73 stavano a 3 dietro la parola "pantalone"
    "kway": 2,
    "k-way": 2,
    "antivento": 2,
    "trackpant": 2,
    "tracktop": 2,
}


# Accessori-contenitore. Di un brand sportivo o street sono capi street a
# tutti gli effetti, qualunque parola usi il titolo: la tabella mette già
# zaino e marsupio a 1, ma "borsa" sta a 3 perché la stessa parola vale anche
# per una borsa in pelle da donna. Da lì l'incoerenza misurata nel catalogo —
# ZAINO THE NORTH FACE BASE CAMP a livello 1 e BORSA THE NORTH FACE REDBOX
# CARRY BAG a livello 2, stessa marca e stessa funzione.
ACCESSORI_CONTENITORE = re.compile(
    r"\b(borsa|borse|sacca|sacche|zaino|zainetto|marsupio|marsupi|tracolla|"
    r"tracolle|pochette|backpack|bkpack|daypack|duffel|carry bag)\b")


# Cappotti travestiti da giubbotto. Stessa famiglia di difetti del cargo e del
# K-Way — la parola generica del titolo nasconde il capo — ma qui il rimedio non
# può stare in GARMENT_FORMALITY_SPECIFICO, perché nel titolo la forma non
# compare affatto: "GIUBBOTTO UOMO NUVOLARI OTIS NAVY" è un caban di lana
# doppiopetto, e lo si scopre solo leggendo la scheda.
CAPPOTTO_FORMA = re.compile(
    r"(caban|montgomery|doppiopetto|sei bottoni|peacoat|cappotto|trench)")
# Il titolo ha l'ultima parola: un gilet imbottito e una giacca a vento citano
# il cappotto nella scheda per confronto, ma cappotti non sono.
CAPPOTTO_ESCLUSI = re.compile(r"\b(gilet|giacca a vento|piumino|smanicato)\b")
# Frase di repertorio che elenca i reparti del negozio invece di descrivere il
# prodotto ("disponibili in vari stili come sneakers, stivali o mocassini").
DESCRIZIONE_DI_REPERTORIO = re.compile(r"variet(?:a|à) di stili come")
CAPPOTTO_FORMALITA_MINIMA = 3


def e_un_cappotto(metadata: dict) -> bool:
    titolo = senza_accenti(metadata.get("title") or "").lower()
    if CAPPOTTO_ESCLUSI.search(titolo):
        return False
    if CAPPOTTO_FORMA.search(titolo):
        return True
    descrizione = senza_accenti(metadata.get("description_text") or "").lower()
    if DESCRIZIONE_DI_REPERTORIO.search(descrizione):
        return False
    return bool(CAPPOTTO_FORMA.search(descrizione))


def formalita_da_tipo(title: str):
    """Formalità 1–5 dedotta dal tipo di capo nel titolo, o None.

    Prima i termini specifici (vedi GARMENT_FORMALITY_SPECIFICO), che battono
    la parola generica che li accompagna. Poi la scansione dal più formale al
    più casual, dove "giacca" batte "camicia" in "GIACCA CAMICIA", che è un
    capospalla. Confine di parola per evitare riscontri dentro altre parole.
    """
    t = senza_accenti(title or "").lower()
    for parola, livello in GARMENT_FORMALITY_SPECIFICO.items():
        if re.search(r"\b" + re.escape(parola) + r"\b", t):
            return livello
    for livello in sorted(GARMENT_FORMALITY, reverse=True):
        for parola in GARMENT_FORMALITY[livello]:
            if re.search(r"\b" + re.escape(parola) + r"\b", t):
                return livello
    return None


STYLE_TAG_FORMALITY = {
    "elegante": 4, "casual": 2, "streetwear": 1, "sportivo": 1, "da_mare": 1,
}
 
BRAND_AFFINITY = {
    "nuvolari": ["casual", "elegante"],
    "k-way": ["casual", "sportivo"],
    "the-north-face": ["sportivo", "casual"],
    "napapijri": ["casual"],
    "carhartt-wip": ["casual", "streetwear"],
    "dickies": ["casual", "streetwear"],
    "deus-ex-machina": ["streetwear", "casual"],
    "vans": ["streetwear"],
    "obey": ["streetwear"],
    "stussy": ["streetwear"],
    "stone-island": ["streetwear"],
    "fred-perry": ["casual"],
    "lyle-scott": ["casual"],
    "lee": ["casual"],
    "levi-s": ["casual"],
    "calvin-klein-jeans": ["casual"],
    "tommy-hilfiger": ["casual"],
    "tommy-jeans": ["streetwear", "casual"],
    "guess-jeans": ["casual"],
    "nike": ["sportivo", "streetwear"],
    "adidas": ["sportivo", "streetwear"],
    "adidas-originals": ["streetwear", "casual"],
    "new-balance": ["sportivo", "streetwear"],
    "asics": ["sportivo"],
    "saucony": ["sportivo", "casual"],
    "hoka": ["sportivo"],
    "dr-martens": ["streetwear", "casual"],
    "colmar-originals": ["sportivo", "elegante"],
    "blauer": ["casual"],
    "woolrich": ["casual", "elegante"],
    "c-p-company": ["streetwear"],
    "ralph-lauren": ["elegante", "casual"],
    "goorin-bros": ["casual", "streetwear"],
    "gas": ["casual"],
    "diesel": ["streetwear", "casual"],
    "psycho-bunny": ["casual", "elegante"],
    "weekend-offender": ["streetwear", "casual"],
    "edwin": ["casual"],
    "lacoste": ["casual", "elegante"],
    "propaganda": ["streetwear"],
}
 
COLOR_WORDS = {
    "nero", "black", "bianco", "white", "blu", "blue", "navy", "verde", "green",
    "rosso", "red", "giallo", "yellow", "grigio", "grey", "gray", "marrone",
    "brown", "beige", "rosa", "pink", "viola", "purple", "arancione", "orange",
    "militare", "military", "panna", "cream", "bordeaux", "azzurro", "celeste",
    "oro", "gold", "argento", "silver", "multicolor", "multi", "fango", "tortora",
    "salvia", "khaki", "denim", "olive", "oliva", "avio", "petrolio", "senape",
    "cammello", "camel", "ecru", "sabbia", "sand", "mustard", "burgundy", "teal",
}

In [15]:
# ═══ nuvolari_generation_pipeline/scrape_with_attributes.py ═══════════════════════════════

 
 
# =====================================================================
# FASE 3: derive_attributes — applica le regole di tagging
# =====================================================================
 
# Parole di stile che restano affidabili anche dentro la prosa della scheda.
#
# Tutte le altre non lo sono, ed è misurato: la descrizione attiva in media 2,9
# tag su 10 per capo, e 308 capi ne attivano 6 o più — nessun capo è insieme
# elegante, streetwear, militare, minimal e workwear. Il motivo è che il copy
# non descrive il capo ma l'occasione e gli abbinamenti: "abbinalo a un blazer
# per un look impeccabile" faceva risultare eleganti 1382 capi (48% del
# catalogo), fra cui tre pantaloni della tuta su sei esempi. Lo stesso bermuda
# G-STAR compariva come esempio per sei tag diversi.
#
# Queste invece nominano una tecnica costruttiva o un trattamento del tessuto:
# sono fatti verificabili, non aggettivi, e per giunta dicono cose che una foto
# NON può vedere — che è esattamente ciò che al testo resta da fare ora che la
# revisione visiva copre il registro stilistico di tutto il catalogo.
# Ora vuoto, e il meccanismo resta: era la deroga che lasciava passare dalla
# DESCRIZIONE (non solo dal titolo) le parole tecniche inequivocabili. Tutte
# quelle che c'erano alimentavano tag che non esistono piu': impermeabile,
# membrana, antivento, trekking -> outdoor_tecnico; camouflage, mimetico ->
# military. Nessuno degli otto registri rimasti ha parole altrettanto sicure
# nella prosa, quindi la deroga non ha piu' clienti (vedi
# guida_classificazione.md).
PAROLE_TECNICHE_AFFIDABILI = set()


def compute_style_scores(metadata: dict) -> tuple[dict, set]:
    """Punteggio grezzo (non sogliato) per ognuno dei registri di STYLE_KEYWORDS,
    piu' l'insieme dei registri che hanno un riscontro esplicito nel testo.

    Il punteggio somma quattro sorgenti, di attendibilità decrescente: una
    parola nel titolo 0.8, un materiale riconosciuto 0.5, il marchio 0.4, la
    vestibilità 0.3. Non è né normalizzato né limitato: sul catalogo attuale
    il massimo su un registro è 1.9, ed è il chiamante ad applicare la soglia.

    Il secondo valore è quello che conta di più. Solo il titolo può metterci
    dentro un registro (la deroga sulla descrizione oggi è inerte, vedi
    PAROLE_TECNICHE_AFFIDABILI), e derive_attributes lo usa per decidere chi
    va in revisione visiva: materiale, marchio e vestibilità alzano il
    punteggio ma NON tolgono un capo dalla quarantena, perché sono attributi
    di categorie di capi, non di quel capo. Un punteggio alto significa
    quindi "molti indizi generici concordi", non "so com'è fatto".

    Usato sia da derive_attributes (che applica la soglia 0.5 per il badge
    style_tags/needs_vision_review) sia dalla Fase 3, che ha bisogno del
    segnale continuo per costruire un vettore stile senza perdere
    l'informazione sotto soglia (vedi feature_engineering.py) — lì però solo
    come ripiego per i capi senza vision_review.
    """
    desc = (metadata.get("description_text") or "").lower()
    title = (metadata.get("title") or "").lower()
    composition = (metadata.get("composition") or "").lower()
    fit = metadata.get("fit")
    brand_slug = metadata.get("brand_slug")

    scores = {tag: 0.0 for tag in STYLE_KEYWORDS}
    text_matched_tags = set()  # tag con supporto esplicito da description_text/title

    for tag, keywords in STYLE_KEYWORDS.items():
        # dal titolo qualunque parola vale: lì la parola NOMINA il capo.
        # dalla prosa solo i fatti tecnici: PAROLE_TECNICHE_AFFIDABILI oggi è
        # vuoto, quindi questo secondo ramo non scatta mai e il titolo resta
        # l'unica strada per uscire dalla quarantena testuale.
        dalla_prosa = PAROLE_TECNICHE_AFFIDABILI.intersection(keywords)
        if any(kw in title for kw in keywords) or any(kw in desc for kw in dalla_prosa):
            scores[tag] += 0.8
            text_matched_tags.add(tag)

    for material, info in MATERIAL_SIGNALS.items():
        if material in composition:
            for tag in info["style"]:
                scores[tag] += 0.5

    if fit and fit in FIT_SIGNALS:
        for tag in FIT_SIGNALS[fit]["style"]:
            scores[tag] += 0.3

    if brand_slug and brand_slug in BRAND_AFFINITY:
        for tag in BRAND_AFFINITY[brand_slug]:
            scores[tag] += 0.4

    return scores, text_matched_tags


def derive_attributes(metadata: dict, category_path: Path) -> dict:
    fit = metadata.get("fit")
    composition = (metadata.get("composition") or "").lower()
    combined_text = f"{(metadata.get('description_text') or '').lower()} {(metadata.get('title') or '').lower()}"
    scores, text_matched_tags = compute_style_scores(metadata)

    THRESHOLD = 0.5
    style_tags = {tag: round(s, 2) for tag, s in scores.items() if s >= THRESHOLD}
    # non basta che UN tag superi la soglia: se nessuno dei tag "vincenti" ha
    # supporto esplicito nel testo (description/title) — cioè il punteggio
    # viene solo da materiale/fit/brand, segnali generici che da soli non
    # bastano a caratterizzare lo stile del capo specifico — il prodotto
    # resta comunque da rivedere. Altrimenti capi con description vuota ma
    # materiale comune (es. "98% cotone") e brand generico (es. la linea
    # propria Nuvolari, molto trasversale) risultano "taggati con sicurezza"
    # senza che il testo dica davvero nulla di distintivo su quel capo.
    needs_vision_review = len(style_tags) == 0 or not (set(style_tags) & text_matched_tags)

    # --- formalità ---
    # Il tipo di capo letto dal TITOLO viene prima del percorso di categoria:
    # il percorso è generico per un terzo del catalogo e per tutti gli
    # accessori, e li lasciava tutti sul neutro 3.
    category_prior = formalita_da_tipo(metadata.get("title"))
    if category_prior is None:
        category_parts = [p.lower() for p in category_path.parts]
        category_prior = 3  # default neutro
        for part in category_parts:
            if part in CATEGORY_FORMALITY_PRIOR:
                category_prior = CATEGORY_FORMALITY_PRIOR[part]
                break
 
    if style_tags:
        weighted_sum = sum(STYLE_TAG_FORMALITY.get(t, 3) * w for t, w in style_tags.items())
        style_formality_avg = weighted_sum / sum(style_tags.values())
    else:
        style_formality_avg = category_prior
 
    fit_delta = FIT_SIGNALS.get(fit, {}).get("formality_delta", 0) if fit else 0
    fit_term = max(1, min(5, category_prior + fit_delta))
 
    formality_raw = 0.5 * category_prior + 0.35 * style_formality_avg + 0.15 * fit_term
    formality_score = max(1, min(5, round(formality_raw)))

    # Il tetto per brand si applica per ultimo: è una proprietà del marchio,
    # non del singolo capo, e vince su quello che dicono tipo, stile e fit.
    tetto = BRAND_FORMALITY_TETTO.get(metadata.get("brand_slug"))
    if tetto is not None:
        # Borse e zaini di questi marchi scendono a 1 anche quando il brand ha
        # tetto 2: un piumino tecnico sta ancora con un maglione, una sacca da
        # montagna no — è della stessa famiglia del marsupio, che è già 1.
        if ACCESSORI_CONTENITORE.search(senza_accenti(metadata.get("title") or "").lower()):
            tetto = 1
        formality_score = min(formality_score, tetto)

    # Pavimento per i cappotti. "Giubbotto" non dice niente sul registro, e un
    # caban di lana doppiopetto a sei bottoni finiva a 2 come una windbreaker —
    # da lì il caban blu sopra la tuta. La forma la nomina la descrizione, non
    # il titolo, quindi si guardano tutt'e due (vedi CAPPOTTO_FORMA).
    if e_un_cappotto(metadata):
        formality_score = max(formality_score, CAPPOTTO_FORMALITA_MINIMA)
 
    # --- stagione ---
    # Il tipo di capo viene per primo: è l'unico segnale non ambiguo. Una
    # sciarpa è invernale anche se è di cotone e la scheda non lo dice, e
    # lasciarla in "tutte" la rendeva abbinabile a una t-shirt estiva.
    season = season_da_tipo(metadata.get("title"))
    # Unica deroga al tipo di capo: una sciarpa di lino, cotone o seta e' un
    # capo da mezza stagione, non da inverno. Vale solo per le sciarpe e solo
    # verso il basso di un gradino -- il tipo resta il segnale forte, e nessun
    # tessuto rende una sciarpa estiva. Senza questa deroga il tipo aveva la
    # precedenza assoluta e una sciarpa di lino risultava invernale.
    if season == "inverno" and SCIARPA.search(
            senza_accenti(metadata.get("title") or "").lower()):
        if TESSUTO_LEGGERO.search(composition):
            season = "mezza_stagione"
    if season is None:
        if STAGIONE_ESTIVA_DEL_CAPO.search(combined_text):
            season = "estate"
        elif STAGIONE_INVERNALE_DEL_CAPO.search(combined_text):
            season = "inverno"
        elif "mezza stagione" in combined_text:
            season = "mezza_stagione"
        else:
            for material, info in MATERIAL_SIGNALS.items():
                if material in composition:
                    season = info["season"]
                    break
    if season is None:
        season = "tutte"
 
    # --- pattern ---
    pattern = "tinta_unita"
    for p, keywords in PATTERN_KEYWORDS.items():
        if any(kw in combined_text for kw in keywords):
            pattern = p
            break
 
    return {
        "style_tags": style_tags,
        "formality_score": formality_score,
        "season": season,
        "pattern": pattern,
        "needs_vision_review": needs_vision_review,
    }

In [16]:
print("registri riconosciuti dal testo:", list(STYLE_KEYWORDS))
print()

punteggi, con_riscontro = compute_style_scores(meta)
print(meta["title"])
print("  punteggi grezzi   :", {k: round(v, 2) for k, v in punteggi.items() if v})
print("  con riscontro nel testo:", con_riscontro or "nessuno")
print("  → sopra soglia 0,5 :", {k: round(v, 2) for k, v in punteggi.items() if v >= 0.5})
print()
print("come è finito dentro metadata.json:")
print("  style_tags          :", meta.get("style_tags"))
print("  needs_vision_review :", meta.get("needs_vision_review"),
      "  ← nessun tag sopra soglia ha riscontro nel testo: quarantena")
print("  formality_score     :", meta.get("formality_score"), "(dalle regole testuali)")
print("  season / pattern    :", meta.get("season"), "/", meta.get("pattern"))

registri riconosciuti dal testo: ['sportivo', 'elegante', 'streetwear', 'da_mare', 'casual']

CAMICIA UOMO LEE CAMP SHIRT SOY BEAN
  punteggi grezzi   : {'elegante': 1.0, 'casual': 1.7}
  con riscontro nel testo: nessuno
  → sopra soglia 0,5 : {'elegante': 1.0, 'casual': 1.7}

come è finito dentro metadata.json:
  style_tags          : {'elegante': 1.0, 'casual': 1.7}
  needs_vision_review : True   ← nessun tag sopra soglia ha riscontro nel testo: quarantena
  formality_score     : 3 (dalle regole testuali)
  season / pattern    : estate / righe


## 2.2 La guida — `guida_classificazione.md`

Il documento che insegna al modello come classificare. Non sta dentro il codice **di proposito**:
è un testo di criteri che si legge e si corregge come si corregge una prosa — quando `casual`
ricomincia a finire ovunque, si aggiusta lì — e viaggia come `system_instruction`, quindi vale
per ogni chiamata senza essere ripetuto in ogni richiesta.

È lo stesso schema di `generation_rules.md` per le foto: le regole in un `.md`, il codice solo
per i dati del singolo capo.

La regola più importante che contiene non riguarda lo stile ma **cosa non leggere**: le schede
del negozio hanno quasi sempre una sezione "Come abbinarla?", e quelle frasi descrivono *altri*
capi. Un pantalone della tuta consigliato con un blazer resta un pantalone della tuta.

Le costanti della revisione visiva, e la guida caricata dal disco. `CATALOGO` qui non si
ridefinisce: nel modulo punta al catalogo dentro `nuvolari_db/`, nel notebook vale quello
relativo alla radice del repo definito all'inizio.

In [17]:
# ═══ nuvolari_generation_pipeline/vision_review.py ═══════════════════════════════

MODEL = "gemini-3.5-flash"
FONTE = "foto+descrizione-5tag"   # marca la guida attuale; vedi --rifai
LATO_MASSIMO = 640          # la foto serve a leggere lo stile, non i dettagli
PAUSA = 0.3                 # fra le chiamate
TENTATIVI = 3

STYLE_TAGS = [
    "sportivo", "casual", "elegante", "streetwear", "da_mare",
]

GUIDA = CODICE / "guida_classificazione.md"

# La guida sta in un file a parte, non qui dentro: e' un documento di criteri
# che si legge e si corregge come un testo (quando "casual" finisce di nuovo
# ovunque, si aggiusta li'), e va al modello come system_instruction — quindi
# vale per ogni chiamata senza essere ripetuta in ogni richiesta.
ISTRUZIONI = GUIDA.read_text(encoding="utf-8")

SCHEDA = """TITOLO: {titolo}

DESCRIZIONE DEL NEGOZIO:
{descrizione}

SCHEDA TECNICA:
{tecnica}

ABBINAMENTI CONSIGLIATI DALLA SCHEDA (sono ALTRI capi — servono solo a capire
il registro d'uso, vedi la guida): {abbinamenti}"""

In [18]:
print("guida:", GUIDA.name, "-", len(ISTRUZIONI), "caratteri, mandata come system_instruction")
print("modello:", MODEL, "| fonte che marca i giudizi:", FONTE)
print()
print("indice della guida:")
for riga in ISTRUZIONI.splitlines():
    if riga.startswith("#"):
        livello = len(riga) - len(riga.lstrip("#"))
        print("   " * (livello - 1) + riga.lstrip("# "))

guida: guida_classificazione.md - 11893 caratteri, mandata come system_instruction
modello: gemini-3.5-flash | fonte che marca i giudizi: foto+descrizione-5tag

indice della guida:
Guida alla classificazione dei capi
   Che cosa NON stai facendo
   La trappola: i consigli di abbinamento parlano di ALTRI capi
   `style_scores` — cinque registri, e la somma fa 1
      Come dosare
      Che cosa significa ciascun registro
      Due cose da non fare
   `season` — decidila dal TESSUTO, non dal tono del testo
   `formality` — segui il procedimento, poi decidi tu
      1. Parti dal tipo di capo
      2. Il termine specifico batte quello generico
      3. Correggi con la vestibilità
      4. Il registro del marchio è un tetto, non un valore
      5. Pavimento per i cappotti
      6. Guarda con che cosa il negozio dice di portarlo
      7. Decidi tu
   `pattern` — guardalo nella FOTO
      Che cosa NON è una fantasia
      Quando usare `stampato`
      Calzature e accessori
   Formato della ris

In [19]:
# Il formato che la guida impone alla risposta, letto dalla guida stessa.
testo = ISTRUZIONI
i = testo.index("## Formato della risposta")
print(testo[i:i + 520])

## Formato della risposta

Rispondi **solo** con questo oggetto JSON, senza testo prima o dopo:

```json
{"style_scores": {"tag": punteggio},
 "formality": 1-5,
 "formality_perche": "poche parole",
 "season": "estate|inverno|mezza_stagione|tutte",
 "pattern": "tinta_unita|righe|quadri|camouflage|stampato|animalier"}
```



## 2.3 La chiamata

### `testo_scheda(metadata)`

- **Input** — il `metadata.json`.
- **Processo** — impagina il materiale testuale nella versione migliore disponibile:
  `descrizione_completa` (presa dalla pagina prodotto, contiene il "Come abbinarla?") e, come
  ripiego, la più corta `description_text`; poi scheda tecnica e abbinamenti suggeriti — questi
  ultimi **dichiarati come tali**, perché il modello sappia che parlano di altri capi.
- **Output** — la stringa che accompagna la foto nella richiesta.

### `foto_ridotta(product_dir)`

- **Input** — la cartella del prodotto.
- **Processo** — prende la prima foto leggibile, la riduce a **640 px** di lato lungo e la
  ricomprime in JPEG all'85%. La foto serve a leggere il **registro**, non i dettagli: mandarla
  a piena risoluzione costerebbe di più senza dire di più.
- **Output** — i byte JPEG, o `None` se non c'è una foto apribile.

### `revisiona(client, metadata, product_dir)`

- **Input** — il client Gemini, il `metadata.json`, la cartella del prodotto.
- **Processo** — costruisce la richiesta (foto + scheda), con la guida come `system_instruction`,
  `temperature=0`, `response_mime_type="application/json"` e **`thinking_budget=0`**: il modello
  di default "ragiona", e i token di ragionamento mangiavano il budget di output facendo arrivare
  le risposte **troncate a metà JSON**. Per una classificazione a cinque etichette il
  ragionamento non compra nulla. Fino a 3 tentativi con attesa crescente.
- **Output** — `(dati_validati, None)` oppure `(None, errore)`.

### `_primo_oggetto_json(testo)`

- **Input** — la risposta grezza del modello.
- **Processo** — estrae il **primo oggetto JSON bilanciato**, e se è troncato lo chiude. Il
  conteggio delle graffe ignora quelle dentro le stringhe: contarle alla cieca sbagliava sulle
  descrizioni che contengono virgolette.
- **Output** — una stringa JSON parsabile.

### `valida_risposta(dati)`

- **Input** — il dizionario appena parsato.
- **Processo** — tiene solo i cinque registri previsti, converte i valori, li limita a 1,0 e
  scarta gli zeri; accetta `formality` solo se è un numero fra 1 e 5 (e ne conserva la
  motivazione, che non serve alla pipeline ma permette di controllare a campione un giudizio
  ormai delegato); accetta `season` e `pattern` solo se sono valori dell'elenco chiuso.
- **Output** — il blocco pulito da salvare, o `None` se la risposta è inutilizzabile.

### `main()`

- **Input** — gli argomenti da riga di comando.
- **Processo** — sceglie l'ambito (`--tutti` = tutto il catalogo, altrimenti solo la quarantena),
  **salta i capi già revisionati** (quindi il run è riprendibile), con `--rifai` rifà quelli
  giudicati sotto una guida precedente, e scrive il risultato dentro `metadata.json` sotto la
  chiave `vision_review`, **separata** dagli attributi testuali — così il ricalcolo degli
  attributi può girare quante volte vuole senza cancellare il giudizio pagato.
- **Output** — i `metadata.json` aggiornati, e a schermo il conto di revisionati ed errori.

In [20]:
# ═══ nuvolari_generation_pipeline/generate_outfit_images.py ═══════════════════════════════

def load_api_key() -> str:
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if key:
        return key
    env_file = CODICE / ".env"
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line.startswith("GEMINI_API_KEY=") or line.startswith("GOOGLE_API_KEY="):
                return line.split("=", 1)[1].strip().strip('"').strip("'")
    sys.exit(
        "Chiave API non trovata.\n"
        "Impostala con:  export GEMINI_API_KEY=...\n"
        f"oppure scrivila in {env_file} come  GEMINI_API_KEY=..."
    )

In [21]:
# ═══ nuvolari_generation_pipeline/vision_review.py ═══════════════════════════════

def testo_scheda(metadata: dict) -> str:
    """Il materiale che descrive il capo, nella versione migliore disponibile.

    descrizione_completa viene dalla pagina prodotto e contiene la sezione
    "Come abbinarla?"; description_text e' l'og:description, piu' corta e
    incoerente — resta come ripiego per i capi non ancora ripassati.
    """
    descrizione = (metadata.get("descrizione_completa")
                   or metadata.get("description_text") or "(nessuna descrizione)")
    abbinamenti = [a.get("testo", "") for a in (metadata.get("abbinamenti_suggeriti") or [])]
    return SCHEDA.format(
        titolo=metadata.get("title") or "(senza titolo)",
        descrizione=descrizione[:4000],
        tecnica=metadata.get("scheda_tecnica") or "(non disponibile)",
        abbinamenti=", ".join(a for a in abbinamenti if a) or "(nessuno)")


def foto_ridotta(product_dir: Path):
    """La prima foto leggibile della galleria, ridotta e ricompressa JPEG."""
    for p in sorted(product_dir.iterdir()):
        if p.suffix.lower() not in (".jpg", ".jpeg", ".png"):
            continue
        try:
            with Image.open(p) as im:
                im = im.convert("RGB")
                im.thumbnail((LATO_MASSIMO, LATO_MASSIMO), Image.LANCZOS)
                buf = io.BytesIO()
                im.save(buf, "JPEG", quality=85)
                return buf.getvalue()
        except Exception:
            continue
    return None


def _primo_oggetto_json(testo: str) -> str:
    """Il primo oggetto JSON bilanciato del testo, chiuso se troncato.

    Serve perche' il modello ogni tanto aggiunge righe dopo l'oggetto, o lo
    tronca a meta'. Il conteggio delle graffe ignora quelle dentro le stringhe:
    contarle alla cieca sbagliava sulle descrizioni che contengono virgolette.
    """
    testo = (testo or "").strip()
    inizio = testo.find("{")
    if inizio < 0:
        return testo
    prof, in_stringa, escape = 0, False, False
    for i in range(inizio, len(testo)):
        c = testo[i]
        if escape:
            escape = False
        elif c == "\\":
            escape = True
        elif c == '"':
            in_stringa = not in_stringa
        elif not in_stringa:
            if c == "{":
                prof += 1
            elif c == "}":
                prof -= 1
                if prof == 0:
                    return testo[inizio:i + 1]
    # troncato: chiudo quello che resta aperto
    coda = testo[inizio:].rstrip().rstrip(",")
    if in_stringa:
        coda += '"'
    return coda + "}" * prof


def valida_risposta(dati: dict):
    """Normalizza e valida il JSON del modello; None se inutilizzabile."""
    if not isinstance(dati, dict):
        return None
    grezzi = dati.get("style_scores")
    if not isinstance(grezzi, dict):
        return None
    scores = {}
    for tag, v in grezzi.items():
        tag = str(tag).strip().lower()
        if tag in STYLE_TAGS:
            try:
                v = float(v)
            except (TypeError, ValueError):
                continue
            if v > 0:
                scores[tag] = round(min(v, 1.0), 2)
    if not scores:
        return None
    out = {"style_scores": scores}
    f = dati.get("formality")
    if isinstance(f, (int, float)) and 1 <= f <= 5:
        out["formality"] = int(round(f))
        # La motivazione non serve alla pipeline: serve a poter controllare a
        # campione un giudizio che ora e' delegato, senza rifare la chiamata.
        perche = str(dati.get("formality_perche") or "").strip()
        if perche:
            out["formality_perche"] = perche[:160]
    s = dati.get("season")
    if s in ("estate", "inverno", "mezza_stagione", "tutte"):
        out["season"] = s
    p = dati.get("pattern")
    if p in ("tinta_unita", "righe", "quadri", "camouflage", "stampato", "animalier"):
        out["pattern"] = p
    return out


def revisiona(client, metadata: dict, product_dir: Path):
    foto = foto_ridotta(product_dir)
    if foto is None:
        return None, "nessuna foto leggibile"
    parti = [
        types.Part.from_bytes(data=foto, mime_type="image/jpeg"),
        testo_scheda(metadata),
    ]
    # thinking_budget=0: gemini-3.5-flash di default "ragiona", e i token di
    # ragionamento mangiano il budget di output — le risposte arrivavano
    # TRONCATE a metà JSON ('..."pattern": "tinta_unita' senza graffe finali).
    # Per una classificazione a 10 etichette il ragionamento non compra nulla.
    config = types.GenerateContentConfig(
        system_instruction=ISTRUZIONI,
        response_mime_type="application/json", temperature=0.0,
        max_output_tokens=1500,
        thinking_config=types.ThinkingConfig(thinking_budget=0))
    ultimo_errore = None
    for tentativo in range(1, TENTATIVI + 1):
        try:
            resp = client.models.generate_content(model=MODEL, contents=parti, config=config)
            dati = valida_risposta(json.loads(_primo_oggetto_json(resp.text)))
            if dati is None:
                return None, f"risposta non valida: {resp.text[:120]}"
            return dati, None
        except Exception as e:
            ultimo_errore = str(e)[:160]
            time.sleep(2 * tentativo)
    return None, ultimo_errore


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--prova", action="store_true", help="solo 3 capi, risposte in chiaro")
    ap.add_argument("--limite", type=int, default=None)
    ap.add_argument("--rifai", action="store_true",
                    help="rifa' anche i capi gia' revisionati sotto una guida precedente "
                         "(vision_review con fonte diversa da quella attuale): il giudizio "
                         "vecchio nasceva senza descrizione completa e senza il procedimento "
                         "sulla formalita', e la pipeline lo ignora comunque")
    ap.add_argument("--tutti", action="store_true",
                    help="rivede TUTTO il catalogo, non solo la quarantena: il segnale "
                         "visivo si somma a quello testuale per ogni capo (vedi "
                         "build_style_vector), cosi' un capo ben descritto e uno muto "
                         "stanno sulla stessa scala")
    args = ap.parse_args()

    client = genai.Client(api_key=load_api_key(),
                          http_options=types.HttpOptions(timeout=60_000))

    da_fare = []
    for meta_path in sorted(CATALOGO.rglob("metadata.json")):
        m = json.loads(meta_path.read_text(encoding="utf-8"))
        fatto = m.get("vision_review") or {}
        if fatto and not (args.rifai and fatto.get("fonte") != FONTE):
            continue
        if args.tutti or m.get("needs_vision_review"):
            da_fare.append((meta_path, m))
    ambito = "nel catalogo" if args.tutti else "in quarantena"
    print(f"[*] capi {ambito} senza revisione: {len(da_fare)}", flush=True)

    if args.prova:
        da_fare = da_fare[:3]
    elif args.limite:
        da_fare = da_fare[:args.limite]

    ok = errori = 0
    for i, (meta_path, m) in enumerate(da_fare, 1):
        dati, err = revisiona(client, m, meta_path.parent)
        if err:
            errori += 1
            print(f"  [{i}/{len(da_fare)}] ERRORE {m.get('relpath','?')[:50]}: {err}", flush=True)
        else:
            ok += 1
            m["vision_review"] = {
                "model": MODEL,
                "fonte": FONTE,
                "quando": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
                **dati,
            }
            meta_path.write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding="utf-8")
            if args.prova or i % 25 == 0 or i == len(da_fare):
                tags = ", ".join(f"{t}={v}" for t, v in dati["style_scores"].items())
                print(f"  [{i}/{len(da_fare)}] {(m.get('title') or '')[:44]:44s} -> {tags}", flush=True)
        time.sleep(PAUSA)

    print(f"\n[OK] revisionati {ok}, errori {errori}", flush=True)
    if ok:
        print("Prossimi passi: recompute_style_vectors -> run_clustering -> pool -> ripesca_orfane")

In [22]:
# La richiesta vera per il capo di esempio, costruita ma NON mandata: qui non si
# chiama l'API (costa, e la risposta di questo capo è già salvata sul disco).
scheda = testo_scheda(meta)
foto = foto_ridotta(cartella)

print("PARTE 1 — la foto:", len(foto) // 1024, "KB in JPEG, lato massimo", LATO_MASSIMO, "px")
print("PARTE 2 — la scheda:")
print("   " + scheda[:600].replace(chr(10), chr(10) + "   "), "...")
print()
print("configurazione: system_instruction=guida_classificazione.md,",
      "temperature=0.0, response_mime_type=application/json, thinking_budget=0,",
      f"max_output_tokens=1500, tentativi={TENTATIVI}")

PARTE 1 — la foto: 18 KB in JPEG, lato massimo 640 px
PARTE 2 — la scheda:
   TITOLO: CAMICIA UOMO LEE CAMP SHIRT SOY BEAN
   
   DESCRIZIONE DEL NEGOZIO:
   Descrizione
   L'autentico spirito della cultura denim americana, la robustezza dei capi di matrice workwear e la freschezza del casualwear estivo si fondono armoniosamente nella collezione di
   camicie uomo Lee
   . Il brand pioniere dell'abbigliamento da lavoro reinterpreta la camiceria maschile puntando su tessuti leggeri, motivi a righe dal sapore rétro e una vestibilità rilassata, ideale per la stagione calda.
   Design, Silhouette e Caratteristiche Principali
   Il Colletto Cubano (Resort Collar) e lo Stile Rétro
   : La collezione es ...

configurazione: system_instruction=guida_classificazione.md, temperature=0.0, response_mime_type=application/json, thinking_budget=0, max_output_tokens=1500, tentativi=3


In [23]:
# Quello che è tornato, e che sta dentro metadata.json:
print(json.dumps(meta["vision_review"], ensure_ascii=False, indent=2))

{
  "model": "gemini-3.5-flash",
  "fonte": "foto+descrizione-5tag",
  "quando": "2026-08-19",
  "style_scores": {
    "casual": 0.6,
    "da_mare": 0.4
  },
  "formality": 2,
  "formality_perche": "Camicia a maniche corte con colletto cubano, tessuto leggero in lino e viscosa, stile vacanziero rilassato",
  "season": "estate",
  "pattern": "righe"
}


In [24]:
# I due guardrail, provati su risposte sporche.
troncata = '{"style_scores": {"casual": 0.6, "elegante": 0.4}, "formality": 3, "pattern": "tinta_unita'
print("risposta troncata a metà →", _primo_oggetto_json(troncata))
print()

sporca = {"style_scores": {"casual": 1.5, "inventato": 0.7, "elegante": 0.2, "sportivo": 0},
          "formality": 9, "season": "boh", "pattern": "righe"}
print("risposta con valori fuori regola:")
print("   in :", sporca)
print("   out:", valida_risposta(sporca))
print("   (1,5 limitato a 1,0 · tag inventato scartato · zero scartato ·")
print("    formalità 9 e stagione 'boh' rifiutate · pattern valido tenuto)")

risposta troncata a metà → {"style_scores": {"casual": 0.6, "elegante": 0.4}, "formality": 3, "pattern": "tinta_unita"}

risposta con valori fuori regola:
   in : {'style_scores': {'casual': 1.5, 'inventato': 0.7, 'elegante': 0.2, 'sportivo': 0}, 'formality': 9, 'season': 'boh', 'pattern': 'righe'}
   out: {'style_scores': {'casual': 1.0, 'elegante': 0.2}, 'pattern': 'righe'}
   (1,5 limitato a 1,0 · tag inventato scartato · zero scartato ·
    formalità 9 e stagione 'boh' rifiutate · pattern valido tenuto)


In [25]:
# Che cosa ha prodotto la revisione su tutto il catalogo.
t0 = time.time()
revisioni = []
for p in CATALOGO.rglob("metadata.json"):
    m = json.loads(p.read_text(encoding="utf-8"))
    v = m.get("vision_review") or {}
    revisioni.append({
        "revisionato": bool(v),
        "fonte": v.get("fonte"), "model": v.get("model"), "quando": v.get("quando"),
        "n_registri": len(v.get("style_scores") or {}),
        "formalita_visiva": v.get("formality"),
        "formalita_testuale": m.get("formality_score"),
        "quarantena_testuale": bool(m.get("needs_vision_review")),
        "pattern": v.get("pattern"),
    })
rev = pd.DataFrame(revisioni)
print(f"{len(rev)} capi scansionati in {time.time()-t0:.1f}s")
print(f"revisionati: {rev['revisionato'].sum()} ({rev['revisionato'].mean()*100:.1f}%)"
      f" | in quarantena testuale: {rev['quarantena_testuale'].sum()}")
print()
print("modello e guida:")
print(rev.groupby(["model", "fonte"], dropna=False).size().to_string())
print()
print("registri usati per capo (la guida dice: di norma uno o due):")
print(rev["n_registri"].value_counts().sort_index().to_string())

2901 capi scansionati in 0.5s
revisionati: 2901 (100.0%) | in quarantena testuale: 2749

modello e guida:
model             fonte                
gemini-3.5-flash  foto+descrizione-5tag    2901

registri usati per capo (la guida dice: di norma uno o due):
n_registri
1     246
2    2150
3     498
4       7


In [26]:
# Quanto la formalità vista si discosta da quella dedotta dal testo: è la misura
# di che cosa ha comprato la revisione visiva.
confronto = rev.dropna(subset=["formalita_visiva", "formalita_testuale"])
scarto = (confronto["formalita_visiva"] - confronto["formalita_testuale"])
print("scarto (visiva − testuale):")
print(scarto.value_counts().sort_index().to_string())
print()
print(f"d'accordo: {(scarto == 0).mean()*100:.1f}%  |  diverse di un livello: {(scarto.abs() == 1).mean()*100:.1f}%"
      f"  |  di due o più: {(scarto.abs() >= 2).mean()*100:.1f}%")
print()
print("tabella incrociata (righe = testo, colonne = foto):")
print(pd.crosstab(confronto["formalita_testuale"], confronto["formalita_visiva"]).to_string())

scarto (visiva − testuale):
-3       2
-2      67
-1     498
 0    2251
 1      82
 2       1

d'accordo: 77.6%  |  diverse di un livello: 20.0%  |  di due o più: 2.4%

tabella incrociata (righe = testo, colonne = foto):
formalita_visiva      1     2    3   4
formalita_testuale                    
1                   153     6    1   0
2                   122  1286   60   0
3                    53   343  789  16
4                     0    14   32  23
5                     0     2    0   1


### Le chiamate vere

Questa è l'unica cella che chiama Gemini, e quindi **l'unica che costa**. Parte solo con
`ESEGUI_REVISIONE_VISIVA = True` ed è tarata su **2 capi**, scelti fra quelli non ancora
revisionati; con `SCRIVI_SU_DISCO = True` riscrive anche il loro `metadata.json`, come fa
`main()`. Tutte le 2901 revisioni sono già state pagate e stanno nel repo: rifarle serve solo
se il catalogo cresce o se cambia `guida_classificazione.md`.

In [27]:
QUANTI_CAPI      = 2       # quante schede mandare a Gemini in questa prova
SCRIVI_SU_DISCO  = False   # True = aggiorna anche i metadata.json, come fa main()

if not ESEGUI_REVISIONE_VISIVA:
    da_rivedere = [p for p in sorted(CATALOGO.rglob("metadata.json"))
                   if not (json.loads(p.read_text(encoding="utf-8")).get("vision_review") or {})]
    print("ESEGUI_REVISIONE_VISIVA = False: nessuna chiamata a Gemini, nessun costo.")
    print(f"   capi senza revisione nel catalogo: {len(da_rivedere)}")
elif not GENAI:
    print("google-genai non è installato: pip install google-genai")
else:
    client = genai.Client(api_key=load_api_key(), http_options=types.HttpOptions(timeout=60_000))
    fatti = 0
    for meta_path in sorted(CATALOGO.rglob("metadata.json")):
        if fatti >= QUANTI_CAPI:
            break
        m = json.loads(meta_path.read_text(encoding="utf-8"))
        if m.get("vision_review"):
            continue                      # già revisionato: si salta, come in main()
        dati, errore = revisiona(client, m, meta_path.parent)
        fatti += 1
        if errore:
            print(f"   ERRORE {m.get('relpath','?')[:50]}: {errore}")
            continue
        print(f"   {m.get('title','')[:52]:54s} → {dati}")
        if SCRIVI_SU_DISCO:
            m["vision_review"] = {"model": MODEL, "fonte": FONTE,
                                  "quando": datetime.now(timezone.utc).strftime("%Y-%m-%d"), **dati}
            meta_path.write_text(json.dumps(m, ensure_ascii=False, indent=2), encoding="utf-8")
        time.sleep(PAUSA)
    print(f"   chiamate fatte: {fatti}")

ESEGUI_REVISIONE_VISIVA = False: nessuna chiamata a Gemini, nessun costo.
   capi senza revisione nel catalogo: 0


---
# 3. I vettori del capo — Fase 3

Il punto di partenza è una cartella per prodotto dentro il catalogo:
le foto e un `metadata.json` con titolo, descrizione, composizione, prezzo, e il
giudizio della revisione visiva (`vision_review`).

Valutare un capo significa ridurlo a **due vettori confrontabili**:

1. **il colore**, estratto dalla foto più pulita della galleria e portato in spazio Lab;
2. **lo stile**, cioè la ripartizione fra i cinque registri, più la formalità e la stagione.

Tutto quello che segue — punteggi, cluster, accoppiamento — lavora solo su questi numeri:
il capo, da qui in poi, *è* la sua riga di `features.parquet`.

In [28]:
# L'uscita di questa fase, se è già sul disco. Se manca, la si rifà con
# build_feature_table qui sotto (ESEGUI_FEATURES).
features = pd.read_parquet(FEATURES) if FEATURES.exists() else None
print("prodotti valutati:", len(features) if features is not None else "features.parquet non c'è ancora")

famiglie = {
    "identità":  ["relpath", "title", "brand_slug", "url"],
    "colore":    ["L", "a", "b", "color_palette", "background_fraction", "representative_image"],
    "stile":     [c for c in (features.columns if features is not None else []) if c.startswith("style_")],
    "formalità": ["formality_norm"],
    "stagione":  [c for c in (features.columns if features is not None else []) if c.startswith("season_")],
    "qualità":   ["needs_vision_review", "pattern"],
    "foto":      ["display_image", "all_images"],
}
if features is not None:
    for nome, cols in famiglie.items():
        print(f"  {nome:10s} {', '.join(c for c in cols if c in features.columns)}")

prodotti valutati: 2901
  identità   relpath, title, brand_slug, url
  colore     L, a, b, color_palette, background_fraction, representative_image
  stile      style_sportivo, style_casual, style_elegante, style_streetwear, style_da_mare
  formalità  formality_norm
  stagione   season_estate, season_inverno, season_mezza_stagione, season_tutte
  qualità    needs_vision_review, pattern
  foto       display_image, all_images


## 3.1 Il colore

### `pick_representative_image(image_paths)`

- **Input** — la lista delle foto di un prodotto (tutte quelle leggibili nella sua cartella).
- **Processo** — misura per ognuna la frazione di **sfondo uniforme** (flood-fill dai bordi) e
  tiene quella che ne ha di più. È tipicamente lo scatto still-life su fondo bianco: la foto
  in cui il colore del capo è meno inquinato da pelle, capelli, ambiente o zoom ravvicinati.
- **Output** — un solo `Path`, la foto da cui si estrarrà il colore.

### `dominant_colors_lab(image_path, n_colors=3)`

- **Input** — il percorso della foto rappresentativa.
- **Processo** — riduce l'immagine a 220 px di lato lungo, individua lo sfondo con un
  flood-fill dai quattro angoli **a tolleranza adattiva** e lo esclude, converte i pixel
  rimasti in Lab e ci esegue un **K-Means a 3 colori**. Se il primo piano risulta troppo
  piccolo (capo quasi bianco su fondo bianco) ricade sull'immagine intera invece di
  restituire il colore di due pixel.
- **Output** — `{"dominant_lab": [L,a,b], "palette": [{"lab": [...], "proportion": ...}, ...],
  "background_fraction": ...}`. La palette è ordinata per proporzione decrescente: è quella
  che la fase 5 confronterà, pesando ogni colore per quanto occupa.

In [29]:
# ═══ nuvolari_generation_pipeline/feature_engineering.py ═══════════════════════════════

STYLE_TAGS = [
    "sportivo", "casual", "elegante", "streetwear", "da_mare",
]
SEASONS = ["estate", "inverno", "mezza_stagione", "tutte"]

MAX_DIM = 220                 # ridimensiona le foto prima del k-means (velocità)
N_COLOR_CLUSTERS = 3
BG_TOLERANCE = 18             # distanza euclidea RGB per il flood-fill dello sfondo
MIN_FOREGROUND_FRAC = 0.03    # sotto questa soglia il background removal si considera fallito

IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png")


# =====================================================================
# Vettore colore (Lab space, k-means, sfondo escluso)
# =====================================================================

def _load_rgb_array(image_path: Path, max_dim: int = MAX_DIM) -> np.ndarray:
    with Image.open(image_path) as im:
        im = im.convert("RGB")
        im.thumbnail((max_dim, max_dim), Image.LANCZOS)
        return np.asarray(im)


def _background_mask(rgb: np.ndarray, tolerance: float = BG_TOLERANCE) -> np.ndarray:
    """Flood-fill dai quattro angoli per isolare lo sfondo studio (bianco/uniforme).

    Usa la distanza euclidea RGB dal colore dell'angolo come mappa per il
    flood-fill: la maschera resta connessa al bordo e non "salta" a zone
    bianche isolate dentro il capo (es. stampe/loghi/etichette bianche),
    perché quelle non sono topologicamente collegate allo sfondo.
    """
    h, w = rgb.shape[:2]
    img_f = rgb.astype(float)
    corners = [(0, 0), (0, w - 1), (h - 1, 0), (h - 1, w - 1)]
    mask = np.zeros((h, w), dtype=bool)
    for cy, cx in corners:
        seed_color = img_f[cy, cx]
        diff = np.linalg.norm(img_f - seed_color, axis=-1)
        mask |= flood(diff, (cy, cx), tolerance=tolerance)
    return mask


# Quota di capo sotto la quale il flood-fill si sta mangiando il prodotto e non
# solo lo sfondo. La mediana del catalogo è ~29% di capo; sotto il 15% non è più
# una foto ravvicinata, è uno scontorno sbagliato.
FOREGROUND_PLAUSIBILE = 0.15
# ...e sopra la quale non è più un capo su fondale ma un dettaglio ravvicinato.
# Una foto di catalogo ha sempre dello sfondo: se non se ne trova, il flood-fill
# non è riuscito a propagarsi, non è sparito il fondale.
FOREGROUND_MASSIMO = 0.80
# Tolleranze provate in ordine, dalla più larga alla più stretta.
TOLLERANZE_SFONDO = (BG_TOLERANCE, 14, 12, 10, 8, 6, 5, 4, 3)


def _maschera_sfondo_adattiva(rgb: np.ndarray):
    """Sceglie la tolleranza che lascia in piedi un capo di dimensioni credibili.

    Con la tolleranza fissa a 18 un capo pallido rientra nella distanza dal
    bianco e viene inghiottito insieme al fondale. Misurato su una t-shirt
    celeste: 96% dell'immagine classificato come sfondo (mediana del catalogo
    71%), e del 4% rimasto la maggior parte era la stampa gialla sul petto —
    così il colore dominante a registro diventava un giallo pallido e la
    maglia risultava abbinabile al nero come se fosse panna.

    Il criterio ha però bisogno di un tetto oltre che di un pavimento. Con il
    solo pavimento, su 26 capi la tolleranza scendeva fino a 3, il riempimento
    non si propagava più e la maschera restava vuota: sfondo 0,00, capo "100%
    dell'immagine", e il colore calcolato sul bianco del fondale. Un difetto
    speculare a quello che si voleva togliere.

    Se nessuna tolleranza produce una quota credibile si prende la meno
    sbagliata — quella che ci va più vicino — invece della più stretta.
    """
    migliore, distanza_migliore = None, float("inf")
    for tolleranza in TOLLERANZE_SFONDO:
        maschera = _background_mask(rgb, tolleranza)
        capo = (~maschera).mean()
        if FOREGROUND_PLAUSIBILE <= capo <= FOREGROUND_MASSIMO:
            return maschera
        distanza = (FOREGROUND_PLAUSIBILE - capo if capo < FOREGROUND_PLAUSIBILE
                    else capo - FOREGROUND_MASSIMO)
        if distanza < distanza_migliore:
            migliore, distanza_migliore = maschera, distanza
    return migliore


def _background_fraction(image_path: Path) -> float:
    # stessa risoluzione usata poi per l'estrazione colore (MAX_DIM): a risoluzioni
    # più basse i pattern fini (righe, micro-texture) si sfocano in un campo quasi
    # uniforme e il flood-fill li scambia per sfondo, gonfiando il punteggio di
    # foto ravvicinate che in realtà non hanno sfondo studio in inquadratura.
    rgb = _load_rgb_array(image_path, max_dim=MAX_DIM)
    return float(_background_mask(rgb).mean())


def is_readable(image_path: Path) -> bool:
    """Verifica che il file sia un'immagine leggibile per intero.

    Alcuni download sono troncati e sollevano OSError solo alla decodifica dei
    pixel, non all'apertura: vanno individuati qui, prima che facciano fallire
    l'estrazione colore o, peggio, finiscano fra le foto inviate all'API.
    """
    try:
        with Image.open(image_path) as im:
            im.load()
        return True
    except Exception:
        return False


def pick_representative_image(image_paths: list) -> Path:
    """Sceglie, tra le foto della galleria, quella con più sfondo uniforme.

    Tipicamente è il flat-lay/ghost-shot più pulito — il migliore per il
    colore reale del capo — mentre scatti indossati o di dettaglio hanno
    sfondo meno uniforme (pelle, capelli, ambiente, zoom ravvicinato).
    """
    if len(image_paths) == 1:
        return image_paths[0]
    scored = [(p, _background_fraction(p)) for p in image_paths]
    scored.sort(key=lambda t: t[1], reverse=True)
    return scored[0][0]


def dominant_colors_lab(image_path: Path, n_colors: int = N_COLOR_CLUSTERS) -> dict:
    """K-means in Lab space sui pixel del capo (sfondo escluso).

    Ritorna il colore dominante e la palette completa con le proporzioni,
    più la frazione di sfondo rilevata (utile per il controllo qualità).
    """
    rgb = _load_rgb_array(image_path)
    mask = _maschera_sfondo_adattiva(rgb)
    foreground = rgb[~mask]

    if foreground.shape[0] < MIN_FOREGROUND_FRAC * rgb.shape[0] * rgb.shape[1]:
        # background removal fallito (es. capo quasi tutto bianco) -> usa l'immagine intera
        foreground = rgb.reshape(-1, 3)

    lab_pixels = rgb2lab((foreground / 255.0).reshape(-1, 1, 3)).reshape(-1, 3)

    k = min(n_colors, len(np.unique(foreground.reshape(-1, 3), axis=0)))
    k = max(k, 1)
    km = KMeans(n_clusters=k, n_init=4, random_state=0).fit(lab_pixels)

    counts = np.bincount(km.labels_, minlength=k)
    order = np.argsort(counts)[::-1]

    palette = [
        {"lab": km.cluster_centers_[i].tolist(), "proportion": float(counts[i] / counts.sum())}
        for i in order
    ]
    return {
        "dominant_lab": palette[0]["lab"],
        "palette": palette,
        "background_fraction": float(mask.mean()),
    }

In [30]:
# Le foto del capo di esempio caricato all'inizio.
foto = sorted(p for p in cartella.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png") and is_readable(p))

print(meta["title"])
print("foto disponibili:", [p.name for p in foto])

rappresentativa = pick_representative_image(foto)
print("rappresentativa :", rappresentativa.name)

colore = dominant_colors_lab(rappresentativa)
print("sfondo rilevato :", round(colore["background_fraction"], 3))
for c in colore["palette"]:
    L, a, b = c["lab"]
    print(f"   L={L:6.2f}  a={a:6.2f}  b={b:6.2f}   {c['proportion']*100:5.1f}% dell'immagine")

CAMICIA UOMO LEE CAMP SHIRT SOY BEAN
foto disponibili: ['2000003579068_1.jpg', '2000003579068_2.jpg', '2000003579068_3.jpg']


rappresentativa : 2000003579068_2.jpg


sfondo rilevato : 0.693
   L= 96.58  a= -0.94  b=  1.97    51.4% dell'immagine
   L= 91.54  a=  2.25  b=  4.44    43.3% dell'immagine
   L= 28.03  a=  2.28  b= -2.11     5.3% dell'immagine


In [31]:
# Gli stessi numeri, visti come colori.
from skimage.color import lab2rgb

def campioni(palette, titolo=""):
    quadri = ""
    for c in palette:
        rgb = (lab2rgb(np.array(c["lab"], dtype=float).reshape(1, 1, 3)).reshape(3) * 255).astype(int)
        quadri += (f"<div style='display:inline-block;text-align:center;margin-right:10px'>"
                   f"<div style='width:74px;height:74px;border:1px solid #999;"
                   f"background:rgb({rgb[0]},{rgb[1]},{rgb[2]})'></div>"
                   f"<small>{c['proportion']*100:.0f}%</small></div>")
    return HTML(f"<div><b>{titolo}</b><br>{quadri}</div>")

display(campioni(colore["palette"], meta["title"]))

## 3.2 Lo stile

Quattro funzioni decidono che cosa vale, per un capo, fra quello che dice il **testo** della
scheda e quello che ha visto la **revisione visiva** (Gemini sulla foto + descrizione).
La regola generale è la stessa ovunque: *se la revisione c'è ed è stata fatta sotto la guida
corrente, vince lei; altrimenti vale il testo*.

### `needs_review_effettivo(metadata)`

- **Input** — il dizionario `metadata.json` di un prodotto.
- **Processo** — il flag testuale `needs_vision_review` è la **diagnosi** ("la scheda non dice
  niente di distintivo"); qui si dà la **prognosi**: se la revisione visiva ha risposto con dei
  punteggi di stile, il capo esce dalla quarantena.
- **Output** — `True` se il capo va ancora escluso da ogni outfit, `False` se è abbinabile.

### `formalita_effettiva(metadata)`

- **Input** — il `metadata.json`.
- **Processo** — se la revisione porta la firma della guida corrente (`fonte == "foto+descrizione-5tag"`)
  e un valore fra 1 e 5, usa quello; altrimenti ricade sulla formalità dedotta dalle regole
  testuali. Il controllo sulla fonte non è pedanteria: i giudizi dati con la guida precedente
  divergevano dal testo di almeno un livello nel 35% dei capi.
- **Output** — un intero 1–5 (o `None` se non si sa).

### `mistura_di_stile(metadata)`

- **Input** — il `metadata.json`.
- **Processo** — costruisce la **ripartizione** fra i cinque registri (`sportivo`, `casual`,
  `elegante`, `streetwear`, `da_mare`) normalizzata a somma 1. Non somma testo e revisione:
  se la revisione c'è è lei la classificazione, perché mescolarle romperebbe proprio ciò che
  le rende leggibili (una camicia 0,6 casual / 0,4 elegante diventerebbe altro solo perché
  nella prosa compare una parola).
- **Output** — `{"sportivo": .., "casual": .., "elegante": .., "streetwear": .., "da_mare": ..}`.

### `fantasia_effettiva(metadata)`

- **Input** — il `metadata.json`.
- **Processo** — prende `pattern` dalla revisione visiva, poi dal testo, poi `tinta_unita`.
  La fantasia è la cosa che il testo nomina peggio: spesso non la nomina affatto.
- **Output** — una fra `tinta_unita`, `righe`, `quadri`, `stampato`, `camouflage`, `animalier`.

### `build_style_vector(metadata)`

- **Input** — il `metadata.json`.
- **Processo** — mette insieme le funzioni sopra: i 5 registri, la formalità normalizzata
  `(livello − 1) / 4` e la stagione in one-hot su 4 colonne.
- **Output** — il dizionario delle **10 colonne** del vettore di stile, che finiscono
  direttamente nella riga del parquet.

In [32]:
# ═══ nuvolari_generation_pipeline/feature_engineering.py ═══════════════════════════════

# =====================================================================
# Vettore stile (multi-hot style_tags + formalità normalizzata + stagione)
# =====================================================================

# Peso del punteggio visivo nella scala dei punteggi grezzi di
# compute_style_scores, dove un riscontro testuale vale 0.8. La revisione
# visiva guarda il capo, non le parole che lo accompagnano: un tag pieno
# (1.0) vale 2.4, cioè tre riscontri testuali — abbastanza da dare al capo
# una norma sopra la mediana del catalogo (0.55) e quindi piena confidenza
# in style_match, che è esattamente ciò che la revisione deve comprare.
VISION_STYLE_WEIGHT = 2.4


def needs_review_effettivo(metadata: dict) -> bool:
    """La quarantena vale finché non c'è una revisione visiva riuscita.

    Il flag testuale needs_vision_review resta com'è (è la diagnosi:
    "il testo non dice nulla"); qui si decide la prognosi: se Gemini ha
    visto la foto e ha risposto, il capo torna abbinabile.
    """
    if not metadata.get("needs_vision_review"):
        return False
    revisione = metadata.get("vision_review") or {}
    return not revisione.get("style_scores")


# La revisione fatta sotto la guida attuale. Il controllo sulla fonte non e'
# pedanteria: i giudizi precedenti nascevano da un prompt che non aveva ne' la
# descrizione completa ne' il procedimento sulla formalita', e divergevano dal
# testo di almeno un livello nel 35% dei capi. Finche' un capo non e' stato
# rivisto sotto la guida nuova, la sua formalita' resta quella testuale.
FONTE_GUIDA = "foto+descrizione-5tag"


def formalita_effettiva(metadata: dict):
    """La formalita' da usare: quella di Gemini se disponibile, altrimenti quella
    delle regole testuali.

    Le regole testuali pesavano per meta' su una parola del titolo, che descrive
    la FORMA del capo e non il registro — "giacca" vale sia per un blazer sia per
    un antivento K-Way. I tetti di brand esistevano solo per rattoppare questo,
    marchio per marchio. Gemini vede foto, composizione e descrizione insieme, e
    il procedimento di quelle regole ce l'ha scritto nella guida.
    """
    revisione = metadata.get("vision_review") or {}
    if revisione.get("fonte") == FONTE_GUIDA:
        f = revisione.get("formality")
        if isinstance(f, (int, float)) and 1 <= f <= 5:
            return int(f)
    return metadata.get("formality_score")


def mistura_di_stile(metadata: dict) -> dict:
    """La ripartizione fra i registri, normalizzata a somma 1.

    Con cinque registri la classificazione non è più un insieme di flag
    indipendenti ma una RIPARTIZIONE: una camicia è metà elegante e metà casual,
    una tuta metà sportiva e metà casual. Sommare i punteggi del testo a quelli
    della revisione romperebbe proprio la cosa che li rende leggibili — un capo
    dato "casual 1.0" diventerebbe casual 0.75 / sportivo 0.25 solo perché nella
    prosa compare la parola "running". Quindi: se la revisione c'è, è lei la
    classificazione; il testo resta solo per i capi che la revisione non ha.
    """
    revisione = metadata.get("vision_review") or {}
    mistura = {t: float(v) for t, v in (revisione.get("style_scores") or {}).items()
               if t in STYLE_TAGS and float(v) > 0}
    if not mistura:
        grezzi, _ = compute_style_scores(metadata)
        mistura = {t: v for t, v in grezzi.items() if t in STYLE_TAGS and v > 0}
    totale = sum(mistura.values())
    if not totale:
        return {t: 0.0 for t in STYLE_TAGS}
    return {t: mistura.get(t, 0.0) / totale for t in STYLE_TAGS}


def fantasia_effettiva(metadata: dict) -> str:
    """La fantasia del capo: quella vista nella foto, o quella del testo.

    La revisione visiva viene prima perche' la fantasia e' la cosa che il testo
    nomina peggio — spesso non la nomina affatto, o nomina un dettaglio.
    """
    revisione = metadata.get("vision_review") or {}
    return revisione.get("pattern") or metadata.get("pattern") or "tinta_unita"


def build_style_vector(metadata: dict) -> dict:
    tag_values = {f"style_{tag}": v for tag, v in mistura_di_stile(metadata).items()}

    formality = formalita_effettiva(metadata)
    formality_norm = (formality - 1) / 4 if formality is not None else 0.5

    season = metadata.get("season") or "tutte"
    season_values = {f"season_{s}": (1.0 if season == s else 0.0) for s in SEASONS}

    return {**tag_values, "formality_norm": formality_norm, **season_values}


STYLE_VECTOR_COLUMNS = (
    [f"style_{t}" for t in STYLE_TAGS] + ["formality_norm"] + [f"season_{s}" for s in SEASONS]
)

In [33]:
print(meta["title"])
print("  testo   → formality_score:", meta.get("formality_score"),
      "| needs_vision_review:", meta.get("needs_vision_review"),
      "| pattern:", meta.get("pattern"))
print("  visione →", json.dumps(meta.get("vision_review"), ensure_ascii=False)[:200], "...")
print()
print("needs_review_effettivo :", needs_review_effettivo(meta))
print("formalita_effettiva    :", formalita_effettiva(meta))
print("fantasia_effettiva     :", fantasia_effettiva(meta))
print("mistura_di_stile       :", {k: round(v, 2) for k, v in mistura_di_stile(meta).items() if v})
print()
print("build_style_vector     :")
for k, v in build_style_vector(meta).items():
    print(f"   {k:26s} {v}")

CAMICIA UOMO LEE CAMP SHIRT SOY BEAN
  testo   → formality_score: 3 | needs_vision_review: True | pattern: righe
  visione → {"model": "gemini-3.5-flash", "fonte": "foto+descrizione-5tag", "quando": "2026-08-19", "style_scores": {"casual": 0.6, "da_mare": 0.4}, "formality": 2, "formality_perche": "Camicia a maniche corte co ...

needs_review_effettivo : False
formalita_effettiva    : 2
fantasia_effettiva     : righe
mistura_di_stile       : {'casual': 0.6, 'da_mare': 0.4}

build_style_vector     :
   style_sportivo             0.0
   style_casual               0.6
   style_elegante             0.0
   style_streetwear           0.0
   style_da_mare              0.4
   formality_norm             0.25
   season_estate              1.0
   season_inverno             0.0
   season_mezza_stagione      0.0
   season_tutte               0.0


### `build_feature_table(root, out_parquet=None, limit=None)`

- **Input** — la radice del catalogo (una cartella per prodotto, con foto e `metadata.json`).
- **Processo** — per ogni prodotto: scarta le foto illeggibili (**la singola foto rotta, non
  il prodotto**), sceglie la rappresentativa, estrae il colore, costruisce il vettore di stile
  e impila tutto in una riga.
- **Output** — il `DataFrame` completo e, se `out_parquet` è dato, `features.parquet`.
  È l'unica funzione della fase 3 che tocca il disco.

Due funzioni di servizio girano intorno a questa:

- **`recompute_style_vectors(features_parquet, root, out_parquet=None)`** — rifà **solo** le
  colonne di stile riusando il colore già estratto (l'estrazione colore è la parte cara).
  *Input*: il parquet esistente + il catalogo. *Output*: lo stesso parquet con i vettori aggiornati.
- **`add_display_and_gallery_columns(...)`** — aggiunge `display_image` e `all_images`, cioè
  la foto da mostrare e la galleria completa che poi la fase 7 allega a Gemini.

`applica_idf(...)` esiste ancora nel modulo ma **non è più in catena** dal 20 agosto: pesava i
tag per rarità quando lo stile era fatto di dieci bandierine indipendenti, e con la
ripartizione a cinque registri cancellava proprio il registro condiviso (`casual`) che tiene
insieme il guardaroba.

Qui non la si riesegue — `build_feature_table` rilegge 2901 cartelle e le loro foto — ma il
risultato è già sul disco, ed è quello che il resto del notebook usa.

In [34]:
# ═══ nuvolari_generation_pipeline/feature_engineering.py ═══════════════════════════════

# =====================================================================
# Pipeline: scansiona metadata.json, calcola entrambi i vettori, salva
# =====================================================================

def build_feature_table(root: str, out_parquet: str = None, limit: int = None) -> pd.DataFrame:
    root_dir = Path(root)
    metadata_files = sorted(root_dir.rglob("metadata.json"))
    if limit:
        metadata_files = metadata_files[:limit]
    print(f"[*] Trovati {len(metadata_files)} metadata.json sotto {root_dir}", flush=True)

    rows = []
    for i, meta_path in enumerate(metadata_files, 1):
        product_dir = meta_path.parent
        metadata = json.loads(meta_path.read_text(encoding="utf-8"))

        tutte = sorted(
            p for p in product_dir.iterdir()
            if p.suffix.lower() in IMAGE_SUFFIXES
        )
        # si scarta la singola foto corrotta, non l'intero prodotto: un download
        # troncato su una foto non deve far perdere un capo che ne ha altre buone
        image_paths = [p for p in tutte if is_readable(p)]
        if len(image_paths) < len(tutte):
            print(f"    [!] {product_dir.name}: {len(tutte) - len(image_paths)} foto illeggibili, "
                  f"uso le altre {len(image_paths)}", flush=True)
        if not image_paths:
            print(f"    [!] {product_dir.name}: nessuna foto utilizzabile, salto", flush=True)
            continue

        try:
            rep_image = pick_representative_image(image_paths)
            color = dominant_colors_lab(rep_image)
        except Exception as e:
            print(f"    [!] {product_dir.name}: errore estrazione colore ({e}), salto", flush=True)
            continue

        style_vec = build_style_vector(metadata)

        row = {
            "relpath": metadata.get("relpath", str(product_dir.relative_to(root_dir))),
            "url": metadata.get("url"),
            "brand_slug": metadata.get("brand_slug"),
            "title": metadata.get("title"),
            "needs_vision_review": needs_review_effettivo(metadata),
            "pattern": fantasia_effettiva(metadata),
            "representative_image": str(rep_image.relative_to(root_dir)),
            "L": color["dominant_lab"][0],
            "a": color["dominant_lab"][1],
            "b": color["dominant_lab"][2],
            "color_palette": json.dumps(color["palette"]),
            "background_fraction": color["background_fraction"],
            **style_vec,
        }
        rows.append(row)

        if i % 50 == 0 or i == len(metadata_files):
            print(f"    [{i}/{len(metadata_files)}] elaborati...", flush=True)

    df = pd.DataFrame(rows)

    if out_parquet:
        df.to_parquet(out_parquet, index=False)
        print(f"[OK] Salvato {len(df)} prodotti in {out_parquet}", flush=True)

    return df


def recompute_style_vectors(features_parquet: str, root: str, out_parquet: str = None) -> pd.DataFrame:
    """Ricalcola solo le colonne del vettore stile (style_*, formality_norm,
    season_*) da metadata.json, riusando L/a/b/color_palette già estratti in
    features_parquet. Utile dopo una modifica alla logica di build_style_vector
    (es. build_style_vector), senza rifare la costosa estrazione colore."""
    df = pd.read_parquet(features_parquet).set_index("relpath")
    root_dir = Path(root)

    for relpath in df.index:
        meta_path = root_dir / relpath / "metadata.json"
        if not meta_path.exists():
            print(f"    [!] {relpath}: metadata.json non trovato, salto", flush=True)
            continue
        metadata = json.loads(meta_path.read_text(encoding="utf-8"))
        for col, val in build_style_vector(metadata).items():
            df.loc[relpath, col] = val
        df.loc[relpath, "brand_slug"] = metadata.get("brand_slug")
        df.loc[relpath, "needs_vision_review"] = needs_review_effettivo(metadata)

    df = df.reset_index()

    if out_parquet:
        df.to_parquet(out_parquet, index=False)
        print(f"[OK] Salvato {len(df)} prodotti in {out_parquet}", flush=True)

    return df


IDF_FILE = BASE / "style_idf.json"


def applica_idf(features_parquet: str, out_parquet: str = None) -> pd.DataFrame:
    """NON PIU' IN USO dal 20 agosto. Vedi sotto prima di rimetterla in catena.

    Serviva quando lo stile era dieci flag indipendenti e "casual" compariva sul
    96% dei capi senza voler dire niente: quel tono di fondo schiacciava lo
    spazio, e pesare i tag per la loro rarita' lo riapriva.

    Con cinque registri assegnati come RIPARTIZIONE che somma a 1, "casual" non
    e' piu' un tono di fondo: e' la base condivisa del guardaroba, dichiarata
    apposta. L'IDF gli dava peso 0,16 e cancellava proprio quella — un giubbotto
    "casual 0.9" e un cappello "casual 0.7" scendevano da 0,914 a 0,226, cioe'
    due capi che stanno benissimo insieme risultavano incompatibili.

    Toglierla non ha riaperto la porta al rumore, misurato su 4000 coppie a
    caso: le coppie sbagliate restano a zero (giacca elegante + running 0,012,
    + costume 0,000). La separazione ora viene dalla tassonomia, non dai pesi.

    --- documentazione originale ---

    Ripesa i 10 tag di stile per quanto sono RARI nel catalogo.

    Serve perché la revisione visiva ha un fortissimo tono di fondo: Gemini
    assegna "casual" al 96% dei capi e lo fa dominante nel 57%. Un tag che
    quasi tutti hanno non distingue niente — e finché pesava come gli altri
    schiacciava lo spazio: il coseno fra due capi a caso saliva a 0,71 (tutto
    somigliava a tutto) e HDBSCAN produceva un cluster solo da 1548 capi con
    intorno il 78% di outlier.

    Il peso è l'IDF classico, log(1/prevalenza): "casual" (96%) scende a 0,04,
    "boho_fantasia" o "elegante" (5%) salgono a ~3. Condividere un tratto
    raro è prova di somiglianza molto più forte che condividerne uno comune.
    Dopo la ripesatura il coseno mediano torna a 0,28 e i cluster a 25, con il
    più grande al 17% del catalogo.

    I pesi si ricalcolano dal catalogo corrente e si salvano in style_idf.json,
    così si può leggere quanto vale ogni tag senza rieseguire nulla. Va
    rilanciata dopo ogni cambio ai vettori di stile, PRIMA del clustering.
    formality_norm e le colonne di stagione non si toccano: non sono tag.
    """
    df = pd.read_parquet(features_parquet)
    colonne = [f"style_{t}" for t in STYLE_TAGS]
    prevalenza = (df[colonne] > 0).mean(axis=0)
    # la prevalenza si stima sui valori GIÀ ripesati se si rilancia due volte:
    # per questo il file dei pesi è la fonte di verità e si riparte dal grezzo
    # Radice dell'IDF, non IDF pieno. Misurato su coppie di capi già giudicate
    # a mano (felpa NASA + jeans, + K-Way, + Saucony come BUONE; blazer o
    # camicia + tuta, mocassino + t-shirt come CATTIVE), lo stacco fra le due
    # famiglie disegna una U rovesciata con il massimo a esponente 0,5:
    #     esponente 0.00 -> buone 0.623, cattive 0.637  (stacco -0.014!)
    #     esponente 0.50 -> buone 0.363, cattive 0.268  (stacco +0.095)
    #     esponente 1.00 -> buone 0.187, cattive 0.179  (stacco +0.007)
    # I due estremi sbagliano per motivi opposti e simmetrici: senza pesi il
    # tono di fondo "casual" rende tutto simile a tutto (e le coppie sbagliate
    # scorano perfino PIÙ alte delle giuste); con l'IDF pieno condividere un
    # tratto comune non vale più niente, ma due capi entrambi casual stanno
    # insieme davvero — la compatibilità non è solo distintività.
    ESPONENTE_IDF = 0.5
    pesi = np.log(1.0 / prevalenza.clip(lower=1e-3)) ** ESPONENTE_IDF
    df[colonne] = df[colonne].to_numpy(float) * pesi.to_numpy(float)

    IDF_FILE.write_text(json.dumps(
        {t: round(float(w), 4) for t, w in zip(colonne, pesi)}, ensure_ascii=False, indent=2),
        encoding="utf-8")
    print("[*] pesi IDF (tag piu' raro = piu' pesante):", flush=True)
    for t, w in sorted(zip(colonne, pesi), key=lambda kv: -kv[1]):
        print(f"      {t.replace('style_',''):18s} prevalenza {prevalenza[t]*100:5.1f}%  peso {w:.2f}", flush=True)

    if out_parquet:
        df.to_parquet(out_parquet, index=False)
        print(f"[OK] Salvato {len(df)} prodotti in {out_parquet}", flush=True)
    return df


def add_display_and_gallery_columns(features_parquet: str, root: str, out_parquet: str = None) -> pd.DataFrame:
    """Aggiunge due colonne, entrambe in locale senza rete:

    - display_image: la PRIMA foto della galleria (ordine alfabetico dei
      file), per mostrare il capo a un umano — a differenza di
      representative_image (Fase 3), che è scelta per la massima frazione
      di sfondo pulito ai fini del k-means colore e spesso è un dettaglio
      ravvicinato, non la foto d'insieme del capo.
    - all_images: lista di TUTTE le foto della galleria — servirà quando
      passeremo i dati a un'eventuale API di generazione immagini, che
      potrebbe voler usare più angolazioni dello stesso capo, non solo una.
    """
    df = pd.read_parquet(features_parquet)
    root_dir = Path(root)

    display_images, all_images_col = [], []
    for relpath in df["relpath"]:
        product_dir = root_dir / relpath
        # stesso filtro di build_feature_table: una foto illeggibile qui
        # finirebbe fra quelle inviate al modello in fase di generazione
        images = sorted(
            str(p.relative_to(root_dir)) for p in product_dir.iterdir()
            if p.suffix.lower() in IMAGE_SUFFIXES and is_readable(p)
        ) if product_dir.exists() else []
        display_images.append(images[0] if images else None)
        all_images_col.append(images)

    df["display_image"] = display_images
    df["all_images"] = all_images_col

    if out_parquet:
        df.to_parquet(out_parquet, index=False)
        print(f"[OK] Salvato {len(df)} prodotti in {out_parquet}", flush=True)

    return df

In [35]:
if ESEGUI_FEATURES:
    # Rilegge tutte le foto del catalogo: ~30 minuti, nessun costo, nessuna rete.
    features = build_feature_table(root=str(CATALOGO), out_parquet=str(FEATURES))
else:
    print("ESEGUI_FEATURES = False: si usa il features.parquet già sul disco.")
    print("   (build_feature_table rilegge 2901 cartelle e le loro foto: ~30 minuti, gratis)")

ESEGUI_FEATURES = False: si usa il features.parquet già sul disco.
   (build_feature_table rilegge 2901 cartelle e le loro foto: ~30 minuti, gratis)


In [36]:
print("needs_vision_review (capi esclusi da ogni outfit):")
print(features["needs_vision_review"].value_counts().to_string())
print()
print("formalità (livello ricostruito da formality_norm):")
print((features["formality_norm"] * 4 + 1).round().astype(int).value_counts().sort_index().to_string())
print()
print("stagione:")
print(features[[c for c in features.columns if c.startswith("season_")]].sum().astype(int).to_string())
print()
print("fantasia:")
print(features["pattern"].value_counts().to_string())

needs_vision_review (capi esclusi da ogni outfit):
needs_vision_review
False    2901

formalità (livello ricostruito da formality_norm):
formality_norm
1     328
2    1651
3     882
4      40

stagione:
season_estate             362
season_inverno            134
season_mezza_stagione      48
season_tutte             2357

fantasia:
pattern
tinta_unita    2500
stampato        226
righe            97
quadri           36
animalier        23
camouflage       19


---
# 4. Clusterizzazione — Fase 4

I capi valutati vengono raggruppati per **registro stilistico**. Serve a due cose:
ridurre lo spazio di ricerca della fase 6 (invece di ogni coppia possibile, solo le coppie
dentro cluster compatibili) e avere un'etichetta che dica quando due capi appartengono
allo stesso mondo.

**Perché HDBSCAN e non K-Means**: non chiede di fissare il numero di gruppi in anticipo,
regge densità diverse e soprattutto **isola gli outlier** (etichetta `-1`) invece di
forzarli dentro un gruppo sbagliato.

### `clustering_feature_columns(include_season=False)`

- **Input** — un flag.
- **Processo** — elenca le colonne su cui clusterizzare: i 5 tag di stile **più**
  `formality_norm`. La stagione è esclusa di proposito — è già un filtro esplicito in fase 6,
  e includerla qui separerebbe un capo invernale da uno estivo dello stesso identico registro.
- **Output** — la lista dei nomi di colonna.

### `run_clustering(features_parquet, out_parquet=None, min_cluster_size=10, cluster_selection_epsilon=0.15, ...)`

- **Input** — il percorso di `features.parquet` (uscita della fase 3).
- **Processo** — estrae la matrice delle colonne di cui sopra e ci lancia **HDBSCAN**
  (metrica euclidea). `min_cluster_size=10`: sotto i dieci capi non è un registro, è un caso.
  `cluster_selection_epsilon=0.15`: senza, un vettore a bassa cardinalità come questo produce
  decine di micro-cluster dai profili identici. Poi stampa dimensioni e profilo medio di ogni
  gruppo, che è il modo in cui si controlla se i gruppi *significano* qualcosa.
- **Output** — il `DataFrame` con due colonne nuove, `style_cluster` (l'etichetta, `-1` = outlier)
  e `cluster_probability`; su disco `features_clustered.parquet` se `out_parquet` è dato.

In [37]:
# ═══ nuvolari_generation_pipeline/clustering.py ═══════════════════════════════

STYLE_TAGS = [
    "sportivo", "casual", "elegante", "streetwear", "da_mare",
]
SEASONS = ["estate", "inverno", "mezza_stagione", "tutte"]

STYLE_TAG_COLUMNS = [f"style_{t}" for t in STYLE_TAGS]
SEASON_COLUMNS = [f"season_{s}" for s in SEASONS]


def clustering_feature_columns(include_season: bool = False) -> list:
    cols = STYLE_TAG_COLUMNS + ["formality_norm"]
    if include_season:
        cols += SEASON_COLUMNS
    return cols


def run_clustering(
    features_parquet: str,
    out_parquet: str = None,
    min_cluster_size: int = 10,
    min_samples: int = None,
    cluster_selection_method: str = "eom",
    cluster_selection_epsilon: float = 0.15,
    include_season: bool = False,
) -> pd.DataFrame:
    df = pd.read_parquet(features_parquet)
    feature_cols = clustering_feature_columns(include_season)
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Colonne mancanti in {features_parquet}: {missing}")

    X = df[feature_cols].to_numpy(dtype=float)

    print(
        f"[*] Clustering HDBSCAN su {len(df)} prodotti, {X.shape[1]} feature "
        f"({'con' if include_season else 'senza'} stagione), "
        f"min_cluster_size={min_cluster_size}, epsilon={cluster_selection_epsilon}...",
        flush=True,
    )

    # cluster_selection_epsilon=0.15 (invece del default 0.0) fonde i
    # micro-cluster quasi duplicati che il puro "excess of mass" di HDBSCAN
    # lascia separati su questo vettore stile a bassa cardinalità — vedi
    # analisi in conversazione: senza, si ottengono decine di cluster con
    # profili praticamente identici invece di gruppi genuinamente distinti.
    clusterer = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method=cluster_selection_method,
        cluster_selection_epsilon=cluster_selection_epsilon,
    )
    labels = clusterer.fit_predict(X)

    df = df.copy()
    df["style_cluster"] = labels
    if hasattr(clusterer, "probabilities_"):
        df["cluster_probability"] = clusterer.probabilities_

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_outliers = int((labels == -1).sum())
    print(
        f"[OK] {n_clusters} cluster trovati, {n_outliers} outlier "
        f"({n_outliers / len(df) * 100:.1f}%)",
        flush=True,
    )

    sizes = df["style_cluster"].value_counts().sort_values(ascending=False)
    print("\nDimensione cluster (top 20):")
    print(sizes.head(20))

    print("\nProfilo medio per cluster (tag di stile dominanti):")
    profile = df.groupby("style_cluster")[STYLE_TAG_COLUMNS + ["formality_norm"]].mean()
    for cluster_id, row in profile.iterrows():
        top_tags = row[STYLE_TAG_COLUMNS].sort_values(ascending=False).head(3)
        tags_str = ", ".join(f"{t.replace('style_', '')} ({v:.2f})" for t, v in top_tags.items())
        label = "outlier" if cluster_id == -1 else f"cluster {cluster_id}"
        n = int(sizes.get(cluster_id, 0))
        print(f"  {label:>12} (n={n:4d}, formalità={row['formality_norm']:.2f}): {tags_str}")

    if out_parquet:
        df.to_parquet(out_parquet, index=False)
        print(f"\n[OK] Salvato in {out_parquet}", flush=True)

    return df

In [38]:
print("colonne del clustering:", clustering_feature_columns())
print("(escluse:", [c for c in features.columns if c.startswith('season_')], ")")

colonne del clustering: ['style_sportivo', 'style_casual', 'style_elegante', 'style_streetwear', 'style_da_mare', 'formality_norm']
(escluse: ['season_estate', 'season_inverno', 'season_mezza_stagione', 'season_tutte'] )


In [39]:
# Rieseguito dal vivo. out_parquet resta None: il file sul disco non si tocca.
t0 = time.time()
clusterizzato = run_clustering(features_parquet=str(FEATURES))
print(f"\n[t] {time.time()-t0:.1f}s")

[*] Clustering HDBSCAN su 2901 prodotti, 6 feature (senza stagione), min_cluster_size=10, epsilon=0.15...


[OK] 7 cluster trovati, 28 outlier (1.0%)



Dimensione cluster (top 20):
style_cluster
 2    1656
 1     883
 6     119
 3     114
 4      53
 0      38
-1      28
 5      10
Name: count, dtype: int64

Profilo medio per cluster (tag di stile dominanti):
       outlier (n=  28, formalità=0.04): da_mare (0.36), casual (0.25), sportivo (0.19)
     cluster 0 (n=  38, formalità=0.75): elegante (0.76), casual (0.24), sportivo (0.00)
     cluster 1 (n= 883, formalità=0.50): casual (0.69), elegante (0.28), da_mare (0.02)
     cluster 2 (n=1656, formalità=0.25): casual (0.68), streetwear (0.18), sportivo (0.09)
     cluster 3 (n= 114, formalità=0.00): sportivo (0.44), casual (0.41), streetwear (0.15)
     cluster 4 (n=  53, formalità=0.00): streetwear (0.76), casual (0.23), da_mare (0.00)
     cluster 5 (n=  10, formalità=0.00): casual (0.54), da_mare (0.26), streetwear (0.20)
     cluster 6 (n= 119, formalità=0.00): da_mare (0.79), casual (0.19), streetwear (0.01)

[t] 0.1s


In [40]:
# Il clustering è deterministico: le etichette appena calcolate coincidono con
# quelle salvate in features_clustered.parquet, che sono quelle con cui è stato
# costruito il pool.
salvato = pd.read_parquet(CLUSTERED)
assert (salvato["relpath"].to_numpy() == clusterizzato["relpath"].to_numpy()).all()
accordo = (salvato["style_cluster"].to_numpy() == clusterizzato["style_cluster"].to_numpy()).mean()
print(f"accordo con le etichette sul disco: {accordo*100:.1f}%")

accordo con le etichette sul disco: 100.0%


In [41]:
# Che cosa è ogni cluster, in una tabella: dimensione, formalità media, registri dominanti.
profilo = (salvato.groupby("style_cluster")[STYLE_TAG_COLUMNS + ["formality_norm"]]
                  .mean().round(2))
profilo.insert(0, "n_capi", salvato["style_cluster"].value_counts().sort_index())
profilo.index = ["outlier" if i == -1 else f"cluster {i}" for i in profilo.index]
profilo

,n_capi,style_sportivo,style_casual,style_elegante,style_streetwear,style_da_mare,formality_norm
outlier,28,0.19,0.25,0.02,0.18,0.36,0.04
cluster 0,38,0.00,0.24,0.76,0.00,0.00,0.75
cluster 1,883,0.01,0.69,0.28,0.01,0.02,0.50
cluster 2,1656,0.09,0.68,0.01,0.18,0.04,0.25
cluster 3,114,0.44,0.41,0.00,0.15,0.00,0.00
cluster 4,53,0.00,0.23,0.00,0.76,0.00,0.00
cluster 5,10,0.00,0.54,0.00,0.20,0.26,0.00
cluster 6,119,0.01,0.19,0.00,0.01,0.79,0.00


---
# 5. Assegnazione dei punteggi — Fase 5

Il cuore aritmetico di tutto: quanto stanno bene insieme **due capi qualsiasi**.

```
score(A,B) = 0,5 · armonia_colore(A,B) + 0,5 · affinità_stile(A,B) − 0,22 · stono_fantasie(A,B)
```

Le tre parti si guardano una per una.

## 5.1 L'armonia di colore

### `hue_circular_distance(h1, h2)`

- **Input** — due tonalità in gradi (0–360).
- **Processo** — la distanza sul **cerchio** delle tonalità, non sulla retta: fra 350° e 10°
  ci sono 20 gradi, non 340.
- **Output** — un valore fra 0 e 180.

### `color_harmony_pair(lab_a, lab_b)`

- **Input** — due colori in Lab.
- **Processo** — calcola ΔE (distanza complessiva), croma (saturazione) e Δh (distanza di
  tonalità), poi applica in cascata le regole della teoria del colore: quasi identici →
  monocromatico voluto; stessa tonalità ma ΔE piccolo → *sembra un errore*; analoghi;
  complementari; split-complementari; e in mezzo la zona senza regole. Sopra ci passano due
  correzioni: il **bonus neutro sfumato** (un colore desaturato va bene con quasi tutto, ma
  proporzionalmente a quanto è davvero grigio) e le due eccezioni qui sotto.
- **Output** — un punteggio 0–1 per **quella coppia di colori**.

### `stono_da_sottotono(La, Lb, aa, ba, ab, bb)` e `stono_nero_su_blu(...)`

- **Input** — le componenti Lab dei due colori.
- **Processo** — misurano, con continuità (non a gradini), due casi in cui il bonus neutro
  premierebbe un abbinamento che l'occhio legge come sbaglio: due **scuri** che si distinguono
  solo per sottotono (nero + blu notte), e il **nero addosso al blu** in generale (nero + denim).
- **Output** — quanto la coppia è quel caso, da 0 a 1. Il punteggio finale viene interpolato
  verso 0,30 nel primo caso e verso 0,45 nel secondo.

### `color_harmony(palette_a, palette_b, top_k=2)`

- **Input** — le due **palette** complete (colore + proporzione), non i due colori dominanti.
- **Processo** — media di `color_harmony_pair` sui primi `top_k` colori di ciascuno, pesata
  per il **prodotto delle proporzioni**: un colore marginale conta meno di quello dominante.
  Con un'eccezione: se a stonare sono i due **dominanti**, vale il loro punteggio e la media
  non si applica — i secondari non riscattano ciò che l'occhio legge a distanza.
- **Output** — l'armonia di colore fra i due capi, 0–1.

In [42]:
# ═══ nuvolari_generation_pipeline/scoring.py ═══════════════════════════════

# --- color_harmony: soglie tarate sulla distribuzione di croma del catalogo
# (mediana ~5, 75° percentile ~12 — un catalogo di abbigliamento generalista
# è dominato da colori tenui/neutri, non da tinte sature) ---
NEUTRAL_CHROMA = 10.0
IDENTICAL_DELTA_E = 3.0
CLASH_DELTA_E_HIGH = 15.0
HUE_MONOCHROME = 20.0
HUE_ANALOGOUS = 60.0
# In spazio Lab le coppie complementari classiche non arrivano a 180°: blu e
# arancio, l'esempio da manuale, distano 135°. Con la soglia a 150 la regola
# non scattava quasi mai e il 9,9% delle coppie sature del catalogo — i
# complementari veri — finiva nella zona intermedia a 0,45, cioè valutato come
# un abbinamento senza qualità. Misurato su 312 coppie campionate.
HUE_COMPLEMENTARY = 130.0
# Fra l'analogo e il complementare c'è la relazione split-complementare, che in
# teoria del colore è un abbinamento riconosciuto (verde oliva e bordeaux):
# merita più della zona senza regole, meno del complementare pieno.
HUE_SPLIT_COMPLEMENTARY = 100.0

# Due colori entrambi scuri che differiscono solo per sottotono — il caso
# nero + blu notte — stonano come due blu leggermente diversi: la differenza
# si legge come errore di illuminazione, non come scelta.
#
# Perché sia uno stono servono due cose insieme, ed entrambe sono questioni di
# grado, non interruttori: i due devono essere abbastanza vicini di luminosità
# da sembrare voler essere lo stesso colore, e il sottotono che li separa deve
# vedersi. La versione precedente le trattava come sì/no e chiedeva la cosa
# sbagliata sulla seconda — la croma di UNO dei due invece della distanza FRA i
# due — con l'effetto che due giubbotti blu notte identici all'occhio, croma
# 4,57 e 5,27, prendevano 0,744 e 0,300 contro la stessa felpa nera.
DARK_L_MAX = 40.0             # oltre questa luminosità il colore non è più "scuro"
DARK_L_SIMILE = 12.0          # sotto questo stacco i due sembrano voler essere lo stesso colore
DARK_L_DELTA = 25.0           # oltre questo stacco la coppia si legge come scelta, non errore
UNDERTONE_DELTA_MIN = 4.0     # distanza cromatica sotto cui il sottotono non si vede
UNDERTONE_DELTA_PIENO = 12.0  # ...e sopra cui si vede senza dubbi
STONO_SCURI = 0.30            # il verdetto pieno, quando entrambe le condizioni sono al massimo

# Il nero non e' neutro con il blu.
#
# Il bonus "neutro" qui sotto dice che un colore desaturato sta bene con tutto,
# e per grigi, bianchi e beige e' vero. Con il blu no: il nero e' l'unico
# neutro che il blu tira dentro la propria famiglia, perche' un blu abbastanza
# scuro E' quasi nero. L'occhio prova ad accoppiarli, non ci riesce, e legge la
# differenza come un errore invece che come un contrasto voluto.
#
# stono_da_sottotono copre solo la punta del problema — nero e blu notte alla
# stessa profondita' — e lascia fuori il caso piu' comune, il nero addosso al
# denim: fra una t-shirt nera (L 5,9) e un giubbotto denim (L 35,2) ci sono 29
# punti di luminosita', sopra DARK_L_DELTA, quindi quella regola non scatta e
# la coppia prendeva 0,90 pieno.
#
# Soglie misurate sui 2901 capi: 1125 hanno per dominante un neutro scuro
# (mediana L 12) e 385 un blu con croma >= 6 nella banda hue 200-330 — p5 247,
# mediana 279, p95 309, croma mediana 14,4 — di cui 213 scuri e 172 chiari.
NERO_L_PIENO = 15.0        # sotto: neutro nero senza dubbi
NERO_L_MAX = 35.0          # sopra: e' un grigio scuro, e con il blu ci sta
BLU_HUE_CENTRO = 280.0     # centro della banda blu misurata
BLU_HUE_PIENO = 25.0       # entro questo scarto dal centro e' blu pieno
BLU_HUE_MAX = 45.0         # oltre non e' piu' blu
BLU_CROMA_MIN = 6.0        # sotto: un grigio con un'ombra di blu, non un blu
BLU_CROMA_PIENO = 14.0     # la croma mediana dei blu del catalogo
# Non 0,30: quello e' il verdetto riservato al nero con il blu notte. Questo e'
# il valore della "zona intermedia" di hue_score — nessuna regola della teoria
# del colore sostiene l'abbinamento — che e' esattamente cio' che resta quando
# al nero si toglie il bonus neutro che non gli spetta.
ARMONIA_NERO_BLU = 0.45


def _quanto_nero(L: float, croma: float) -> float:
    """Quanto un colore e' nero (non un grigio scuro), da 0 a 1."""
    if croma >= NEUTRAL_CHROMA or L >= NERO_L_MAX:
        return 0.0
    if L <= NERO_L_PIENO:
        return 1.0
    return (NERO_L_MAX - L) / (NERO_L_MAX - NERO_L_PIENO)


def _quanto_blu(a: float, b: float) -> float:
    """Quanto un colore e' blu, da 0 a 1. La luminosita' non conta: il blu
    chiaro del denim lavato e il blu notte fanno lo stesso effetto contro il
    nero, ed e' il caso che si voleva coprire."""
    croma = math.hypot(a, b)
    if croma < BLU_CROMA_MIN:
        return 0.0
    scarto = hue_circular_distance(math.degrees(math.atan2(b, a)) % 360, BLU_HUE_CENTRO)
    if scarto >= BLU_HUE_MAX:
        return 0.0
    dentro = 1.0 if scarto <= BLU_HUE_PIENO else (BLU_HUE_MAX - scarto) / (BLU_HUE_MAX - BLU_HUE_PIENO)
    saturo = min(1.0, (croma - BLU_CROMA_MIN) / (BLU_CROMA_PIENO - BLU_CROMA_MIN))
    return dentro * saturo


def stono_nero_su_blu(La, aa, ba, Lb, ab, bb) -> float:
    """Quanto la coppia e' "nero addosso al blu", da 0 a 1. Simmetrica: non
    importa quale dei due sia il nero."""
    return max(_quanto_nero(La, math.hypot(aa, ba)) * _quanto_blu(ab, bb),
               _quanto_nero(Lb, math.hypot(ab, bb)) * _quanto_blu(aa, ba))


def hue_circular_distance(h1: float, h2: float) -> float:
    d = abs(h1 - h2) % 360
    return min(d, 360 - d)


def stono_da_sottotono(La, Lb, aa, ba, ab, bb) -> float:
    """Quanto la coppia di scuri è il caso nero+blu notte, da 0 (per niente) a
    1 (in pieno). Continua e non a gradini: la differenza fra due capi che si
    somigliano non deve dipendere da quale lato di una soglia cadono.

    Due fattori, moltiplicati perché servono entrambi:
      - quanto si vede il sottotono: la distanza cromatica FRA i due colori,
        cioè la distanza nel piano a-b. Non la croma di uno solo: due blu
        notte con la stessa identica tinta hanno croma alta ed è comunque un
        monocromatico, non uno sbaglio.
      - quanto sembra voluto: se i due hanno quasi la stessa luminosità
        l'occhio li legge come un tentativo di abbinare fallito; man mano che
        lo stacco cresce la coppia si legge come due profondità scelte apposta.
    """
    if max(La, Lb) >= DARK_L_MAX:
        return 0.0
    stacco_luce = abs(La - Lb)
    if stacco_luce >= DARK_L_DELTA:
        return 0.0

    distanza_cromatica = math.hypot(aa - ab, ba - bb)
    si_vede = (distanza_cromatica - UNDERTONE_DELTA_MIN) / (UNDERTONE_DELTA_PIENO - UNDERTONE_DELTA_MIN)
    si_vede = min(1.0, max(0.0, si_vede))

    if stacco_luce <= DARK_L_SIMILE:
        sembra_voluto = 1.0
    else:
        sembra_voluto = (DARK_L_DELTA - stacco_luce) / (DARK_L_DELTA - DARK_L_SIMILE)
    return si_vede * sembra_voluto

In [43]:
# ═══ nuvolari_generation_pipeline/scoring.py ═══════════════════════════════

def color_harmony_pair(lab_a, lab_b) -> float:
    """Punteggio di armonia [0-1] tra due colori Lab, via regole di teoria
    del colore sul cerchio delle tonalità (vedi docstring del modulo)."""
    La, aa, ba = lab_a
    Lb, ab_, bb = lab_b

    delta_e = math.sqrt((La - Lb) ** 2 + (aa - ab_) ** 2 + (ba - bb) ** 2)
    Ca = math.hypot(aa, ba)
    Cb = math.hypot(ab_, bb)

    ha = math.degrees(math.atan2(ba, aa)) % 360
    hb = math.degrees(math.atan2(bb, ab_)) % 360
    dh = hue_circular_distance(ha, hb)

    if delta_e < IDENTICAL_DELTA_E:
        hue_score = 0.85  # colori praticamente identici -> monocromatico intenzionale
    elif dh < HUE_MONOCHROME:
        hue_score = 0.25 if delta_e < CLASH_DELTA_E_HIGH else 0.80  # "quasi uguale ma non uguale" vs monocromatico vero
    elif dh < HUE_ANALOGOUS:
        hue_score = 0.75  # tonalità vicine -> analogo
    elif dh > HUE_COMPLEMENTARY:
        hue_score = 0.70  # tonalità opposte -> complementare
    elif dh > HUE_SPLIT_COMPLEMENTARY:
        hue_score = 0.60  # split-complementare: relazione riconosciuta, più debole
    else:
        hue_score = 0.45  # zona intermedia, nessuna regola forte della teoria del colore

    # un colore neutro (poco saturo: nero/bianco/grigio/beige) va bene con
    # quasi tutto — ma "poco saturo" non è un interruttore netto: un grigio
    # puro (croma ~0) è neutro davvero, mentre un grigio-blu tenue (croma
    # appena sotto soglia) ha comunque un sottotono freddo riconoscibile che
    # può stonare con toni caldi (es. sabbia) anche restando desaturato.
    # Il bonus "neutro" viene quindi sfumato in base a quanto il colore meno
    # saturo dei due si avvicina al grigio puro, non applicato in blocco.
    min_chroma = min(Ca, Cb)
    if min_chroma >= NEUTRAL_CHROMA:
        return hue_score

    neutral_weight = 1 - (min_chroma / NEUTRAL_CHROMA)
    bonus_neutro = neutral_weight * 0.90 + (1 - neutral_weight) * hue_score

    # Eccezione al bonus neutro: due scuri che si distinguono solo per il
    # sottotono. È il nero con il blu notte — il colore neutro c'è (croma ~0)
    # ma non "assorbe" nulla, perché senza stacco di luminosità l'occhio legge
    # solo la differenza di sottotono, e la legge come sbaglio. Lo stesso caso
    # fra due colori saturi è già penalizzato sopra a 0,25; senza questa
    # eccezione il bonus neutro lo premiava invece a 0,90.
    stono = stono_da_sottotono(La, Lb, aa, ba, ab_, bb)
    punteggio = (1 - stono) * bonus_neutro + stono * STONO_SCURI

    # Il nero contro il blu: il bonus neutro qui sopra non gli spetta (vedi
    # ARMONIA_NERO_BLU). Il min impedisce alla regola di ALZARE un punteggio
    # gia' basso: e' una penalita', non una taratura verso 0,45.
    nero_blu = stono_nero_su_blu(La, aa, ba, Lb, ab_, bb)
    if nero_blu:
        punteggio = min(punteggio, (1 - nero_blu) * punteggio + nero_blu * ARMONIA_NERO_BLU)
    return punteggio


# Sotto questo valore, color_harmony_pair sta segnalando uno stono vero e
# proprio: il "quasi uguale ma non uguale" (0,25) e i due scuri distinti solo
# dal sottotono (0,30). Non sono punteggi bassi qualsiasi, sono verdetti.
SOGLIA_STONO = 0.30


def color_harmony(palette_a, palette_b, top_k: int = 2) -> float:
    """Media pesata di color_harmony_pair sui top_k colori di ogni palette,
    pesata dal prodotto delle proporzioni — così un colore minoritario in
    una foto (es. un dettaglio) conta meno di quello dominante.

    Con un'eccezione: se a stonare sono i due colori DOMINANTI, la media non
    si applica e vale il loro punteggio. I dominanti sono quelli che l'occhio
    legge a distanza, e non vengono riscattati dal fatto che i secondari
    vadano d'accordo. È lo stesso motivo per cui il punteggio dell'outfit è
    il minimo delle coppie e non la media: la media nasconde l'anello debole.

    Senza questa eccezione il nero con il blu notte prendeva 0,30 sulla coppia
    dominante e risaliva a 0,61 di media, perché i secondari chiari fra loro
    prendevano 0,80 — e l'abbinamento restava sopra soglia.
    """
    dominante = color_harmony_pair(palette_a[0]["lab"], palette_b[0]["lab"])
    if dominante <= SOGLIA_STONO:
        return dominante

    pa = palette_a[:top_k]
    pb = palette_b[:top_k]
    total_score, total_weight = 0.0, 0.0
    for ca in pa:
        for cb in pb:
            w = ca["proportion"] * cb["proportion"]
            total_score += w * color_harmony_pair(ca["lab"], cb["lab"])
            total_weight += w
    return total_score / total_weight if total_weight > 0 else 0.5

In [44]:
# Le regole di tonalità su colori costruiti apposta, per vederle scattare una per una.
casi = [
    ("identici (monocromatico voluto)", (40, 20, 30), (40, 20, 31)),
    ("stessa tonalità, ΔE medio",       (40, 20, 30), (52, 26, 39)),
    ("analoghi",                        (55, 30, 10), (55, 10, 30)),
    ("split-complementare",             (55, 35,  5), (55, -20, 25)),
    ("complementare",                   (55, 35,  5), (55, -30, -15)),
    ("zona intermedia",                 (55, 35,  5), (55,  5, -35)),
    ("neutro (grigio) con saturo",      (60,  0,  0), (55, 35,   5)),
    ("nero + blu notte",                (10,  0,  0), (14,  2,  -12)),
    ("nero + denim chiaro",             ( 6,  0,  0), (35,  2,  -22)),
]
righe = []
for nome, A, B in casi:
    dE = math.dist(A, B)
    dh = hue_circular_distance(math.degrees(math.atan2(A[2], A[1])) % 360,
                               math.degrees(math.atan2(B[2], B[1])) % 360)
    righe.append({"caso": nome, "ΔE": round(dE, 1), "Δh°": round(dh, 0),
                  "sottotono": round(stono_da_sottotono(A[0], B[0], A[1], A[2], B[1], B[2]), 2),
                  "nero_su_blu": round(stono_nero_su_blu(*A, *B), 2),
                  "armonia": round(color_harmony_pair(A, B), 3)})
pd.DataFrame(righe)

,caso,ΔE,Δh°,sottotono,nero_su_blu,armonia
0,identici (monocromatico voluto),1.0,1.0,0.0,0.00,0.85
1,"stessa tonalità, ΔE medio",16.2,0.0,0.0,0.00,0.80
2,analoghi,28.3,53.0,0.0,0.00,0.75
3,split-complementare,58.5,121.0,0.0,0.00,0.60
4,complementare,68.0,162.0,0.0,0.00,0.70
5,zona intermedia,50.0,90.0,0.0,0.00,0.45
6,neutro (grigio) con saturo,35.7,8.0,0.0,0.00,0.90
7,nero + blu notte,12.8,81.0,1.0,0.77,0.30
8,nero + denim chiaro,36.5,85.0,0.0,1.00,0.45


In [45]:
# Sui capi veri: la palette è quella salvata nel parquet.
def palette_di(riga):
    p = riga["color_palette"]
    return json.loads(p) if isinstance(p, str) else p

a = salvato[salvato["title"].str.contains("CAMICIA UOMO LEE CAMP SHIRT", na=False)].iloc[0]
b = salvato[salvato["title"].str.startswith("BERMUDA", na=False)].iloc[0]

display(campioni(palette_di(a)[:3], a["title"]))
display(campioni(palette_di(b)[:3], b["title"]))
print("dominante contro dominante :", round(color_harmony_pair(palette_di(a)[0]["lab"], palette_di(b)[0]["lab"]), 3))
print("armonia sulle palette      :", round(color_harmony(palette_di(a), palette_di(b)), 3))
print("(soglia sotto cui il dominante decide da solo:", SOGLIA_STONO, ")")

dominante contro dominante : 0.867
armonia sulle palette      : 0.859
(soglia sotto cui il dominante decide da solo: 0.3 )


## 5.2 L'affinità di stile

### `coseno_pesato(a, b, A=None)`

- **Input** — due vettori di stile (5 registri) e, opzionale, la matrice di affinità.
- **Processo** — un coseno **generalizzato** `x'Ay / √(x'Ax · y'By)`. Il coseno secco misura se
  due capi occupano *gli stessi* registri, non se i loro registri **stanno bene insieme**: con
  cinque assi ortogonali una sneaker sportiva e un jeans streetwear davano 0,219, cioè
  l'abbinamento più comune del campionario risultava fra i peggiori. La matrice
  `AFFINITA_REGISTRI` dice quanto due registri convivono. Il risultato viene poi steso sopra
  il pavimento osservato (0,40) e **elevato al quadrato**, per restituire alla scala la
  dispersione che il coseno pesato comprimeva.
- **Output** — un valore 0–1.

### `style_match(vec_a, vec_b, formality_a, formality_b, formality_penalty_weight=0.5)`

- **Input** — i due vettori di stile e le due formalità normalizzate.
- **Processo** — `coseno_pesato` meno **metà della differenza di formalità**: un blazer
  sartoriale e dei pantaloncini da running non si abbinano nemmeno se condividono dei tag.
  (Nel codice c'è anche uno smorzamento per "segnale debole", oggi disattivato: aveva senso
  quando i tag erano bandierine ricavate dal testo, non ora che ogni vettore è una
  ripartizione che somma a 1 e porta quindi la stessa massa.)
- **Output** — l'affinità di stile, 0–1.

In [46]:
# ═══ nuvolari_generation_pipeline/scoring.py ═══════════════════════════════

# Soglia di norma sotto cui un capo è considerato "poco caratterizzato":
# per disegno è la mediana della norma dei vettori stile nel catalogo, così
# circa metà dei capi riceve un po' di smorzamento (vedi style_match).
#
# Rimisurata dopo la revisione visiva su tutti i 2901 capi e la ripesatura
# radice-IDF: 0.687 (era 0.574 sui soli vettori testuali). Il significato è
# cambiato in meglio: prima norma bassa voleva dire "scheda muta, non
# sappiamo cosa sia"; ora vuol dire "ha solo tratti comuni a tutti", perché
# i tag rari pesano di più. Resta giusto smorzare: di un capo che è solo
# genericamente casual sappiamo davvero poco di distintivo.
REFERENCE_STYLE_NORM = 0.69

# Coseno mediano fra due capi presi a caso nel catalogo: è, alla lettera, il
# valore di "non ne so niente". Verso questo si smorza un capo dal segnale
# debole, invece che verso zero (vedi style_match).
#
# Rimisurato su 20.000 coppie dopo revisione visiva e radice-IDF: 0.401
# (era 0.60 sui soli vettori testuali). Senza la ripesatura sarebbe stato
# 0.71, perché il tono di fondo "casual" della visione — presente sul 96% dei
# capi — rendeva tutto simile a tutto.
COSENO_NEUTRO = 0.40


# --------------------------------------------------------------------------
# Affinita' fra registri stilistici
# --------------------------------------------------------------------------
# Il coseno secco misura quanto due capi occupano GLI STESSI registri, non
# quanto i loro registri stanno bene insieme. Con cinque assi ortogonali per
# costruzione, ogni abbinamento misto veniva punito: sneaker (sportivo 0.7) e
# jeans (streetwear 0.6) davano coseno 0.219, cioe' l'abbinamento piu' comune
# del campionario risultava fra i peggiori. Misurato su coppie casuali, tutta
# la diagonale stava fra 0.80 e 0.94 e tutto il resto fra 0.03 e 0.41.
#
# Con questa matrice il prodotto diventa x'Ay: le caselle dicono quanto due
# registri convivono. I valori sono una scelta di gusto, non una misura --
# 'casual' e' il ponte verso tutto perche' e' cosi' che e' fatto il
# campionario (2397 capi su 2901 lo hanno come dominante), ed 'elegante' e'
# l'unico registro che si chiude davvero.
AFFINITA_REGISTRI = {
    ("sportivo",   "casual"):     0.85,
    ("sportivo",   "elegante"):   0.15,
    ("sportivo",   "streetwear"): 0.80,
    ("sportivo",   "da_mare"):    0.60,
    ("casual",     "elegante"):   0.65,
    ("casual",     "streetwear"): 0.85,
    ("casual",     "da_mare"):    0.75,
    ("elegante",   "streetwear"): 0.25,
    ("elegante",   "da_mare"):    0.20,
    ("streetwear", "da_mare"):    0.55,
}


# Coseno pesato piu' basso osservato fra due capi del catalogo: sotto questo
# valore la scala e' vuota. Rimisurare se la matrice cambia.
PAVIMENTO_AFFINITA = 0.40


def _matrice_affinita():
    """Costruita dall'ordine di STYLE_TAG_COLUMNS, non da un ordine scritto a
    mano: se un tag cambia posto la matrice segue, invece di scambiare due
    registri in silenzio."""
    tag = [c[len("style_"):] for c in STYLE_TAG_COLUMNS]
    n = len(tag)
    A = np.eye(n)
    for i in range(n):
        for j in range(i + 1, n):
            v = AFFINITA_REGISTRI.get((tag[i], tag[j]))
            if v is None:
                v = AFFINITA_REGISTRI.get((tag[j], tag[i]))
            if v is None:
                raise KeyError(f"affinita' mancante fra {tag[i]} e {tag[j]}")
            A[i, j] = A[j, i] = float(v)
    return A


MATRICE_AFFINITA = _matrice_affinita()


def coseno_pesato(a, b, A=None) -> float:
    """Coseno generalizzato x'Ay / sqrt(x'Ax * y'By). Vale 1 su vettori
    identici come il coseno normale, ma due registri diversi e affini non
    danno piu' zero."""
    A = MATRICE_AFFINITA if A is None else A
    na = float(a @ A @ a)
    nb = float(b @ A @ b)
    if na <= 0 or nb <= 0:
        return 0.0
    grezzo = float(a @ A @ b) / math.sqrt(na * nb)

    # Il coseno pesato comprime tutto in alto: misurato su 20.000 coppie
    # casuali dava mediana 0,955 e sd 0,086, cioe' l'ordine giusto ma nessuna
    # capacita' di distinguere. Le caselle piene della matrice alzano il
    # pavimento, non il soffitto, e sotto PAVIMENTO_AFFINITA non ci arriva
    # nessuno: quella parte di scala non porta informazione.
    #
    # Il quadrato non cambia l'ordine -- lo decide la matrice -- ma riporta la
    # dispersione a quella del coseno secco, cosi' la soglia degli slot
    # opzionali e il peso della penalita' di formalita' restano tarati.
    steso = (grezzo - PAVIMENTO_AFFINITA) / (1.0 - PAVIMENTO_AFFINITA)
    return max(0.0, min(1.0, steso)) ** 2


def style_match(style_vec_a, style_vec_b, formality_a: float, formality_b: float,
                 formality_penalty_weight: float = 0.5) -> float:
    """Coseno tra vettori stile, penalizzato dalla distanza di formalità E
    smorzato se uno dei due capi ha un segnale di stile debole.

    Il coseno da solo guarda solo la DIREZIONE del vettore, non la sua
    "forza": un capo il cui testo non dice quasi nulla di distintivo (es.
    un panciotto con description_text scarna, tutti i tag vicini a zero)
    può comunque risultare puntare "nella stessa direzione" di un capo dal
    segnale forte e ottenere un coseno altissimo — sembrando compatibile
    con qualunque cosa, anche quando in realtà semplicemente non sappiamo
    cosa sia quel capo stilisticamente. Smorzando per la norma minima dei
    due vettori (rispetto alla mediana del catalogo), un capo dal segnale
    debole non viene più trattato come "jolly" automatico.

    Lo smorzamento però tira verso COSENO_NEUTRO, non verso zero. Moltiplicare
    e basta affermava una cosa che non sappiamo: che i due capi sono lontani.
    Una felpa e un jeans con vettori IDENTICI (coseno 1,000) e la stessa
    formalità uscivano a 0,485, perché la felpa ha una scheda scarna — 46% del
    catalogo sta sotto la soglia e prendeva lo stesso trattamento. Ora un capo
    poco descritto finisce al valore che avrebbe con un capo qualsiasi: non
    diventa un jolly che batte tutti, e non viene nemmeno dichiarato
    incompatibile con qualcosa con cui va d'accordo.
    """
    a = np.asarray(style_vec_a, dtype=float)
    b = np.asarray(style_vec_b, dtype=float)
    norm_a, norm_b = np.linalg.norm(a), np.linalg.norm(b)
    cos = coseno_pesato(a, b) if norm_a > 0 and norm_b > 0 else 0.0

    # Smorzamento disattivato. Presupponeva che una norma bassa volesse dire
    # "non sappiamo cosa sia questo capo": vero finche' i tag erano bandierine
    # indipendenti ricavate dal testo, e una scheda muta lasciava il vettore
    # quasi a zero. Ora i punteggi sono una ripartizione che somma sempre a 1,
    # assegnata dalla revisione visiva su tutti e 2901 i capi: ogni vettore
    # porta la stessa massa e la norma misura solo QUANTO E' SPARSA la mistura,
    # non quanto ne sappiamo. Smorzare avrebbe colpito 368 capi, cioe' proprio
    # quelli a mistura voluta -- le camicie 0,6 casual / 0,4 elegante sono il
    # 25esimo percentile della norma -- appiattendo la distinzione piu' curata
    # che abbiamo. Il codice resta perche' tornerebbe valido se un giorno
    # entrassero in catalogo capi non revisionati.
    if False:  # pragma: no cover
        confidence = min(1.0, min(norm_a, norm_b) / REFERENCE_STYLE_NORM)
        cos = confidence * cos + (1 - confidence) * COSENO_NEUTRO

    formality_diff = abs(formality_a - formality_b)  # formality_norm è già 0-1
    penalty = formality_penalty_weight * formality_diff
    return max(0.0, cos - penalty)

In [47]:
registri = [c[len("style_"):] for c in STYLE_TAG_COLUMNS]
display(pd.DataFrame(MATRICE_AFFINITA, index=registri, columns=registri))

,sportivo,casual,elegante,streetwear,da_mare
sportivo,1.00,0.85,0.15,0.80,0.60
casual,0.85,1.00,0.65,0.85,0.75
elegante,0.15,0.65,1.00,0.25,0.20
streetwear,0.80,0.85,0.25,1.00,0.55
da_mare,0.60,0.75,0.20,0.55,1.00


In [48]:
def vettore_di(riga):
    return riga[STYLE_TAG_COLUMNS].to_numpy(dtype=float)

# I capi si scelgono dal PROFILO, non dal titolo: "CAMICIA SUIT JAY NAVY BLAZER"
# contiene la parola blazer ed e' una camicia -- e' esattamente l'errore che la
# fase 6 evita leggendo il titolo parola per parola (vedi classify_slot).
def campione_di(registro):
    return salvato.loc[salvato[f"style_{registro}"].idxmax()]

capi = {r: campione_di(r) for r in ["elegante", "sportivo", "streetwear", "casual"]}
print(pd.DataFrame([{"registro": r, "titolo": c["title"][:46],
                     "profilo": {k[len("style_"):]: round(c[k], 2) for k in STYLE_TAG_COLUMNS if c[k] > 0},
                     "formalita": round(c["formality_norm"], 2)} for r, c in capi.items()]).to_string(index=False))
print()

for ra, rb in [("sportivo", "streetwear"), ("casual", "streetwear"),
               ("elegante", "casual"), ("elegante", "sportivo")]:
    x, y = capi[ra], capi[rb]
    cos = coseno_pesato(vettore_di(x), vettore_di(y))
    sm  = style_match(vettore_di(x), vettore_di(y), x["formality_norm"], y["formality_norm"])
    d = abs(x["formality_norm"] - y["formality_norm"])
    print(f"{ra:11s} + {rb:11s}  coseno pesato {cos:.3f} | Δformalità {d:.2f} → penalità {0.5*d:.3f}"
          f" | style_match {sm:.3f}")

  registro                                     titolo                            profilo  formalita
  elegante           GIACCA UOMO NUVOLARI FLOWER NERO   {'casual': 0.1, 'elegante': 0.9}       0.75
  sportivo SCARPA UOMO SAUCONY PROGRID GUIDE 7 CANARY   {'sportivo': 0.9, 'casual': 0.1}       0.25
streetwear    FELPA UOMO PROPAGANDA RIBS LORD HD NERO {'casual': 0.1, 'streetwear': 0.9}       0.25
    casual                   GILET DONNA HAILYS CREAM                    {'casual': 1.0}       0.25

sportivo    + streetwear   coseno pesato 0.522 | Δformalità 0.00 → penalità 0.000 | style_match 0.522
casual      + streetwear   coseno pesato 0.632 | Δformalità 0.00 → penalità 0.000 | style_match 0.632
elegante    + casual       coseno pesato 0.263 | Δformalità 0.50 → penalità 0.250 | style_match 0.013
elegante    + sportivo     coseno pesato 0.000 | Δformalità 0.50 → penalità 0.250 | style_match 0.000


## 5.3 Le fantasie

### `stono_da_fantasie(pattern_a, pattern_b)`

- **Input** — le due fantasie (`tinta_unita`, `righe`, `quadri`, `stampato`, `camouflage`, `animalier`).
- **Processo** — **prodotto** delle due "forze", non somma: serve che si impongano entrambe.
  Basta che uno dei due capi sia tinta unita (forza 0) perché la penalità sparisca del tutto —
  è il caso normale, e non deve pagare niente.
- **Output** — 0–1, moltiplicato per 0,22 e sottratto al punteggio finale.

In [49]:
# ═══ nuvolari_generation_pipeline/scoring.py ═══════════════════════════════

# Colonne precalcolate da load_and_prepare (vedi outfit_generation). Non sono
# dati nuovi: sono gli stessi valori già pronti nella forma in cui servono.
# Il motivo è misurato — profilando la generazione del pool, il 70% del tempo
# se ne andava dentro pandas a rileggere per etichetta gli stessi dieci numeri
# a ogni confronto, e i confronti sono milioni.
COLONNA_VETTORE = "_vettore_stile"
COLONNA_PALETTE = "_palette"


def _vettore_stile(row: pd.Series):
    pronto = row.get(COLONNA_VETTORE)
    if pronto is not None:
        return pronto
    return row[STYLE_TAG_COLUMNS].to_numpy(dtype=float)


def _palette(row: pd.Series):
    pronta = row.get(COLONNA_PALETTE)
    if pronta is not None:
        return pronta
    grezza = row["color_palette"]
    return json.loads(grezza) if isinstance(grezza, str) else grezza


# Due fantasie forti addosso alla stessa persona si contendono lo sguardo: e'
# la regola di stile piu' vecchia che ci sia, e finora il campo "pattern" —
# calcolato per tutti i capi da Fase 2 e dalla revisione visiva — non lo leggeva
# nessuno.
#
# La forza dice quanto la fantasia si impone: le righe si portano sotto una
# giacca a quadri e non succede niente, un animalier e un camouflage insieme
# sono un'altra cosa. Una tinta unita vale 0, quindi basta che UNO dei due capi
# sia tinta unita perche' la penalita' sparisca del tutto: e' il caso normale,
# e non deve pagare nulla.
FANTASIA_FORZA = {
    "tinta_unita": 0.0,
    "righe": 0.45,
    "quadri": 0.55,
    "stampato": 0.85,
    "camouflage": 0.90,
    "animalier": 1.00,
}
PENALITA_FANTASIE = 0.22   # sottratta al punteggio quando entrambe sono al massimo


def stono_da_fantasie(pattern_a: str, pattern_b: str) -> float:
    """Quanto le due fantasie si contendono lo sguardo, da 0 a 1.

    E' il PRODOTTO delle due forze, non la somma: serve che siano forti
    entrambe. Un capo a righe con una tinta unita non e' un errore, e con la
    somma lo sarebbe stato a meta'.
    """
    fa = FANTASIA_FORZA.get(pattern_a, 0.0)
    fb = FANTASIA_FORZA.get(pattern_b, 0.0)
    return fa * fb

In [50]:
fant = list(FANTASIA_FORZA)
tabella = pd.DataFrame([[round(PENALITA_FANTASIE * stono_da_fantasie(x, y), 3) for y in fant] for x in fant],
                       index=fant, columns=fant)
print(f"penalità sottratta al punteggio (max {PENALITA_FANTASIE}):")
tabella

penalità sottratta al punteggio (max 0.22):


,tinta_unita,righe,quadri,stampato,camouflage,animalier
tinta_unita,0.0,0.000,0.000,0.000,0.000,0.000
righe,0.0,0.045,0.054,0.084,0.089,0.099
quadri,0.0,0.054,0.067,0.103,0.109,0.121
stampato,0.0,0.084,0.103,0.159,0.168,0.187
camouflage,0.0,0.089,0.109,0.168,0.178,0.198
animalier,0.0,0.099,0.121,0.187,0.198,0.220


## 5.4 Il punteggio completo

### `score_pair(row_a, row_b, w_colore=0.5, w_stile=0.5, use_palette=True)`

- **Input** — due **righe** di `features_clustered.parquet` (non due id: le righe portano già
  con sé palette, vettore e formalità).
- **Processo** — mette insieme i tre pezzi: `color_harmony` sulle palette, `style_match` sui
  vettori, meno la penalità di fantasia; poi taglia in [0, 1].
- **Output** — un dizionario con le **componenti separate** — `color_harmony`, `style_match`,
  `stono_fantasie` — e il totale `score`. Tenere le componenti visibili è ciò che permette di
  capire *perché* un abbinamento vale quanto vale, invece di doverlo indovinare.

È la funzione più chiamata di tutta la pipeline: la fase 6 la invoca milioni di volte, ed è il
motivo per cui `load_and_prepare` precalcola vettore e palette in due colonne apposta.

In [51]:
# ═══ nuvolari_generation_pipeline/scoring.py ═══════════════════════════════

def score_pair(row_a: pd.Series, row_b: pd.Series, w_colore: float = 0.5, w_stile: float = 0.5,
               use_palette: bool = True) -> dict:
    """score(A,B) completo tra due righe di features_clustered.parquet."""
    if use_palette:
        color_h = color_harmony(_palette(row_a), _palette(row_b))
    else:
        color_h = color_harmony_pair((row_a["L"], row_a["a"], row_a["b"]), (row_b["L"], row_b["a"], row_b["b"]))

    style_m = style_match(_vettore_stile(row_a), _vettore_stile(row_b),
                          row_a["formality_norm"], row_b["formality_norm"])

    stono = stono_da_fantasie(row_a.get("pattern"), row_b.get("pattern"))
    total = w_colore * color_h + w_stile * style_m - PENALITA_FANTASIE * stono
    total = max(0.0, min(1.0, total))
    return {"color_harmony": round(color_h, 3), "style_match": round(style_m, 3),
            "stono_fantasie": round(stono, 3), "score": round(total, 3)}

In [52]:
for x, y in [(capi["casual"], capi["streetwear"]),
             (capi["elegante"], capi["sportivo"]),
             (a, b)]:
    r = score_pair(x, y)
    print(f"{x['title'][:36]:38s} + {y['title'][:36]:38s}")
    print(f"   {r}")

GILET DONNA HAILYS CREAM               + FELPA UOMO PROPAGANDA RIBS LORD HD N  
   {'color_harmony': 0.897, 'style_match': 0.632, 'stono_fantasie': 0.0, 'score': 0.764}
GIACCA UOMO NUVOLARI FLOWER NERO       + SCARPA UOMO SAUCONY PROGRID GUIDE 7   
   {'color_harmony': 0.9, 'style_match': 0.0, 'stono_fantasie': 0.0, 'score': 0.45}
CAMICIA UOMO LEE CAMP SHIRT SOY BEAN   + BERMUDA UOMO NUVOLARI LOKO COLONIALE  
   {'color_harmony': 0.859, 'style_match': 1.0, 'stono_fantasie': 0.0, 'score': 0.93}


In [53]:
# Come è fatta la scala: 4000 coppie a caso fra i capi abbinabili.
rng = random.Random(12345)
abbinabili = salvato[~salvato["needs_vision_review"].astype(bool)].reset_index(drop=True)
punteggi = [score_pair(abbinabili.iloc[rng.randrange(len(abbinabili))],
                       abbinabili.iloc[rng.randrange(len(abbinabili))])["score"]
            for _ in range(4000)]
s = pd.Series(punteggi)
print(s.describe().round(3).to_string())
print("\ndistribuzione:")
print(pd.cut(s, bins=[0, .2, .4, .46, .6, .8, 1.0]).value_counts().sort_index().to_string())

count    4000.000
mean        0.735
std         0.153
min         0.111
25%         0.648
50%         0.769
75%         0.852
max         0.950

distribuzione:
(0.0, 0.2]       14
(0.2, 0.4]      115
(0.4, 0.46]     142
(0.46, 0.6]     468
(0.6, 0.8]     1638
(0.8, 1.0]     1623


---
# 6. Accoppiamento — Fase 6

Qui i capi diventano outfit. Tre momenti distinti:

1. **preparare** il catalogo, cioè dedurre dal titolo quello che il parquet non ha
   (slot, genere, manica, gamba, forma del capo);
2. **filtrare** i candidati ammissibili per uno slot, dato quello che si è già scelto;
3. **cercare** la combinazione migliore con una beam search, e giudicarla.

## 6.1 Preparare il catalogo

### `classify_slot(title)` e `slot_da_percorso(relpath)`

- **Input** — il titolo del capo (e, come rete, il percorso di categoria).
- **Processo** — scandisce il titolo **parola per parola da sinistra** e prende il primo
  riscontro nel dizionario dei tipi. Non "la prima categoria che compare da qualche parte nel
  testo": nel nome colore `N.BLAZER/B TAN` di una sneaker la parola `blazer` la faceva
  classificare capospalla, e `jeans` dentro il marchio Guess Jeans veniva letto come tipo di capo.
  Il trattino resta dentro la parola, altrimenti `t-shirt` si spezza in due pezzi irriconoscibili.
- **Output** — uno fra `top`, `bottom`, `shoes`, `outerwear`, `accessory`, `dress_or_suit`,
  oppure `None` (capo escluso dalla generazione).

### `classify_sleeve(title, relpath)` · `classify_leg_length(title)` · `classify_gender(title, relpath)`

- **Input** — titolo e percorso.
- **Processo** — manica corta/lunga (per i capi ambigui come la camicia si guarda la
  sottocategoria del sito, poi il testo; in mancanza **lunga**, che è la scelta conservativa),
  gamba corta/lunga, e genere (`DONNA`/`UOMO` nel titolo, poi il percorso, poi la regola sugli
  accessori maschili; altrimenti `None` = compatibile con entrambi).
- **Output** — rispettivamente `"corta"|"lunga"`, `"corta"|"lunga"`, `"uomo"|"donna"|None`.

### `classify_calzatura` · `classify_top_senza_collo` · `classify_cappotto` · `classify_bottom_da_tuta`

- **Input** — titolo e, per le prime e le terze, la **descrizione** presa dal catalogo.
- **Processo** — riconoscono la **forma** del capo, che il titolo da solo non dà: la scarpa da
  barca che si chiama "sneakers", il caban che si chiama "giubbotto". Servono alle regole 3 e 4.
- **Output** — un booleano per capo.

### `load_and_prepare(features_parquet)`

- **Input** — il percorso di `features_clustered.parquet`.
- **Processo** — legge il parquet e ci attacca le colonne dedotte qui sopra; ricostruisce
  `season` dalle quattro colonne one-hot; legge le descrizioni dal catalogo **solo** per scarpe
  e capispalla (453 schede su 2901, non tutte); infine precalcola vettore di stile e palette in
  due colonne dedicate — non sono dati nuovi, sono gli stessi già pronti nella forma in cui
  servono, e senza di loro il 70% del tempo di generazione se ne andava dentro pandas.
- **Output** — il `DataFrame` su cui lavora tutta la fase 6.

La prima cella porta anche `DISTANZA_MINIMA_VARIANTI`, `TOLLERANZA_PAREGGIO` e `IndiceNovita`,
che nel modulo stanno in cima e che si documentano nella sezione 7 (il pool): `build_outfits`
usa la tolleranza di pareggio per scegliere fra capi facoltativi a punteggio identico, quindi
devono esistere prima.

In [54]:
# ═══ nuvolari_generation_pipeline/outfit_generation.py ═══════════════════════════════

GARMENT_TYPE_KEYWORDS = {
    "top": ["camicia", "camicie", "t-shirt", "tshirt", "maglia", "maglie", "maglietta",
            "polo", "canotta", "canotte", "felpa", "felpe", "top", "blusa", "camicetta",
            "body", "overshirt", "maglione", "pullover", "pull", "cardigan"],
    "bottom": ["pantalone", "pantaloni", "jeans", "bermuda", "short", "shorts", "gonna",
               "leggings", "pantaloncini"],
    "outerwear": ["giacca", "giacche", "giubbotto", "giubbotti", "cappotto", "piumino",
                  "bomber", "parka", "trench", "blazer", "gilet", "smanicato", "tracktop",
                  "anorak", "jkt", "windbreaker", "giaccone"],
    "shoes": ["scarpe", "scarpa", "sneakers", "sneaker", "stivaletto", "stivaletti",
              "mocassino", "mocassini", "sandali", "sandalo", "ciabatte", "stringate",
              "ballerine"],
    "accessory": ["borsa", "tracolla", "zaino", "cintura", "cappello", "sciarpa", "guanti",
                  "occhiali", "portafoglio", "calzini", "cravatta", "berretto"],
    # abiti/completi/costumi sono "capi unici" (coprono da soli top+bottom):
    # fuori scope per la generazione a slot in questa v1, vedi nota sotto.
    "dress_or_suit": ["abito", "vestito", "tuta", "completo", "costume"],
}

# Quanti capi devono cambiare perche' due outfit contino come proposte diverse
# e non come lo stesso outfit ritoccato. La distanza e' quanti capi di uno
# mancano all'altro, presa nel verso peggiore: sostituire un pantalone vale 1,
# e cosi' vale 1 anche aggiungere soltanto un cappello a un outfit esistente.
# A 2 servono due capi cambiati davvero.
#
# A 1 (cioe' senza regola) il pool si riempiva di quasi-doppioni: il 76% delle
# coppie generate dalla stessa ancora differiva di un capo solo su cinque, e
# triplicare il pool aggiungeva 21 capi distinti su 2514.
DISTANZA_MINIMA_VARIANTI = 2

# Quanto due punteggi possono distare e contare ancora come pari merito nello
# slot opzionale. A 0 si ruota solo fra punteggi identici al millesimo, quindi
# la rotazione non costa NIENTE in qualita': i capi scambiati sono equivalenti
# per il punteggio, non semplicemente vicini.
TOLLERANZA_PAREGGIO = 0.0


class IndiceNovita:
    """Dice se un outfit e' abbastanza diverso da TUTTI quelli gia' accettati.

    Il confronto ingenuo e' quadratico — a 7000 outfit sono 49 milioni di
    confronti fra insiemi. Ma due outfit possono essere troppo simili solo se
    condividono quasi tutti i capi, quindi basta guardare quelli che hanno
    almeno un capo in comune e contare le sovrapposizioni: l'indice inverso
    capo -> outfit riduce il confronto a una manciata di candidati.
    """

    def __init__(self, distanza_minima: int = DISTANZA_MINIMA_VARIANTI):
        self.distanza_minima = distanza_minima
        self._per_capo = collections.defaultdict(list)
        self._firme = []

    def __len__(self):
        return len(self._firme)

    def e_nuovo(self, firma: frozenset) -> bool:
        conteggio = collections.Counter()
        for capo in firma:
            for i in self._per_capo.get(capo, ()):
                conteggio[i] += 1
        for i, comuni in conteggio.items():
            altra = self._firme[i]
            # |A - B| e |B - A|: si prende il verso piu' generoso verso di noi,
            # cioe' il massimo, altrimenti un outfit contenuto in un altro
            # (stessi capi piu' un cappello) passerebbe per nuovo.
            if max(len(firma) - comuni, len(altra) - comuni) < self.distanza_minima:
                return False
        return True

    def aggiungi(self, firma: frozenset) -> None:
        i = len(self._firme)
        self._firme.append(firma)
        for capo in firma:
            self._per_capo[capo].append(i)

MANDATORY_SLOTS = ["top", "bottom", "shoes"]
OPTIONAL_SLOTS = ["outerwear", "accessory"]
# Sotto questa soglia, meglio niente accessorio/capospalla che uno stonato.
# Allineata alla soglia di accettazione del pool (0.46 — riancorata per
# percentile dopo la revisione visiva: vedi COSENO_NEUTRO in scoring.py, la
# scala dei punteggi si è compressa verso il basso e 0.46 occupa oggi lo
# stesso posto che 0.6 occupava prima): quando stava a 0.5 un
# capo opzionale poteva trascinare l'outfit nella fascia 0.5-0.6, dove il pool
# scarta l'outfit INTERO — il capo facoltativo, per definizione rinunciabile,
# affondava anche i tre obbligatori che da soli sarebbero passati.
OPTIONAL_SLOT_MIN_SCORE = 0.46
# I candidati outlier (style_cluster=-1) per il capospalla non sono più
# esclusi in blocco: il 70% dei capispalla eleggibili è outlier — non perché
# stonino, ma perché la scheda di un giubbotto dice poco e HDBSCAN non li
# aggrega — e l'esclusione concentrava 956 presenze su 34 giacche (la più
# usata compariva 170 volte). Al posto del divieto, un'asticella più alta:
# l'outlier entra solo se l'outfit resta comunque BUONO, non appena passabile.
OPTIONAL_OUTLIER_MIN_SCORE = 0.60


_KEYWORD_TO_SLOT = {kw: slot for slot, kws in GARMENT_TYPE_KEYWORDS.items() for kw in kws}


def _find_slot_and_keyword(title: str):
    """Il tipo di capo è quasi sempre la prima parola del titolo (es.
    "SCARPE HEYDUDE..."), ma altre parole del titolo (es. il nome del
    colore "N.BLAZER/B TAN" di una sneaker) possono corrispondere per
    caso a una parola chiave di UN'ALTRA categoria. Per questo si scansiona
    il titolo parola per parola, da sinistra, e si prende il primo match —
    non "la prima categoria del dizionario con un match ovunque nel testo"."""
    if not title:
        return None, None
    # il trattino va mantenuto nel token: "t-shirt" tokenizzato senza trattino
    # diventerebbe "t" + "shirt", nessuno dei due presente nel dizionario, e la
    # scansione proseguirebbe fino a un'altra parola nel titolo che potrebbe
    # corrispondere per sbaglio a un'ALTRA categoria (successo con capi di
    # marchi tipo "Guess Jeans": "jeans" nel nome del brand veniva letto come
    # tipo di capo, non "t-shirt" come tipo di capo reale)
    words = re.findall(r"[a-zà-ü'-]+", title.lower())
    for w in words:
        if w in _KEYWORD_TO_SLOT:
            return _KEYWORD_TO_SLOT[w], w
    return None, None


def classify_slot(title: str):
    slot, _keyword = _find_slot_and_keyword(title)
    return slot


# Ripiego sul percorso di categoria quando il titolo non nomina il tipo di
# capo ("DREW PEAK CREW NF0A4SVR..."). Erano 165 capi invisibili alla
# generazione; per 57 la categoria stava scritta nel percorso. Le chiavi sono
# i segmenti reali del catalogo, confrontati per segmento intero — non come
# sottostringhe, per non leggere "lacoste" dentro un percorso brand.
_SEGMENTO_SLOT = {
    "felpe": "top", "t-shirt": "top", "camicie": "top", "polo": "top",
    "maglieria": "top", "canotte": "top",
    "pantaloni": "bottom", "jeans": "bottom", "bermuda": "bottom",
    "giubbotti": "outerwear", "giubbotti-donna": "outerwear", "gilet": "outerwear",
    "scarpe": "shoes", "scarpe-donna": "shoes", "sneakers": "shoes",
    "stivaletti": "shoes", "sandali": "shoes",
    "calzini-uomo": "accessory", "calzini": "accessory", "borse": "accessory",
    "sciarpe": "accessory", "occhiali": "accessory", "cappelli": "accessory",
    "cinture": "accessory",
}


def slot_da_percorso(relpath: str):
    for segmento in (relpath or "").lower().split("/"):
        if segmento in _SEGMENTO_SLOT:
            return _SEGMENTO_SLOT[segmento]
    return None

In [55]:
# ═══ nuvolari_generation_pipeline/outfit_generation.py ═══════════════════════════════

SHORT_SLEEVE_KEYWORDS = {"t-shirt", "tshirt", "polo", "canotta", "canotte", "maglietta", "top"}
LONG_SLEEVE_KEYWORDS = {"maglia", "maglie", "felpa", "felpe", "maglione", "pullover", "pull",
                         "cardigan", "overshirt"}
# camicia/camicie/blusa/camicetta/body: manica ambigua dal solo tipo di capo,
# serve la sottocategoria del sito o il testo (vedi classify_sleeve)

SHORT_LEG_KEYWORDS = {"bermuda", "short", "shorts", "pantaloncini"}


def classify_sleeve(title: str, relpath: str):
    """Lunghezza manica del top: 'corta' o 'lunga' — necessaria per la
    Regola 1 (bermuda nell'outfit -> il top deve avere maniche corte,
    altrimenti stona per coerenza stagionale). Alcuni tipi di capo hanno la
    manica implicita nel nome (t-shirt/polo/canotta = sempre corta; maglia/
    felpa/maglione = sempre lunga); per camicia/blusa (ambigue) si usa prima
    la sottocategoria del sito (.../manica-lunga/... o .../mezza-manica/...,
    copre il 97% delle camicie), poi il testo del titolo. Se non
    determinabile, default 'lunga' — scelta conservativa: meglio escludere
    un top per sbaglio che abbinarlo a un bermuda se in realtà è invernale.
    """
    _slot, keyword = _find_slot_and_keyword(title)
    if keyword in SHORT_SLEEVE_KEYWORDS:
        return "corta"
    if keyword in LONG_SLEEVE_KEYWORDS:
        return "lunga"
    rp = (relpath or "").lower()
    if "manica-lunga" in rp:
        return "lunga"
    if "mezza-manica" in rp:
        return "corta"
    t = (title or "").lower()
    if "manica corta" in t or "mezza manica" in t:
        return "corta"
    if "manica lunga" in t:
        return "lunga"
    return "lunga"


def classify_leg_length(title: str):
    """Lunghezza gamba del bottom: 'corta' (bermuda/short/pantaloncini) o
    'lunga' (pantaloni/jeans/leggings). Serve sia alla Regola 1 (coerenza
    col top) sia alla Regola 2 (niente giacche sui pantaloncini)."""
    _slot, keyword = _find_slot_and_keyword(title)
    return "corta" if keyword in SHORT_LEG_KEYWORDS else "lunga"


# --- Regola 3 (utente): il mocassino vuole un top con un collo -------------
#
# "Un mocassino è troppo elegante per stare con maglie che non sono polo o
# camicie". La formalità da sola non bastava a tenerli separati: un mocassino
# sta a 3 e una t-shirt a 2, cioè un solo gradino, e FORMALITY_SPREAD_MAX ne
# tollera esattamente uno.
#
# Solo parole che nominano LA FORMA. Restano fuori di proposito:
#   slip-on   -> lo sono anche le Vans Authentic
#   stringato -> vuol dire allacciato, lo sono anche le running
#   ballerina -> è una scarpa da donna piatta, non un mocassino
CALZATURA_ELEGANTE = re.compile(
    r"(mocassin[oi]|loafer|penny loafer|scarp[ae] da barca|boat shoe)", re.I)

# Frase di repertorio del catalogo: elenca i reparti del negozio, non descrive
# il prodotto. Senza questa esclusione un anfibio e una New Balance risultavano
# mocassini perché la scheda diceva "disponibili in vari stili come sneakers,
# scarpe eleganti, stivali o mocassini".
DESCRIZIONE_DI_REPERTORIO = re.compile(r"variet[àa] di stili come", re.I)

# Quando è la sola descrizione a parlare, si accettano solo le forme che un
# copywriter non usa per traslato. "Mocassino" da solo non basta: la scheda
# della Vans Authentic — una skate shoe in tela, verificata sulla foto — la
# chiama "questo mocassino sportivo". "Scarpa da barca" e "loafer" no: quelle
# nominano un oggetto preciso.
FORMA_INEQUIVOCABILE = re.compile(r"(scarp[ae] da barca|boat shoe|loafer)", re.I)

# Top che con un mocassino non ci stanno. La maglieria (maglia, maglione,
# cardigan) NON è qui: il maglione col mocassino è un abbinamento classico,
# e in italiano "maglia" vuol dire quasi sempre quello.
TOP_SENZA_COLLO = {"t-shirt", "tshirt", "maglietta", "canotta", "canotte", "top", "felpa", "felpe"}


def classify_calzatura(title: str, descrizione: str = "") -> bool:
    """Vero se la scarpa ha la forma del mocassino o della scarpa da barca.

    Guarda anche la descrizione, non solo il titolo, perché il titolo mente: la
    scarpa in camoscio beige della segnalazione si chiama "SNEAKERS UOMO FRED
    PERRY B2346" ed è archiviata fra le sneakers, ma la sua stessa scheda dice
    "rilegge la silhouette classica della scarpa da barca". Sono 7 casi su 28:
    un quarto dei mocassini del catalogo non si dichiara tale nel titolo.
    """
    if CALZATURA_ELEGANTE.search(title or ""):
        return True
    if DESCRIZIONE_DI_REPERTORIO.search(descrizione or ""):
        return False
    return bool(FORMA_INEQUIVOCABILE.search(descrizione or ""))


def classify_top_senza_collo(title: str) -> bool:
    """Vero se il top è una t-shirt, una canotta, un top o una felpa."""
    _slot, keyword = _find_slot_and_keyword(title)
    return keyword in TOP_SENZA_COLLO


# --- Regola 4 (utente): il cappotto non va sopra una tuta -----------------
#
# Stessa storia del mocassino, sul capospalla: "giubbotto" non dice il registro.
# Il caban di lana blu doppiopetto a sei bottoni — verificato sulla foto del
# prodotto — si chiama GIUBBOTTO UOMO NUVOLARI OTIS NAVY e sta a formalità 2,
# quindi finiva sopra una tuta con le sneakers.
#
# Il vincolo non passa dalla formalità di proposito: alzare quel capo da 2 a 3
# non avrebbe tolto l'outfit (la dispersione ammessa è esattamente un gradino,
# 0.25 <= 0.30), e alzarlo a 4 lo avrebbe tolto anche dagli abbinamenti giusti
# col jeans. La formalità è un asse solo: dice quanto, non dice con cosa.
FORMA_CAPPOTTO = re.compile(r"(caban|montgomery|doppiopetto|sei bottoni|peacoat|cappotto|trench)", re.I)

# Un gilet imbottito e una giacca a vento citano il cappotto nella scheda ma
# cappotti non sono: il titolo qui è affidabile e ha l'ultima parola.
NON_E_UN_CAPPOTTO = re.compile(r"\b(gilet|giacca a vento|piumino|smanicato)\b", re.I)

BOTTOM_DA_TUTA = re.compile(r"\b(tuta|jogger\w*|sweatpant\w*)\b", re.I)


def classify_cappotto(title: str, descrizione: str = "") -> bool:
    """Vero se il capospalla ha la forma di un cappotto (caban compreso)."""
    if NON_E_UN_CAPPOTTO.search(title or ""):
        return False
    if FORMA_CAPPOTTO.search(title or ""):
        return True
    if DESCRIZIONE_DI_REPERTORIO.search(descrizione or ""):
        return False
    return bool(FORMA_CAPPOTTO.search(descrizione or ""))


def classify_bottom_da_tuta(title: str) -> bool:
    return bool(BOTTOM_DA_TUTA.search(title or ""))


def _descrizioni(relpath_voluti: set) -> dict:
    """relpath -> description_text, solo per i capi richiesti.

    Il parquet porta il titolo ma non la descrizione, e la descrizione è
    l'unico posto dove sta la forma del capo: la scarpa da barca che si chiama
    "sneakers", il caban che si chiama "giubbotto". Si legge dal catalogo, che è
    la stessa fonte da cui il parquet è nato. Se il catalogo non c'è (una copia
    del solo codice) si continua senza: le regole perdono i casi che si
    dichiarano solo nella scheda, non l'intero funzionamento.
    """
    if not CATALOGO.exists():
        return {}
    out = {}
    for p in CATALOGO.rglob("metadata.json"):
        rel = str(p.parent.relative_to(CATALOGO))
        if rel not in relpath_voluti:
            continue
        try:
            out[rel] = json.loads(p.read_text(encoding="utf-8")).get("description_text") or ""
        except (OSError, ValueError):
            continue
    return out


# Accessori che, quando il titolo non dichiara il genere, non vanno trattati
# come unisex. Sono capi connotati: una tracolla street senza indicazione
# risultava "neutra" e finiva su outfit donna. Cappelli, occhiali, zaini e
# sciarpe restano invece genuinamente trasversali e non compaiono qui.
ACCESSORI_MASCHILI = r"\b(borsa|borse|tracolla|tracolle|marsupio|marsupi)\b"


def classify_gender(title: str, relpath: str):
    t = (title or "").lower()
    # Confine di parola solo in coda: il catalogo scrive il brand attaccato al
    # genere ("CAMICIA ICHIDONNA IHESTAMA"), e con \bdonna\b quel capo restava
    # senza genere — cioè jolly — e finiva negli outfit uomo. Otto capi ICHI
    # erano in questo stato; uno, un chino donna, era in un outfit uomo con
    # polo e mocassini. Nessun titolo del catalogo contiene parole che
    # finiscono in -donna o -uomo con un altro significato (misurato).
    if re.search(r"donna\b", t):
        return "donna"
    if re.search(r"uomo\b", t):
        return "uomo"
    # fallback sul percorso di categoria, meno affidabile del titolo
    if "abbigliamento-donna" in (relpath or ""):
        return "donna"
    # borse e tracolle senza genere dichiarato: assegnate a uomo invece che
    # lasciate neutre, altrimenti il jolly del genere le fa entrare ovunque
    if re.search(ACCESSORI_MASCHILI, t):
        return "uomo"
    return None  # genere non determinato -> trattato come compatibile con entrambi


SEASONS = ["estate", "inverno", "mezza_stagione", "tutte"]


def _season_label(row: pd.Series) -> str:
    for s in SEASONS:
        if row.get(f"season_{s}") == 1.0:
            return s
    return "tutte"


def load_and_prepare(features_parquet: str) -> pd.DataFrame:
    """Carica features_clustered.parquet e aggiunge le colonne slot/gender/
    season/sleeve/leg_length necessarie per la generazione outfit (season è
    ricostruita dalle colonne one-hot season_* della Fase 3; le altre sono
    dedotte dal titolo/percorso, vedi sopra)."""
    df = pd.read_parquet(features_parquet)
    df["slot"] = df["title"].apply(classify_slot)
    # titolo muto -> si guarda il percorso di categoria (vedi slot_da_percorso)
    manca = df["slot"].isna()
    df.loc[manca, "slot"] = df.loc[manca, "relpath"].apply(slot_da_percorso)
    df["gender"] = df.apply(lambda r: classify_gender(r["title"], r["relpath"]), axis=1)
    df["season"] = df.apply(_season_label, axis=1)
    df["sleeve"] = df.apply(lambda r: classify_sleeve(r["title"], r["relpath"]), axis=1)
    df["leg_length"] = df["title"].apply(classify_leg_length)

    # La forma del capo è l'unico attributo che il titolo non basta a dare (vedi
    # classify_calzatura e classify_cappotto): serve la descrizione, che sta nel
    # catalogo e non nel parquet. Si legge solo per scarpe e capispalla — 453
    # schede su 2901.
    interessati = set(df.loc[df["slot"].isin(["shoes", "outerwear"]), "relpath"])
    descrizioni = _descrizioni(interessati)
    df["mocassino"] = df.apply(
        lambda r: r["slot"] == "shoes"
        and classify_calzatura(r["title"], descrizioni.get(r["relpath"], "")), axis=1)
    df["top_senza_collo"] = df.apply(
        lambda r: r["slot"] == "top" and classify_top_senza_collo(r["title"]), axis=1)
    df["cappotto"] = df.apply(
        lambda r: r["slot"] == "outerwear"
        and classify_cappotto(r["title"], descrizioni.get(r["relpath"], "")), axis=1)
    df["bottom_da_tuta"] = df.apply(
        lambda r: r["slot"] == "bottom" and classify_bottom_da_tuta(r["title"]), axis=1)

    # Vettore di stile e tavolozza pronti da usare. Sono gli stessi valori di
    # sempre, solo già estratti: score_pair viene chiamata milioni di volte, e
    # rifare a ogni chiamata la selezione pandas per etichetta e il json.loads
    # della tavolozza costava il 70% del tempo di generazione del pool.
    df[COLONNA_VETTORE] = list(df[STYLE_TAG_COLUMNS].to_numpy(dtype=float))
    df[COLONNA_PALETTE] = df["color_palette"].apply(
        lambda p: json.loads(p) if isinstance(p, str) else p)
    return df

In [56]:
t0 = time.time()
df = load_and_prepare(str(CLUSTERED))
print(f"load_and_prepare: {df.shape[0]} capi, {df.shape[1]} colonne, {time.time()-t0:.1f}s")
print()
print("colonne aggiunte:", [c for c in df.columns if c not in salvato.columns])

load_and_prepare: 2901 capi, 37 colonne, 0.3s

colonne aggiunte: ['slot', 'gender', 'season', 'sleeve', 'leg_length', 'mocassino', 'top_senza_collo', 'cappotto', 'bottom_da_tuta', '_vettore_stile', '_palette']


In [57]:
prove = ["T-SHIRT UOMO OBEY LIPS VINTAGE BLACK",
         "SNEAKERS UOMO NIKE N.BLAZER/B TAN",
         "BERMUDA DONNA PROJ3CT CORDOVA DENIM BLK",
         "CAMICIA UOMO RAYZ CLAN SABBIA/NERO",
         "GIACCA K-WAY LE VRAI CLAUDE",
         "CAPPELLO GOORIN BROS. BLACK SHEEP VOID"]
pd.DataFrame([{"titolo": t,
               "slot": classify_slot(t),
               "manica": classify_sleeve(t, ""),
               "gamba": classify_leg_length(t),
               "genere": classify_gender(t, "")} for t in prove])

,titolo,slot,manica,gamba,genere
0,T-SHIRT UOMO OBEY LIPS VINTAGE BLACK,top,corta,lunga,uomo
1,SNEAKERS UOMO NIKE N.BLAZER/B TAN,shoes,lunga,lunga,uomo
2,BERMUDA DONNA PROJ3CT CORDOVA DENIM BLK,bottom,lunga,corta,donna
3,CAMICIA UOMO RAYZ CLAN SABBIA/NERO,top,lunga,lunga,uomo
4,GIACCA K-WAY LE VRAI CLAUDE,outerwear,lunga,lunga,None
5,CAPPELLO GOORIN BROS. BLACK SHEEP VOID,accessory,lunga,lunga,None


In [58]:
print("capi per slot (già escluso chi va rivisto):")
ammessi = df[~df["needs_vision_review"].astype(bool)]
print(ammessi["slot"].value_counts(dropna=False).to_string())
print()
print("obbligatori:", MANDATORY_SLOTS, " facoltativi:", OPTIONAL_SLOTS)
print("dress_or_suit e slot None restano fuori: coprono da soli top+bottom o non si riconoscono.")

capi per slot (già escluso chi va rivisto):
slot
top              1397
bottom            689
outerwear         274
shoes             236
accessory         182
dress_or_suit      69
None               54

obbligatori: ['top', 'bottom', 'shoes']  facoltativi: ['outerwear', 'accessory']
dress_or_suit e slot None restano fuori: coprono da soli top+bottom o non si riconoscono.


## 6.2 I filtri sui candidati

Otto controlli decidono se un capo può entrare in uno slot. I primi tre sono predicati
elementari, gli altri guardano l'outfit già composto.

### `_gender_compatible(a, b)` · `_season_compatible(a, b)` · `_cluster_compatible(a, b)`

- **Input** — i due valori da confrontare.
- **Processo** — genere: uguali, o almeno uno indeterminato. Stagione: uguali, o almeno una
  `tutte`. Cluster: uguali, **oppure uno dei due è outlier** — escludere a priori gli outlier
  sarebbe uno spreco, e in quel caso decide solo il punteggio.
- **Output** — booleano.

### `_formality_ok(items, candidate_formality)`

- **Input** — i capi già scelti e la formalità del candidato.
- **Processo** — verifica che aggiungerlo non allarghi la **dispersione** dell'outfit oltre
  `FORMALITY_SPREAD_MAX = 0.3`, cioè un solo gradino su una scala a passi di 0,25. Non basta che
  il candidato stia bene *in coppia* con ciascuno: l'insieme deve restare coerente.
- **Output** — booleano.

### `_seasonal_coherence_ok(current_items, slot, candidate_row)`

- **Input** — i capi già scelti, lo slot da riempire, la riga del candidato.
- **Processo** — la Regola 1: con un bottom a gamba corta il top **deve** avere manica corta;
  nessun capo `inverno` in **nessuno slot** se gambe o braccia sono scoperte; e viceversa nessun
  capo corto se c'è già un invernale. È unidirezionale: manica corta con pantaloni lunghi resta
  ammesso. Prima il controllo esisteva solo fra top e bottom, e accessori e capispalla erano
  l'unica via per cui una sciarpa entrava in un outfit estivo.
- **Output** — booleano.

### `_calzatura_ok(...)` · `_cappotto_ok(...)` · `_outerwear_allowed(current_items)`

- **Input** — gli stessi.
- **Processo** — Regola 3: niente t-shirt, canotte o felpe sotto un mocassino. Regola 4: niente
  cappotto sopra una tuta. Regola 2: con i pantaloncini lo slot capospalla si salta del tutto.
  Le prime due valgono nei **due ordini** di riempimento, perché le scarpe possono entrare
  prima o dopo il top.
- **Output** — booleano.

### `candidates_for_slot(df, slot, anchor_row, used_relpaths, require_real_cluster=False, current_items=None)`

- **Input** — il catalogo preparato, lo slot da riempire, l'**ancora**, i capi già usati e
  (opzionale) il parziale corrente.
- **Processo** — applica la catena: slot giusto → non da rivedere → non già usato → genere,
  stagione e cluster compatibili **con l'ancora** → e, se c'è un parziale, genere e stagione
  compatibili con **tutti** i capi già scelti, dispersione di formalità, e le regole 1, 3, 4.
  Il confronto con tutti e non con la sola ancora non è un dettaglio: se l'ancora ha il valore
  indeterminato è compatibile con chiunque, e due capi incompatibili *fra loro* passavano
  entrambi — 31 outfit mescolavano uomo e donna, 138 estate e inverno.
- **Output** — il sotto-`DataFrame` dei capi ammissibili.

In [59]:
# ═══ nuvolari_generation_pipeline/outfit_generation.py ═══════════════════════════════

def _gender_compatible(a: str, b: str) -> bool:
    return a is None or b is None or a == b


def _season_compatible(a: str, b: str) -> bool:
    return a == "tutte" or b == "tutte" or a == b


def _cluster_compatible(a: int, b: int) -> bool:
    if a == -1 or b == -1:
        return True  # outlier: nessun filtro di cluster, decide solo score()
    return a == b


FORMALITY_SPREAD_MAX = 0.3  # formality_norm ha 5 livelli discreti (0, .25, .5, .75, 1) -> tollera un solo "gradino"


def _formality_range(items) -> tuple:
    values = [row["formality_norm"] for _, row in items]
    return min(values), max(values)


def _formality_ok(items, candidate_formality: float) -> bool:
    """Verifica che aggiungere candidate_formality non allarghi la dispersione
    di formalità dell'outfit oltre FORMALITY_SPREAD_MAX. Non basta che il
    candidato stia bene in coppia con ogni singolo capo già scelto (vedi
    style_match) — l'outfit nel suo insieme deve restare coerente: un pezzo
    molto più formale/informale degli altri stona anche se il suo score
    pairwise con ciascuno preso singolarmente sembrava accettabile."""
    lo, hi = _formality_range(items)
    new_lo, new_hi = min(lo, candidate_formality), max(hi, candidate_formality)
    return (new_hi - new_lo) <= FORMALITY_SPREAD_MAX


def _existing_value(current_items, slot: str, column: str):
    return next((row[column] for s, row in current_items if s == slot), None)


def _seasonal_coherence_ok(current_items, slot: str, candidate_row: pd.Series) -> bool:
    """Regola 1 (utente): se l'outfit ha un bottom a gamba corta (bermuda/
    short), il top deve avere maniche corte — un top invernale a maniche
    lunghe coi pantaloncini stona per coerenza stagionale. La regola è
    unidirezionale (un top a maniche corte con pantaloni lunghi è normale,
    es. mezza stagione), quindi si applica in entrambi gli ordini di
    riempimento slot senza vietare quel caso."""
    if slot == "top":
        existing_bottom_leg = _existing_value(current_items, "bottom", "leg_length")
        if existing_bottom_leg == "corta" and candidate_row["sleeve"] != "corta":
            return False
    elif slot == "bottom":
        existing_top_sleeve = _existing_value(current_items, "top", "sleeve")
        if candidate_row["leg_length"] == "corta" and existing_top_sleeve == "lunga":
            return False

    # Un capo dichiaratamente invernale non convive con la pelle scoperta,
    # qualunque slot occupi: la sciarpa sopra la t-shirt a maniche corte
    # nasceva proprio qui, perché il controllo esisteva solo fra top e bottom
    # e gli accessori non lo attraversavano mai.
    gambe_scoperte = _existing_value(current_items, "bottom", "leg_length") == "corta"
    braccia_scoperte = _existing_value(current_items, "top", "sleeve") == "corta"
    if candidate_row["season"] == "inverno" and (gambe_scoperte or braccia_scoperte):
        return False
    if candidate_row["leg_length"] == "corta" or candidate_row["sleeve"] == "corta":
        if any(r["season"] == "inverno" for _, r in current_items):
            return False
    return True


def _calzatura_ok(current_items, slot: str, candidate_row: pd.Series) -> bool:
    """Regola 3 (utente): niente t-shirt, canotte, top o felpe sotto un
    mocassino. Vale nei due ordini di riempimento, perché le scarpe possono
    entrare prima o dopo il top."""
    if slot == "shoes" and candidate_row["mocassino"]:
        if _existing_value(current_items, "top", "top_senza_collo"):
            return False
    elif slot == "top" and candidate_row["top_senza_collo"]:
        if _existing_value(current_items, "shoes", "mocassino"):
            return False
    return True


def _cappotto_ok(current_items, slot: str, candidate_row: pd.Series) -> bool:
    """Regola 4 (utente): niente cappotto o caban sopra una tuta."""
    if slot == "outerwear" and candidate_row["cappotto"]:
        if _existing_value(current_items, "bottom", "bottom_da_tuta"):
            return False
    elif slot == "bottom" and candidate_row["bottom_da_tuta"]:
        if _existing_value(current_items, "outerwear", "cappotto"):
            return False
    return True


def _outerwear_allowed(current_items) -> bool:
    """Regola 2 (utente): giacche/outerwear solo se i pantaloni sono lunghi
    — niente giacca sui pantaloncini. (La parte "niente felpe sui
    pantaloncini" della stessa regola è già coperta dalla Regola 1, perché
    una felpa ha sempre manica lunga per tipo di capo.)"""
    existing_bottom_leg = _existing_value(current_items, "bottom", "leg_length")
    return existing_bottom_leg != "corta"


def _filtra(candidates: pd.DataFrame, predicato, colonna: str = None) -> pd.DataFrame:
    """Filtra riga per riga aggirando un caso limite di pandas.

    Su un DataFrame vuoto .apply() restituisce una Series di dtype object,
    non bool: candidates[serie_non_booleana] non viene letto come maschera di
    righe ma come selezione di COLONNE, quindi il risultato perde tutte le
    colonne e il filtro successivo muore con KeyError sul nome della colonna.
    Con i filtri a catena qui sotto capita spesso che i candidati si esauriscano
    a metà strada, quindi il caso vuoto va intercettato prima.
    """
    if candidates.empty:
        return candidates
    if colonna is None:
        maschera = candidates.apply(predicato, axis=1)
    else:
        maschera = candidates[colonna].apply(predicato)
    return candidates[maschera.astype(bool)]


def candidates_for_slot(df: pd.DataFrame, slot: str, anchor_row: pd.Series, used_relpaths: set,
                         require_real_cluster: bool = False, current_items=None) -> pd.DataFrame:
    """require_real_cluster=True esclude i capi outlier (style_cluster=-1)
    dai candidati, anche se l'ancora stessa è un outlier. Usato per gli slot
    OPZIONALI (outerwear/accessorio): un capo "jolly" per debolezza di
    segnale (vedi style_match) può comunque superare la soglia di score
    puro — meglio quindi richiedere lì un'appartenenza di cluster reale e
    confermata, piuttosto che rischiare di aggiungere un pezzo stonato.
    Per gli slot obbligatori (top/bottom/scarpe) resta permissivo, perché
    il 44% del catalogo è outlier e negarli del tutto renderebbe molti
    outfit non completabili.

    current_items, se passato, applica anche il filtro di dispersione di
    formalità (vedi _formality_ok) e la coerenza stagionale manica/gamba
    (vedi _seasonal_coherence_ok) rispetto ai capi già scelti nell'outfit.
    """
    candidates = df[
        (df["slot"] == slot)
        & (~df["needs_vision_review"].astype(bool))
        & (~df["relpath"].isin(used_relpaths))
    ]
    if require_real_cluster:
        candidates = candidates[candidates["style_cluster"] != -1]
    candidates = _filtra(candidates, lambda g: _gender_compatible(g, anchor_row["gender"]), "gender")
    candidates = _filtra(candidates, lambda s: _season_compatible(s, anchor_row["season"]), "season")
    candidates = _filtra(candidates, lambda c: _cluster_compatible(c, anchor_row["style_cluster"]), "style_cluster")
    if current_items:
        # Genere e stagione vanno verificati contro TUTTI i capi già scelti, non
        # solo contro l'ancora: se l'ancora ha il valore indeterminato (None per
        # il genere, "tutte" per la stagione) è compatibile con chiunque, e due
        # capi incompatibili fra loro possono entrare entrambi passando ciascuno
        # il confronto con lei. Succedeva: 31 outfit mescolavano uomo e donna,
        # 138 mescolavano estate e inverno.
        candidates = _filtra(
            candidates,
            lambda g: all(_gender_compatible(g, r["gender"]) for _, r in current_items), "gender")
        candidates = _filtra(
            candidates,
            lambda s: all(_season_compatible(s, r["season"]) for _, r in current_items), "season")
        candidates = _filtra(candidates, lambda f: _formality_ok(current_items, f), "formality_norm")
        # su TUTTI gli slot, non più solo top/bottom: accessori e capospalla
        # erano l'unica via per cui un capo invernale entrava in un outfit estivo
        candidates = _filtra(candidates, lambda r: _seasonal_coherence_ok(current_items, slot, r))
        candidates = _filtra(candidates, lambda r: _calzatura_ok(current_items, slot, r))
        candidates = _filtra(candidates, lambda r: _cappotto_ok(current_items, slot, r))
    return candidates

In [60]:
# L'imbuto, filtro per filtro. Quanto stringe dipende dall'ancora: un capo del
# cluster piu' popoloso lascia passare quasi tutto, uno elegante quasi niente.
def imbuto(ancora, slot):
    print(f"ancora: {ancora['title']}")
    print(f"  slot={ancora['slot']} genere={ancora['gender']} stagione={ancora['season']} "
          f"cluster={ancora['style_cluster']} formalità={ancora['formality_norm']:.2f} manica={ancora['sleeve']}")
    items = [(ancora["slot"], ancora)]
    passi, c = [], df
    passi.append(("catalogo intero", len(c)))
    c = c[c["slot"] == slot];                                                              passi.append((f"slot = {slot}", len(c)))
    c = c[~c["needs_vision_review"].astype(bool)];                                         passi.append(("non da rivedere", len(c)))
    c = c[c["relpath"] != ancora["relpath"]];                                              passi.append(("non già usato", len(c)))
    c = c[c["gender"].apply(lambda g: _gender_compatible(g, ancora["gender"]))];           passi.append(("genere", len(c)))
    c = c[c["season"].apply(lambda s: _season_compatible(s, ancora["season"]))];           passi.append(("stagione", len(c)))
    c = c[c["style_cluster"].apply(lambda k: _cluster_compatible(k, ancora["style_cluster"]))]; passi.append(("cluster", len(c)))
    c = c[c["formality_norm"].apply(lambda ff: _formality_ok(items, ff))];                 passi.append(("formalità ≤ 0,3", len(c)))
    c = c[c.apply(lambda r: _seasonal_coherence_ok(items, slot, r), axis=1)];              passi.append(("regola 1 stagionale", len(c)))
    for nome, n in passi:
        print(f"  {nome:22s} {n:5d}  {'█' * max(1, round(n / 40))}")
    veri = candidates_for_slot(df, slot, ancora, {ancora["relpath"]}, current_items=items)
    print(f"  → candidates_for_slot: {len(veri)} candidati")
    print()

top_ammessi = df[(df["slot"] == "top") & (~df["needs_vision_review"].astype(bool))].sort_values("relpath")
ancora = top_ammessi.iloc[100]                                   # un capo del registro piu' comune
ancora_formale = top_ammessi.loc[top_ammessi["formality_norm"].idxmax()]   # il top piu' formale

imbuto(ancora, "bottom")
imbuto(ancora_formale, "bottom")

ancora: T-SHIRT UOMO RIP CURL DESTI ANIMALS TEE BLACK
  slot=top genere=uomo stagione=tutte cluster=2 formalità=0.25 manica=corta
  catalogo intero         2901  █████████████████████████████████████████████████████████████████████████
  slot = bottom            689  █████████████████
  non da rivedere          689  █████████████████
  non già usato            689  █████████████████
  genere                   628  ████████████████
  stagione                 628  ████████████████
  cluster                  388  ██████████
  formalità ≤ 0,3          388  ██████████
  regola 1 stagionale      388  ██████████
  → candidates_for_slot: 388 candidati

ancora: CAMICIA UOMO DANIELE ALESSANDRINI OFF WHITE
  slot=top genere=uomo stagione=tutte cluster=0 formalità=0.75 manica=lunga
  catalogo intero         2901  █████████████████████████████████████████████████████████████████████████
  slot = bottom            689  █████████████████
  non da rivedere          689  █████████████████
  non già usa

Il secondo imbuto si chiude a **zero**, e non è un errore: è il modo in cui il catalogo dice
che quel capo non ha compagni. Il registro elegante è sottile — poche decine di capi in tutto —
e i pochi bottom del suo cluster stanno fuori dalla finestra di formalità. Un'ancora così non
produce nessun outfit, e infatti nel pool non compare: la copertura del catalogo, misurata alla
fine di questo notebook, è il conto di quanti capi si trovano in questa condizione.

## 6.3 La ricerca — beam search

### `build_outfits(df, anchor_relpath, w_colore=0.5, w_stile=0.5, beam_width=5, candidates_per_step=30, varianti=1, contatore_uso=None)`

- **Input** — il catalogo preparato e il `relpath` del capo **ancora** (che deve stare in uno
  slot obbligatorio).
- **Processo** — riempie gli slot obbligatori mancanti uno alla volta tenendo aperte le
  `beam_width` combinazioni parziali migliori, non solo la migliore: un bottom perfetto con
  l'ancora può rivelarsi un vicolo cieco quando si arriva alle scarpe. Per velocità i candidati
  si pre-ordinano per punteggio **contro la sola ancora** e se ne valutano a fondo — contro
  tutto il parziale — solo i primi `candidates_per_step`. Il criterio di ogni passo è il
  **minimo** dei punteggi di coppia, lo stesso del punteggio finale: ottimizzare la media
  significherebbe farsi ingannare da un anello debole nascosto dietro tanti buoni.
  Poi, su **tutti** i rami rimasti (non solo il primo), prova gli slot facoltativi: entrano solo
  sopra la loro asticella — 0,46 per un capo di cluster, **0,60** per un outlier — e a parità di
  punteggio vince il capo **meno usato finora**, altrimenti l'ordine del dataframe farebbe
  comparire lo stesso bomber nero in migliaia di outfit.
- **Output** — la lista delle proposte di questa ancora, dalla migliore in giù, ognuna con
  `slots`, `pairwise_scores` (tutte le coppie, per trasparenza) e `outfit_score`.

Intorno a lei due involucri: **`build_outfit(...)`** restituisce solo la migliore, e
**`generate_outfits(df, n_outfits=10, anchor_slot="top", ...)`** scorre le ancore di un solo
slot in ordine deterministico finché non raccoglie `n_outfits` outfit sopra soglia — è la
funzione da usare per guardare qualche risultato senza generare il pool.

In [61]:
# ═══ nuvolari_generation_pipeline/outfit_generation.py ═══════════════════════════════

def build_outfits(df: pd.DataFrame, anchor_relpath: str, w_colore: float = 0.5, w_stile: float = 0.5,
                   beam_width: int = 5, candidates_per_step: int = 30,
                   varianti: int = 1, contatore_uso=None) -> list:
    """Costruisce fino a `varianti` outfit distinti a partire da un capo
    ancora, via beam search slot per slot, ordinati dal migliore. Ritorna
    lista vuota se l'ancora non ha uno slot riconosciuto o se non si riesce
    a riempire tutti gli slot obbligatori.

    Il tetto vero e' beam_width: oltre quello non ci sono rami da consegnare."""
    # Senza contatore in ingresso la rotazione vale solo dentro questa ancora:
    # e' il comportamento giusto per un chiamante singolo (audit_pipeline), non
    # per il pool, che ne passa uno condiviso da tutte le ancore.
    if contatore_uso is None:
        contatore_uso = collections.Counter()

    anchor_rows = df[df["relpath"] == anchor_relpath]
    if anchor_rows.empty:
        return []
    anchor = anchor_rows.iloc[0]
    anchor_slot = anchor["slot"]
    if anchor_slot not in MANDATORY_SLOTS:
        return []

    remaining_mandatory = [s for s in MANDATORY_SLOTS if s != anchor_slot]

    # ogni elemento del beam: (lista di (slot, row), set di relpath già usati)
    beam = [([(anchor_slot, anchor)], {anchor_relpath})]

    def _min_pairwise(items):
        # stesso criterio del punteggio outfit finale (minimo, non media — vedi
        # docstring del modulo): la beam search deve ottimizzare l'anello più
        # debole, non farsi ingannare da una media alta durante la costruzione
        if len(items) < 2:
            return 1.0
        scores = [
            score_pair(items[i][1], items[j][1], w_colore, w_stile)["score"]
            for i in range(len(items)) for j in range(i + 1, len(items))
        ]
        return min(scores)

    for slot in remaining_mandatory:
        new_beam = []
        for items, used in beam:
            candidates = candidates_for_slot(df, slot, anchor, used, current_items=items)
            if candidates.empty:
                continue
            # per velocità, si pre-ordinano i candidati per score contro l'ancora
            # e se ne valutano approfonditamente (contro tutto il parziale) solo i migliori
            candidates = candidates.copy()
            candidates["_anchor_score"] = candidates.apply(
                lambda r: score_pair(anchor, r, w_colore, w_stile)["score"], axis=1
            )
            candidates = candidates.sort_values("_anchor_score", ascending=False).head(candidates_per_step)

            for _, cand in candidates.iterrows():
                new_items = items + [(slot, cand)]
                new_score = _min_pairwise(new_items)
                new_used = used | {cand["relpath"]}
                new_beam.append((new_items, new_used, new_score))

        if not new_beam:
            return []  # nessun candidato disponibile per questo slot -> outfit non completabile
        new_beam.sort(key=lambda t: t[2], reverse=True)
        beam = [(items, used) for items, used, _ in new_beam[:beam_width]]

    # Slot obbligatori riempiti. Gli opzionali si provano su TUTTE le
    # combinazioni rimaste nel beam, non solo sulla prima.
    #
    # Prima si prendeva beam[0] e si buttava il resto: la ricerca era larga 5
    # fino al terzetto e diventava larga 1 esattamente dove restavano ancora
    # due decisioni da prendere. Ma quale terzetto regga meglio un capospalla
    # non si sa finché non lo si prova, e il terzetto in testa spesso non è
    # quello che vince alla fine. Misurato sulla felpa NASA: il terzetto in
    # testa (0,781) finiva a 0,728, il secondo (0,770) sarebbe finito a 0,754
    # — e quel secondo veniva scartato senza mai essere provato.
    def _completa(items, used):
        for slot in OPTIONAL_SLOTS:
            if slot == "outerwear" and not _outerwear_allowed(items):
                continue  # Regola 2: niente giacca/outerwear sui pantaloncini
            # Gli outlier di cluster sono ammessi anche qui (vedi
            # OPTIONAL_OUTLIER_MIN_SCORE): il vincolo di formalità e
            # l'asticella più alta fanno il lavoro che prima faceva il
            # divieto, senza congelare il 70% dei capispalla fuori da ogni
            # outfit.
            candidates = candidates_for_slot(df, slot, anchor, used,
                                              current_items=items)
            if candidates.empty:
                continue
            candidates = candidates.copy()
            candidates["_score"] = candidates.apply(
                lambda r: _min_pairwise(items + [(slot, r)]), axis=1
            )
            # Ognuno con la propria asticella, PRIMA di scegliere il migliore:
            # altrimenti un outlier a 0.65 (bocciato) nasconderebbe un capo di
            # cluster a 0.62 (promosso) solo per avere il punteggio più alto.
            soglie = candidates["style_cluster"].apply(
                lambda c: OPTIONAL_OUTLIER_MIN_SCORE if c == -1 else OPTIONAL_SLOT_MIN_SCORE)
            ammessi = candidates[candidates["_score"] >= soglie]
            if not ammessi.empty:
                # A parita' di punteggio vince il capo MENO usato finora.
                #
                # Qui i pareggi non sono l'eccezione, sono la regola: misurato
                # su 60 outfit, i candidati capospalla ammessi erano 99 su 99 e
                # il margine fra il primo e il secondo aveva mediana 0,0000, con
                # in media sei capi appaiati al massimo (fino a 102). Il nero
                # armonizza col massimo su tutto, quindi mezzo guardaroba di
                # bomber neri casual finisce con lo stesso identico punteggio.
                #
                # `idxmax` rompeva quei pareggi prendendo la prima riga, sempre
                # la stessa: un solo bomber compariva in 4132 outfit e 143
                # capispalla su 274 non entravano in NESSUNO. Non era una scelta
                # di gusto, era l'ordine del dataframe. Ruotando fra i pari
                # merito il punteggio non cambia di un centesimo -- sono
                # identici per definizione -- e il guardaroba entra tutto.
                massimo = ammessi["_score"].max()
                pari = ammessi[ammessi["_score"] >= massimo - TOLLERANZA_PAREGGIO]
                best_candidate = pari.loc[
                    pari["relpath"].map(lambda r: contatore_uso[r]).idxmin()]
                contatore_uso[best_candidate["relpath"]] += 1
                items = items + [(slot, best_candidate)]
                used = used | {best_candidate["relpath"]}
        return items

    def _confeziona(items):
        # punteggi pairwise finali, per trasparenza/debug
        pairwise_scores = {}
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                slot_i, row_i = items[i]
                slot_j, row_j = items[j]
                pairwise_scores[f"{slot_i}-{slot_j}"] = score_pair(row_i, row_j, w_colore, w_stile)["score"]

        outfit_score = min(pairwise_scores.values()) if pairwise_scores else 1.0

        slots_out = {slot: None for slot in MANDATORY_SLOTS + OPTIONAL_SLOTS}
        for slot, row in items:
            slots_out[slot] = {
                "relpath": row["relpath"],
                "title": row["title"],
                "url": row["url"],
                "display_image": row.get("display_image"),
                "all_images": row.get("all_images"),
            }
        return {"slots": slots_out,
                "pairwise_scores": {k: round(v, 3) for k, v in pairwise_scores.items()},
                "outfit_score": round(outfit_score, 3)}

    # Tutto il menu che questa ancora sa produrre, dal migliore in giu'. La
    # beam search completa beam_width rami e prima ne consegnava uno solo: gli
    # altri erano gia' costruiti, valutati e buttati.
    #
    # Qui NON si decide piu' quali tenere. Se due rami sono quasi uguali lo
    # sono rispetto a questa ancora, ma la somiglianza che conta e' quella
    # verso l'INTERO pool: un'altra ancora puo' aver gia' prodotto lo stesso
    # outfit, e da qui non si vede. La scelta sta in run_outfit_pipeline.
    completi = [_completa(items, used) for items, used in beam]
    completi.sort(key=_min_pairwise, reverse=True)

    fuori, visti = [], set()
    for items in completi:
        firma = frozenset(r["relpath"] for _, r in items)
        if firma in visti:
            continue  # due rami possono chiudersi sullo stesso identico insieme
        visti.add(firma)
        fuori.append(_confeziona(items))
        if len(fuori) >= max(1, varianti):
            break
    return fuori


def build_outfit(df: pd.DataFrame, anchor_relpath: str, w_colore: float = 0.5, w_stile: float = 0.5,
                  beam_width: int = 5, candidates_per_step: int = 30) -> dict:
    """Il solo outfit migliore per questa ancora, o None. Involucro su
    build_outfits per i chiamanti che ne vogliono uno (audit_pipeline,
    generate_outfits)."""
    fuori = build_outfits(df, anchor_relpath, w_colore, w_stile,
                          beam_width, candidates_per_step, varianti=1)
    return fuori[0] if fuori else None


def generate_outfits(df: pd.DataFrame, n_outfits: int = 10, anchor_slot: str = "top",
                      min_score: float = 0.46, w_colore: float = 0.5, w_stile: float = 0.5,
                      beam_width: int = 5, candidates_per_step: int = 30) -> list:
    """Genera outfit iterando i capi ancora dello slot indicato in ordine
    deterministico (per relpath — categoria/sottocategoria/slug), non via
    campionamento casuale: così nessun capo del catalogo viene escluso per
    sfortuna nel campionamento, e il risultato è riproducibile a parità di
    dati. Si ferma quando si raggiungono n_outfits outfit sopra min_score,
    o quando si esauriscono le ancore disponibili. Deduplica outfit
    identici (stesso insieme esatto di capi) che possono emergere da
    ancore diverse (es. l'outfit {A,B,C} può essere il migliore risultato
    sia partendo da A come ancora sia da B)."""
    anchor_pool = df[(df["slot"] == anchor_slot) & (~df["needs_vision_review"].astype(bool))]
    anchor_pool = anchor_pool.sort_values("relpath")

    outfits = []
    seen_signatures = set()
    for _, anchor in anchor_pool.iterrows():
        outfit = build_outfit(df, anchor["relpath"], w_colore, w_stile, beam_width, candidates_per_step)
        if not outfit or outfit["outfit_score"] < min_score:
            continue
        signature = frozenset(v["relpath"] for v in outfit["slots"].values() if v)
        if signature in seen_signatures:
            continue
        seen_signatures.add(signature)
        outfits.append(outfit)
        if len(outfits) >= n_outfits:
            break

    outfits.sort(key=lambda o: o["outfit_score"], reverse=True)
    return outfits

In [62]:
t0 = time.time()
proposte = build_outfits(df, ancora["relpath"], varianti=5)
print(f"{len(proposte)} proposte da questa ancora in {time.time()-t0:.1f}s\n")

for i, o in enumerate(proposte, 1):
    capi = " + ".join(v["title"] for v in o["slots"].values() if v)
    print(f"{i}. score {o['outfit_score']:.3f}  {capi}")

5 proposte da questa ancora in 0.3s

1. score 0.880  T-SHIRT UOMO RIP CURL DESTI ANIMALS TEE BLACK + BERMUDA UOMO BLACK RAGS MARRAKECH NERO + SCARPE HEYDUDE WALLY WELL WORN A BROWN/MULTI + OCCHIALI PROJ3CT ECTOR CHAMPAGNE/BLU SF
2. score 0.880  T-SHIRT UOMO RIP CURL DESTI ANIMALS TEE BLACK + BERMUDA UOMO BLACK RAGS MARRAKECH NERO + SCARPE HEYDUDE WALLY COMF B OLIVE/B TAN + OCCHIALI PROJ3CT ECTOR CHAMPAGNE/BLU SF
3. score 0.878  T-SHIRT UOMO RIP CURL DESTI ANIMALS TEE BLACK + BERMUDA UOMO BLACK RAGS MARRAKECH NERO + SCARPE HEYDUDE WALLY BRAIDED B OLIVE/D BROWN + OCCHIALI PROJ3CT ECTOR CHAMPAGNE/BLU SF
4. score 0.876  T-SHIRT UOMO RIP CURL DESTI ANIMALS TEE BLACK + BERMUDA UOMO NUVOLARI LI BRANCH PERLAGE + SCARPE HEYDUDE WALLY WELL WORN A BROWN/MULTI + OCCHIALI PROJ3CT ECTOR CHAMPAGNE/BLU SF
5. score 0.872  T-SHIRT UOMO RIP CURL DESTI ANIMALS TEE BLACK + BERMUDA UOMO BLACK RAGS MARRAKECH NERO + SCARPE HEYDUDE WALLY BRAIDED F GREY/O GREY + OCCHIALI PROJ3CT GAGA TARTA/BLU


In [63]:
# Il migliore, aperto: da dove viene il suo punteggio.
migliore = proposte[0]
print("slot riempiti")
for slot, v in migliore["slots"].items():
    print(f"   {slot:10s} {v['title'] if v else '—'}")
print("\npunteggi di coppia")
for coppia, s in sorted(migliore["pairwise_scores"].items(), key=lambda kv: kv[1]):
    print(f"   {coppia:22s} {s:.3f}")
print(f"\noutfit_score = minimo = {migliore['outfit_score']:.3f}")

slot riempiti
   top        T-SHIRT UOMO RIP CURL DESTI ANIMALS TEE BLACK
   bottom     BERMUDA UOMO BLACK RAGS MARRAKECH NERO
   shoes      SCARPE HEYDUDE WALLY WELL WORN A BROWN/MULTI
   outerwear  —
   accessory  OCCHIALI PROJ3CT ECTOR CHAMPAGNE/BLU SF

punteggi di coppia
   top-accessory          0.880
   shoes-accessory        0.886
   bottom-accessory       0.898
   bottom-shoes           0.942
   top-shoes              0.943
   top-bottom             0.949

outfit_score = minimo = 0.880


## 6.4 Il giudizio: punteggio, soglie, validazione

Il **punteggio dell'outfit è il minimo** dei punteggi di coppia, non la media: una media può
nascondere un singolo abbinamento pessimo dietro tanti buoni, e il punteggio deve riflettere
l'anello più debole della catena.

| soglia | valore | dove |
|---|---|---|
| accettazione nel pool | **0,46** | `run_outfit_pipeline(min_score=...)` |
| slot facoltativo, capo di cluster | **0,46** | `OPTIONAL_SLOT_MIN_SCORE` |
| slot facoltativo, capo outlier | **0,60** | `OPTIONAL_OUTLIER_MIN_SCORE` |
| dispersione di formalità | **≤ 0,30** | `FORMALITY_SPREAD_MAX` |

### `valida_composizione(o, df)`

- **Input** — un outfit **già composto** (anche uno vecchio, letto dal pool) e il catalogo
  indicizzato per `relpath`.
- **Processo** — riapplica a posteriori tutto quello che `candidates_for_slot` applica durante
  la costruzione: capi ancora esistenti, nessuno da rivedere, punteggio minimo sopra soglia,
  genere unico, stagioni compatibili, dispersione di formalità, e le quattro regole.
  Esiste perché la beam search non può più rivedere un outfit già chiuso, e il pool deve poter
  dire se una proposta di ieri **regge ancora le regole di oggi**.
- **Output** — `(punteggio, None)` se passa, `(punteggio, motivo)` se no.

In [64]:
# ═══ nuvolari_generation_pipeline/outfit_generation.py ═══════════════════════════════

def valida_composizione(o: dict, df) -> tuple:
    """(score, None) se la composizione passa le regole attuali, altrimenti
    (score, motivo). Applica i vincoli di candidates_for_slot a un outfit
    GIA' composto, che la beam search non puo' piu' rivedere.

    Sta qui e non in ripesca_orfane perche' servono a due chiamanti: il
    ripescaggio delle immagini pagate e l'accumulo del pool, che deve poter
    dire se un outfit generato ieri regge ancora le regole di oggi."""
    items = []
    for s, v in o["slots"].items():
        if not v:
            continue
        if v["relpath"] not in df.index:
            return None, "capo sparito dal catalogo"
        items.append((s, df.loc[v["relpath"]]))
    rows = dict(items)

    if any(r["needs_vision_review"] for _, r in items):
        return None, "needs_vision_review"

    coppie = {}
    for (sa, a), (sb, b) in itertools.combinations(items, 2):
        coppie[f"{sa}-{sb}"] = score_pair(a, b)["score"]
    m = min(coppie.values())
    if m < OPTIONAL_SLOT_MIN_SCORE:
        return m, f"score sotto {OPTIONAL_SLOT_MIN_SCORE}"

    generi = {r["gender"] for _, r in items} - {None}
    if len(generi) > 1:
        return m, "genere misto"
    for (_, a), (_, b) in itertools.combinations(items, 2):
        if not (a["season"] == "tutte" or b["season"] == "tutte" or a["season"] == b["season"]):
            return m, f"stagioni {a['season']}/{b['season']}"
    fs = [r["formality_norm"] for _, r in items]
    if max(fs) - min(fs) > FORMALITY_SPREAD_MAX:
        return m, "dispersione formalita'"

    top, bot, ow, sh = (rows.get(k) for k in ("top", "bottom", "outerwear", "shoes"))
    if bot is not None and bot["leg_length"] == "corta":
        if top is not None and top["sleeve"] != "corta":
            return m, "R1 maniche lunghe su shorts"
        if ow is not None:
            return m, "R2 giacca su shorts"
    if any(r["season"] == "inverno" for _, r in items) and (
            (bot is not None and bot["leg_length"] == "corta")
            or (top is not None and top["sleeve"] == "corta")):
        return m, "R1 capo invernale su pelle scoperta"
    if sh is not None and sh["mocassino"] and top is not None and top["top_senza_collo"]:
        return m, "R3 mocassino su top senza collo"
    if ow is not None and ow["cappotto"] and bot is not None and bot["bottom_da_tuta"]:
        return m, "R4 cappotto su tuta"
    return m, None

In [65]:
df_idx = df.set_index("relpath", drop=False)

# Gli outfit veri del pool, ricontrollati con le regole di adesso.
esiti = collections.Counter()
with open(POOL, encoding="utf-8") as f:
    for i, riga in enumerate(f):
        if i >= 300:
            break
        punteggio, motivo = valida_composizione(json.loads(riga), df_idx)
        esiti[motivo or "valido"] += 1
print("primi 300 outfit del pool:", dict(esiti))

primi 300 outfit del pool: {'valido': 300}


In [66]:
# Una composizione costruita apposta per violare la Regola 1: bermuda + capo a manica lunga.
bermuda = df[(df["slot"] == "bottom") & (df["leg_length"] == "corta") & (~df["needs_vision_review"].astype(bool))]
maniche_lunghe = df[(df["slot"] == "top") & (df["sleeve"] == "lunga") & (~df["needs_vision_review"].astype(bool))]
scarpe = df[(df["slot"] == "shoes") & (~df["needs_vision_review"].astype(bool))]

trovato = None
for _, bo in bermuda.head(40).iterrows():
    for _, to in maniche_lunghe.head(40).iterrows():
        for _, sh in scarpe.head(20).iterrows():
            finto = {"slots": {"top": {"relpath": to["relpath"]},
                               "bottom": {"relpath": bo["relpath"]},
                               "shoes": {"relpath": sh["relpath"]},
                               "outerwear": None, "accessory": None}}
            punteggio, motivo = valida_composizione(finto, df_idx)
            if motivo and motivo.startswith("R1"):
                trovato = (to, bo, sh, punteggio, motivo)
                break
        if trovato: break
    if trovato: break

to, bo, sh, punteggio, motivo = trovato
print("top   :", to["title"], f"(manica {to['sleeve']})")
print("bottom:", bo["title"], f"(gamba {bo['leg_length']})")
print("scarpe:", sh["title"])
print(f"\npunteggio minimo di coppia: {punteggio:.3f}  →  RIFIUTATO: {motivo}")

top   : CAMICIA UOMO LEE CAMP SHIRT SOY BEAN (manica lunga)
bottom: SHORT PROPAGANDA PRBR738 BLUEBERRY SWIMTRUNK V.U. (gamba corta)
scarpe: CIABATTE CROCS ECHO CLOG BLACK

punteggio minimo di coppia: 0.822  →  RIFIUTATO: R1 maniche lunghe su shorts


---
# 7. Dal singolo outfit al pool

La fase 6 non produce un outfit: produce un **catalogo di proposte**. Servono altre tre cose
oltre alla ricerca — decidere da dove partire, evitare i quasi-doppioni, e dare identità stabili.

### `compatibilita_media(df, relpaths, campione=200, seme=12345)`

- **Input** — il catalogo e i `relpath` di cui misurare la versatilità.
- **Processo** — per ogni capo, la media di `score_pair` contro **200 capi estratti a caso**
  con seme fisso. Serve a decidere l'**ordine delle ancore**, che non è un dettaglio: chi arriva
  prima trova lo spazio libero, chi arriva dopo trova l'indice di novità già pieno. Prima
  l'ordine era alfabetico, cioè lo decideva il nome della cartella.
- **Output** — `{relpath: media}`.

### `IndiceNovita(distanza_minima=2)` — `e_nuovo(firma)` / `aggiungi(firma)`

- **Input** — la **firma** di un outfit, cioè l'insieme dei `relpath` dei suoi capi.
- **Processo** — dice se differisce da ogni outfit già accettato per almeno 2 capi, presi nel
  verso peggiore (così un outfit contenuto in un altro non passa per nuovo). Il confronto
  ingenuo è quadratico — a 7000 outfit sono 49 milioni di confronti — quindi tiene un indice
  inverso capo → outfit e guarda solo quelli che condividono almeno un capo.
- **Output** — booleano; `aggiungi` registra la firma.

### `build_garment_id(relpath)` e `build_outfit_id(relpaths)`

- **Input** — un percorso, o l'insieme dei percorsi di un outfit.
- **Processo** — SHA-256 troncato a 12 caratteri; per l'outfit la firma è **ordinata**, così non
  dipende dall'ordine degli slot.
- **Output** — un id stabile: lo stesso outfit rigenerato mesi dopo mantiene lo stesso id — ed
  è per questo che la sua foto, che costa, non va rifatta.

In [67]:
# ═══ nuvolari_generation_pipeline/outfit_generation.py ═══════════════════════════════

# =====================================================================
# Pipeline completa: pool di outfit per API/UI a valle
# =====================================================================

def build_garment_id(relpath: str) -> str:
    """Identificativo stabile e "pulito" per un capo, derivato da relpath
    via hash (non il path stesso) — pensato per un'API che non deve far
    trapelare la struttura di cartelle interna (es. /products/{id})."""
    return hashlib.sha256(relpath.encode("utf-8")).hexdigest()[:12]


def build_outfit_id(relpaths) -> str:
    """Identificativo stabile di un outfit, derivato dall'insieme (ordinato,
    per essere indipendente dall'ordine) dei relpath dei capi che lo
    compongono. Poiché la generazione è ora deterministica (vedi
    generate_outfits/run_outfit_pipeline), lo stesso outfit rigenerato in
    run successive ottiene sempre lo stesso id — utile per un'API/UI che
    debba referenziarlo nel tempo (es. outfit salvati/preferiti)."""
    signature = "|".join(sorted(relpaths))
    return hashlib.sha256(signature.encode("utf-8")).hexdigest()[:12]


def _serialize_slots(slots: dict) -> dict:
    out = {}
    for slot, item in slots.items():
        if item is None:
            out[slot] = None
            continue
        out[slot] = {
            "product_id": build_garment_id(item["relpath"]),
            "relpath": item["relpath"],
            "title": item["title"],
            "url": item["url"],
            "display_image": item.get("display_image"),
            # all_images arriva dal parquet come ndarray: "or []" su ndarray
            # solleva ValueError (verità ambigua), quindi il None si gestisce
            # esplicitamente prima di convertire in lista
            "all_images": [] if item.get("all_images") is None else list(item["all_images"]),
        }
    return out


_GENDER_LABEL = {"uomo": "uomo", "donna": "donna"}


def _build_outfit_label(gender, slots: dict) -> str:
    """Riga leggibile a colpo d'occhio, es. 'Outfit uomo: Camicia Nuvolari
    Petapa Sottobosco + Bermuda Nuvolari Zoe Verde Salvia + Sneakers
    Lacoste Elite Active' — pensata per scorrere il file JSONL senza dover
    aprire ogni oggetto slots per capire di che outfit si tratta."""
    gender_label = _GENDER_LABEL.get(gender, "unisex")
    parts = [item["title"] for item in slots.values() if item]
    return f"Outfit {gender_label}: " + " + ".join(parts)


def compatibilita_media(df: pd.DataFrame, relpaths, campione: int = 200,
                        w_colore: float = 0.5, w_stile: float = 0.5,
                        seme: int = 12345) -> dict:
    """Quanto ogni capo va d'accordo col resto del campionario, in media.

    Serve a decidere da quali capi PARTIRE. L'ordine delle ancore non e' un
    dettaglio: chi arriva prima trova lo spazio libero e piazza la sua
    composizione migliore, chi arriva dopo trova l'indice di novita' gia'
    pieno e viene scartato. Finora l'ordine era alfabetico per relpath --
    deterministico ma arbitrario, cioe' il posto in cima al pool lo decideva
    il nome della cartella.

    La misura e' su un campione casuale con seme fisso e non su tutto il
    catalogo: 2322 ancore per 2901 capi sarebbero quasi sette milioni di
    confronti per un dato che serve solo a mettere in fila. Con 200 estrazioni
    l'errore sulla media e' di qualche millesimo, molto meno delle distanze
    che separano un capo versatile da uno difficile.
    """
    rng = random.Random(seme)
    altri = df[df["slot"].isin(MANDATORY_SLOTS + OPTIONAL_SLOTS)]
    altri = altri[~altri["needs_vision_review"].astype(bool)]
    righe = [altri.iloc[i] for i in range(len(altri))]
    fuori = {}
    for rel in relpaths:
        r = df.loc[rel] if rel in df.index else None
        if r is None:
            fuori[rel] = 0.0
            continue
        punteggi = []
        for _ in range(campione):
            b = righe[rng.randrange(len(righe))]
            if b["relpath"] == rel:
                continue
            punteggi.append(score_pair(r, b, w_colore, w_stile)["score"])
        fuori[rel] = sum(punteggi) / len(punteggi) if punteggi else 0.0
    return fuori

In [68]:
campione = list(df[df["slot"] == "shoes"]["relpath"].head(6))
medie = compatibilita_media(df_idx, campione, campione=60)
for rel, m in sorted(medie.items(), key=lambda kv: -kv[1]):
    print(f"  {m:.3f}  {df_idx.loc[rel, 'title'][:60]}")

  0.775  SANDALO DONNA UGG GOLDENGLOW TOGGLE BLACK
  0.733  MOCASSINI UOMO TIMBERLAND CLASSIC BOAT BOAT BROWN
  0.732  CIABATTE DONNA UGG TAZZ II SAND
  0.642  SANDALO DONNA UGG GOLDENGLOW TOGGLE MUSTARD SEED
  0.580  SABOT DONNA COLOUR OF CALIFORNIA FRINGES TAN
  0.559  CIABATTE DONNA COLOURS OF CALIFORNIA TWO STRAP RAFFIA BLACK-


In [69]:
indice = IndiceNovita()          # distanza minima = 2 capi
A = frozenset({"a", "b", "c", "d"})
B = frozenset({"a", "b", "c", "e"})          # un capo diverso
C = frozenset({"a", "b", "c", "d", "e"})     # A più un accessorio
D = frozenset({"a", "b", "x", "y"})          # due capi diversi

indice.aggiungi(A)
for nome, f in [("B (1 capo diverso)", B), ("C (A + un accessorio)", C), ("D (2 capi diversi)", D)]:
    print(f"  {nome:26s} nuovo? {indice.e_nuovo(f)}")

print("\nid stabile dello stesso insieme, in ordini diversi:")
print("  ", build_outfit_id(["a/b", "c/d", "e/f"]))
print("  ", build_outfit_id(["e/f", "a/b", "c/d"]))

  B (1 capo diverso)         nuovo? False
  C (A + un accessorio)      nuovo? False
  D (2 capi diversi)         nuovo? True

id stabile dello stesso insieme, in ordini diversi:
   638470893986
   638470893986


### `run_outfit_pipeline(df, out_jsonl, min_score=0.46, beam_width=15, candidates_per_step=30, varianti_per_ancora=3, accumula=True)`

- **Input** — il catalogo preparato e il percorso del pool da scrivere.
- **Processo** — quattro momenti:
  1. **eredita** il pool esistente, se `accumula`: ogni outfit già scritto viene ricontrollato con
     `valida_composizione` e **riscritto con i punteggi ricalcolati** — chi non regge più le regole
     di oggi cade, gli altri restano (le loro foto sono già pagate);
  2. per ogni slot obbligatorio scorre le ancore **dalle più versatili in giù**
     (`compatibilita_media`) e chiede a `build_outfits` tutto il menu che sanno produrre;
  3. scrive in **passate**: prima la proposta migliore di ogni ancora, poi la seconda, e ogni
     volta solo se `IndiceNovita` la giudica lontana da tutto il resto — così i primi migliaia di
     outfit coprono il catalogo il più possibile;
  4. quello che le passate scartano non è perso: finisce in coda, quando lo spazio distinto è finito.
- **Output** — `outfits_pool.jsonl`, un outfit per riga, con `outfit_id`, `label` leggibile,
  `gender`, `outfit_score`, l'ancora di partenza, gli `slots` serializzati (con `product_id`,
  titolo, url, foto) e tutti i `pairwise_scores`. È il file che la UI e il database consumano.

**Non viene eseguita qui**: è il run lungo (ore, tutto il catalogo come ancora). Il suo
risultato è già sul disco, e qui sotto lo si legge.

In [70]:
# ═══ nuvolari_generation_pipeline/outfit_generation.py ═══════════════════════════════

def run_outfit_pipeline(df: pd.DataFrame, out_jsonl: str, min_score: float = 0.46,
                         w_colore: float = 0.5, w_stile: float = 0.5,
                         beam_width: int = 15, candidates_per_step: int = 30,
                         varianti_per_ancora: int = 3, accumula: bool = True) -> dict:
    """Genera l'intero pool di outfit scansionando OGNI slot obbligatorio
    (top, bottom, scarpe) come ancora a turno — non solo "top" come fa
    generate_outfits di default — in ordine deterministico per relpath.
    Così ogni capo di ogni slot obbligatorio ha la sua possibilità di
    comparire in almeno un outfit, non solo i top (vedi discussione:
    ancorare solo dai top garantisce copertura ai top, non a bottom/scarpe,
    che potrebbero perdere sistematicamente come candidati senza mai
    essere loro stessi ancora).

    Deduplica GLOBALMENTE (stesso insieme esatto di capi trovato da ancore
    diverse — anche di slot diversi — conta una volta sola).

    Scrive incrementalmente su out_jsonl (un outfit per riga, JSONL) così
    il run lungo è resumibile/ispezionabile a metà. Formato di ogni riga
    pensato per essere consumato direttamente da un'API a valle: ogni capo
    ha un product_id stabile (hash di relpath, non il path stesso) e ogni
    outfit un outfit_id stabile (vedi build_garment_id/build_outfit_id) —
    lo stesso outfit rigenerato in run successive mantiene lo stesso id.
    """
    total_anchors_tried = 0
    total_outfits_written = 0

    # Accumulo: un outfit gia' nel pool che regge ancora le regole non viene
    # buttato solo perche' la beam search di oggi ha preferito altro. Il pool
    # e' un catalogo di proposte valide, non la classifica di un singolo run:
    # ogni rigenerazione da zero costava outfit legittimi (e le loro immagini
    # gia' pagate, che finivano orfane). I superstiti si riscrivono con i
    # punteggi RICALCOLATI, non con quelli vecchi, altrimenti il pool
    # mescolerebbe due tarature diverse.
    ereditati = []
    per_passata = []
    scartati = 0
    if accumula and os.path.exists(out_jsonl):
        df_idx = df.set_index("relpath", drop=False)
        for riga in open(out_jsonl, encoding="utf-8"):
            riga = riga.strip()
            if not riga:
                continue
            o = json.loads(riga)
            punteggio, motivo = valida_composizione(o, df_idx)
            if motivo or punteggio is None or punteggio < min_score:
                scartati += 1
                continue
            items = [(s, df_idx.loc[v["relpath"]]) for s, v in o["slots"].items() if v]
            coppie = {f"{sa}-{sb}": round(score_pair(a, b, w_colore, w_stile)["score"], 3)
                      for (sa, a), (sb, b) in itertools.combinations(items, 2)}
            o["pairwise_scores"] = coppie
            o["outfit_score"] = round(min(coppie.values()), 3)
            ereditati.append(o)
        print(f"[*] Pool esistente: {len(ereditati)} outfit confermati, "
              f"{scartati} non passano piu' le regole", flush=True)

    # Gli ereditati fanno gia' parte del pool: i nuovi devono essere diversi
    # anche da loro, non solo fra di se'.
    indice = IndiceNovita()
    for o in ereditati:
        indice.aggiungi(frozenset(v["relpath"] for v in o["slots"].values() if v))

    with open(out_jsonl, "w", encoding="utf-8") as f:
        for o in ereditati:
            f.write(json.dumps(o, ensure_ascii=False) + "\n")
            total_outfits_written += 1
        f.flush()

        # Un solo contatore per tutta la generazione: e' cosi' che la rotazione
        # fra pari merito distribuisce capispalla e accessori sull'intero
        # guardaroba invece che dentro una singola ancora. Parte da quello che
        # il pool ereditato usa gia', altrimenti i capi consumati dai run
        # precedenti ripartirebbero da zero e vincerebbero di nuovo.
        contatore_uso = collections.Counter(
            v["relpath"] for o in ereditati for s, v in o["slots"].items()
            if v and s in OPTIONAL_SLOTS)

        # Menu completo di ogni ancora, calcolato una volta sola: la beam
        # search e' la parte cara, l'ammissione e' aritmetica su insiemi.
        menu = []
        for anchor_slot in MANDATORY_SLOTS:
            anchor_pool = df[(df["slot"] == anchor_slot) & (~df["needs_vision_review"].astype(bool))]
            # Dai capi piu' compatibili in giu': partire da un capo che va
            # d'accordo con tutto significa che la prima passata riempie il
            # pool con le composizioni piu' solide, e i capi difficili trovano
            # comunque posto piu' avanti. Il relpath resta come spareggio, cosi'
            # l'ordine e' riproducibile.
            medie = compatibilita_media(df.set_index("relpath", drop=False),
                                        list(anchor_pool["relpath"]),
                                        w_colore=w_colore, w_stile=w_stile)
            anchor_pool = anchor_pool.assign(_compat=anchor_pool["relpath"].map(medie))
            anchor_pool = anchor_pool.sort_values(["_compat", "relpath"],
                                                  ascending=[False, True])
            n = len(anchor_pool)
            print(f"[*] Slot ancora '{anchor_slot}': {n} candidati "
                  f"(compatibilita' media da {anchor_pool['_compat'].iloc[0]:.3f} "
                  f"a {anchor_pool['_compat'].iloc[-1]:.3f})...", flush=True)

            for i, (_, anchor) in enumerate(anchor_pool.iterrows(), 1):
                total_anchors_tried += 1
                proposte = build_outfits(df, anchor["relpath"], w_colore, w_stile,
                                          beam_width, candidates_per_step,
                                          varianti=beam_width,
                                          contatore_uso=contatore_uso)
                proposte = [o for o in proposte if o["outfit_score"] >= min_score]
                if proposte:
                    menu.append((anchor_slot, anchor, proposte))
                if i % 200 == 0 or i == n:
                    print(f"    [{anchor_slot} {i}/{n}] ancore con proposte: {len(menu)}", flush=True)

        # ORDINE, non selezione. Nel pool finisce tutto quello che e' valido,
        # quasi-doppioni compresi: quello che l'indice decide e' soltanto QUANDO.
        #
        # Le passate mettono per primo l'outfit migliore di ogni ancora, poi il
        # secondo, e ogni volta solo se e' lontano da tutto il resto: cosi' i
        # primi migliaia di outfit coprono il catalogo il piu' possibile. Quello
        # che le passate scartano non e' perso -- resta nel menu e viene scritto
        # in coda, quando lo spazio distinto e' finito.
        firme_esatte = {frozenset(v["relpath"] for v in o["slots"].values() if v)
                        for o in ereditati}

        def _scrivi(anchor_slot, anchor, outfit):
            relpaths = [v["relpath"] for v in outfit["slots"].values() if v]
            record = {
                "outfit_id": build_outfit_id(relpaths),
                "label": _build_outfit_label(anchor.get("gender"), outfit["slots"]),
                "gender": anchor.get("gender"),
                "outfit_score": outfit["outfit_score"],
                "anchor_slot": anchor_slot,
                "anchor_relpath": anchor["relpath"],
                "slots": _serialize_slots(outfit["slots"]),
                "pairwise_scores": outfit["pairwise_scores"],
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

        presi = [set() for _ in menu]
        for passata in range(1, max(1, varianti_per_ancora) + 1):
            ammessi_passata = 0
            for m, (anchor_slot, anchor, proposte) in enumerate(menu):
                for n_prop, outfit in enumerate(proposte):
                    if n_prop in presi[m]:
                        continue
                    firma = frozenset(v["relpath"] for v in outfit["slots"].values() if v)
                    if firma in firme_esatte:
                        presi[m].add(n_prop)  # gia' nel pool: mai piu' da guardare
                        continue
                    if not indice.e_nuovo(firma):
                        continue  # troppo simile: lo riprende la coda
                    indice.aggiungi(firma)
                    firme_esatte.add(firma)
                    presi[m].add(n_prop)
                    _scrivi(anchor_slot, anchor, outfit)
                    total_outfits_written += 1
                    ammessi_passata += 1
                    break
            f.flush()
            per_passata.append(ammessi_passata)
            print(f"[*] Passata {passata}: {ammessi_passata} outfit distinti "
                  f"(pool a {total_outfits_written})", flush=True)
            if not ammessi_passata:
                break  # nessuno spazio distinto rimasto: le passate dopo sono vuote

        # Coda: tutto il resto del menu. Qui l'unico filtro e' l'identita' --
        # due volte lo stesso identico insieme di capi sarebbe lo stesso
        # outfit_id, non una proposta in piu'.
        in_coda = 0
        for m, (anchor_slot, anchor, proposte) in enumerate(menu):
            for n_prop, outfit in enumerate(proposte):
                if n_prop in presi[m]:
                    continue
                firma = frozenset(v["relpath"] for v in outfit["slots"].values() if v)
                if firma in firme_esatte:
                    continue
                firme_esatte.add(firma)
                _scrivi(anchor_slot, anchor, outfit)
                total_outfits_written += 1
                in_coda += 1
        f.flush()
        print(f"[*] Coda (quasi-doppioni validi): {in_coda} outfit "
              f"(pool a {total_outfits_written})", flush=True)


    stats = {
        "ancore_processate": total_anchors_tried,
        "outfit_ereditati": len(ereditati),
        "outfit_scartati_dal_pool_vecchio": scartati,
        "outfit_nuovi": total_outfits_written - len(ereditati),
        "nuovi_per_passata": per_passata,
        "nuovi_in_coda": in_coda,
        "outfit_unici_scritti": total_outfits_written,
        "output": out_jsonl,
    }
    print(f"\n[OK] Pipeline completata: {stats}", flush=True)
    return stats

In [71]:
if ESEGUI_POOL:
    # Ore di lavoro: ogni capo di top/bottom/scarpe fa da ancora, a turno.
    # accumula=True tiene gli outfit già nel pool che reggono ancora le regole.
    esito = run_outfit_pipeline(df, out_jsonl=str(POOL), accumula=True)
    print(esito)
else:
    print("ESEGUI_POOL = False: si legge il pool già generato.")
    print("   (run_outfit_pipeline scandisce 2322 ancore: ore di CPU, nessun costo)")

ESEGUI_POOL = False: si legge il pool già generato.
   (run_outfit_pipeline scandisce 2322 ancore: ore di CPU, nessun costo)


In [72]:
t0 = time.time()
righe, usati = [], set()
with open(POOL, encoding="utf-8") as f:
    for r in f:
        o = json.loads(r)
        righe.append({"score": o["outfit_score"], "ancora": o["anchor_slot"], "genere": o.get("gender"),
                      "n_capi": sum(1 for v in o["slots"].values() if v),
                      "capospalla": o["slots"].get("outerwear") is not None,
                      "accessorio": o["slots"].get("accessory") is not None})
        usati.update(v["relpath"] for v in o["slots"].values() if v)
pool = pd.DataFrame(righe)
print(f"{len(pool)} outfit nel pool, letti in {time.time()-t0:.1f}s\n")
print(pool["score"].describe().round(3).to_string())
print("\ncapi per outfit:"); print(pool["n_capi"].value_counts().sort_index().to_string())
print("\nslot dell'ancora:"); print(pool["ancora"].value_counts().to_string())
print("\ngenere:"); print(pool["genere"].value_counts(dropna=False).to_string())

67564 outfit nel pool, letti in 0.9s

count    67564.000
mean         0.891
std          0.068
min          0.463
25%          0.878
50%          0.916
75%          0.935
max          0.950

capi per outfit:
n_capi
3      513
4     8305
5    58746

slot dell'ancora:
ancora
top       43341
bottom    18724
shoes      5499

genere:
genere
uomo     53146
None      8195
donna     6223


In [73]:
# Copertura: quanti capi abbinabili sono finiti in almeno un outfit, e quanti no.
eleggibili = df[(df["slot"].isin(MANDATORY_SLOTS + OPTIONAL_SLOTS))
                & (~df["needs_vision_review"].astype(bool))].copy()
eleggibili["nel_pool"] = eleggibili["relpath"].isin(usati)

print(f"capi eleggibili: {len(eleggibili)} | in almeno un outfit: {eleggibili['nel_pool'].sum()} "
      f"({eleggibili['nel_pool'].mean()*100:.1f}%)\n")
print("per slot:")
print((eleggibili.groupby("slot")["nel_pool"].agg(["sum", "count"])
       .assign(percentuale=lambda d: (d["sum"] / d["count"] * 100).round(1))).to_string())
print("\nper cluster stilistico:")
print((eleggibili.groupby("style_cluster")["nel_pool"].agg(["sum", "count"])
       .assign(percentuale=lambda d: (d["sum"] / d["count"] * 100).round(1))).to_string())

capi eleggibili: 2778 | in almeno un outfit: 2711 (97.6%)

per slot:
            sum  count  percentuale
slot                               
accessory   167    182         91.8
bottom      689    689        100.0
outerwear   226    274         82.5
shoes       236    236        100.0
top        1393   1397         99.7

per cluster stilistico:
                sum  count  percentuale
style_cluster                          
-1               23     23        100.0
 0                2     35          5.7
 1              865    870         99.4
 2             1587   1615         98.3
 3              104    104        100.0
 4               47     47        100.0
 5               10     10        100.0
 6               73     74         98.6


In [74]:
# Un outfit del pool com'è scritto sul disco, campi lunghi accorciati.
with open(POOL, encoding="utf-8") as f:
    esempio_pool = json.loads(f.readline())

for k, v in esempio_pool.items():
    if k == "slots":
        print("slots:")
        for slot, item in v.items():
            print(f"   {slot:10s}", "—" if item is None else
                  f"{item['product_id']}  {item['title'][:52]}")
    else:
        print(f"{k}: {v}")

outfit_id: f348e6a87fe2
label: Outfit donna: CANOTTA DONNA LEE KNITTED TANK WHITE + SHORT PROPAGANDA PRBR738 BLUEBERRY SWIMTRUNK V.U. + W GIZEH EXQUISITE PLATF BLK LEATHR-BLK LEATH + BORSA DONNA GAELLE PARIS PAGLIA
gender: donna
outfit_score: 0.884
anchor_slot: top
anchor_relpath: abbigliamento-donna/canotta-donna-lee-knitted-tank-white
slots:
   top        d61b1dc5e2ca  CANOTTA DONNA LEE KNITTED TANK WHITE
   bottom     a0cb30daf13f  SHORT PROPAGANDA PRBR738 BLUEBERRY SWIMTRUNK V.U.
   shoes      f1c34a3b4851  W GIZEH EXQUISITE PLATF BLK LEATHR-BLK LEATH
   outerwear  —
   accessory  f70f5c09b26b  BORSA DONNA GAELLE PARIS PAGLIA
pairwise_scores: {'top-bottom': 0.884, 'top-shoes': 0.945, 'top-accessory': 0.916, 'bottom-shoes': 0.904, 'bottom-accessory': 0.889, 'shoes-accessory': 0.95}


---
# 7.1 Il pool ordinato per capo

Il pool (`outfits_pool.jsonl`) è ordinato per outfit: una riga per outfit, ognuna con i suoi
slot. Per rispondere alla domanda opposta — *dato questo capo, in quali outfit compare?* —
serve girare l'indice: chiave = capo, valore = elenco degli outfit che lo contengono.

- **Input** — `outfits_pool.jsonl`, il pool già su disco (nessuna lettura da rete, nessun costo).
- **Processo** — una riga sola per outfit: si guarda ogni slot occupato e si accoda quell'outfit
  alla lista del capo in quello slot, con gli **altri** capi dello stesso outfit già pronti a
  fianco (non serve un secondo giro per trovarli, come nella query inversa lato server).
- **Output** — un dizionario `capo_a_outfit`, chiave = `relpath` del capo (l'identificativo
  stabile del catalogo, non l'hash dell'outfit); resta in memoria, e con
  `SCRIVI_SU_DISCO = True` va anche su `nuvolari_db/capo_a_outfit.json`.


In [75]:
SCRIVI_SU_DISCO = False   # True = scrive anche nuvolari_db/capo_a_outfit.json

capo_a_outfit = {}
with open(POOL, encoding="utf-8") as f:
    for riga in f:
        o = json.loads(riga)
        slot_occupati = {slot: capo for slot, capo in o["slots"].items() if capo}
        for slot, capo in slot_occupati.items():
            voce = capo_a_outfit.setdefault(capo["relpath"], {"title": capo["title"], "outfit": []})
            voce["outfit"].append({
                "outfit_id": o["outfit_id"],
                "gender": o["gender"],
                "outfit_score": o["outfit_score"],
                "slot": slot,
                "altri_capi": [{"slot": s, "relpath": c["relpath"], "title": c["title"]}
                               for s, c in slot_occupati.items() if s != slot],
            })

# capi in ordine alfabetico, outfit di ognuno in ordine di id — stessa convenzione
# di server/genera_outfit_per_capo.js
capo_a_outfit = {relpath: {**capo_a_outfit[relpath],
                            "numero_outfit": len(capo_a_outfit[relpath]["outfit"]),
                            "outfit": sorted(capo_a_outfit[relpath]["outfit"], key=lambda o: o["outfit_id"])}
                 for relpath in sorted(capo_a_outfit)}

print(f"capi indicizzati: {len(capo_a_outfit)}")
un_capo = next(iter(capo_a_outfit))
print(f"esempio — {un_capo}:")
print(f"   title: {capo_a_outfit[un_capo]['title']}")
print(f"   numero_outfit: {capo_a_outfit[un_capo]['numero_outfit']}")
print(f"   primo outfit: {capo_a_outfit[un_capo]['outfit'][0]}")

if SCRIVI_SU_DISCO:
    out_path = DATI / "capo_a_outfit.json"
    out_path.write_text(json.dumps(capo_a_outfit, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\nscritto {out_path.relative_to(RADICE)} ({out_path.stat().st_size/1e6:.1f} MB)")
else:
    print("\nSCRIVI_SU_DISCO = False: capo_a_outfit resta solo in memoria in questa sessione.")


capi indicizzati: 2711
esempio — abbigliamento-donna/24colours-short-60969a-rust-186:
   title: PANTALONCINO SHORT 24COLOURS RUST
   numero_outfit: 29
   primo outfit: {'outfit_id': '0093c322d085', 'gender': 'donna', 'outfit_score': 0.891, 'slot': 'bottom', 'altri_capi': [{'slot': 'top', 'relpath': 'abbigliamento-donna/t-shirt/t-shirt-donna-levis-perfect-tee-caviar', 'title': "T-SHIRT DONNA LEVI'S PERFECT TEE CAVIAR"}, {'slot': 'shoes', 'relpath': 'abbigliamento/scarpe/ciabatte/donna-goldenglow-toggle-1179370-black-blk', 'title': 'SANDALO DONNA UGG GOLDENGLOW TOGGLE BLACK'}, {'slot': 'accessory', 'relpath': 'abbigliamento-donna/accessori-donna/gaelle-paris-borsa-gaacw03183-beige-be00', 'title': 'BORSA DONNA GAELLE PARIS BEIGE'}]}

SCRIVI_SU_DISCO = False: capo_a_outfit resta solo in memoria in questa sessione.


---
# 8. Riepilogo — ogni funzione in una riga

La stessa cosa detta in tabella: che cosa entra, che cosa esce, in che ordine.

In [76]:
riepilogo = [
 ("1 sito",        "slugify / get_session",     "stringa · niente",          "slug · sessione requests"),
 ("1 sito",        "discover_brand_list",       "sessione",                  "elenco marchi del sito"),
 ("1 sito",        "match_brand",               "titolo, url",               "(marchio, slug)"),
 ("1 sito",        "discover_category_seeds",   "sessione",                  "categorie di partenza"),
 ("1 sito",        "crawl_categories",          "sessione",                  "url prodotto → cartella"),
 ("1 sito",        "fetch_product_page",        "sessione, url",             "(DOM, errore)"),
 ("1 sito",        "extract_gallery_images_from_soup", "DOM",                "url delle foto vere"),
 ("1 sito",        "extract_metadata_from_soup","DOM",                       "scheda grezza del prodotto"),
 ("1 sito",        "download_image",            "url foto, cartella",        "il file su disco"),
 ("1 sito",        "estrai_descrizione_completa","DOM",                      "descrizione lunga + scheda tecnica"),
 ("1 sito",        "run_scrape_with_attributes","dove scrivere",             "il catalogo: foto + metadata.json"),
 ("1 sito",        "refresh_metadata",          "catalogo",                  "metadata riscritti (senza foto)"),
 ("1 sito",        "refix_brand_and_attributes","catalogo",                  "marchio e attributi corretti, senza rete"),
 ("2 valutazione", "compute_style_scores",      "metadata.json",             "punteggi grezzi + tag con riscontro"),
 ("2 valutazione", "derive_attributes",         "metadata.json",             "attributi testuali + needs_vision_review"),
 ("2 valutazione", "season_da_tipo / formalita_da_tipo", "titolo",           "stagione · formalità 1-5"),
 ("2 valutazione", "testo_scheda",              "metadata.json",             "la scheda impaginata per il modello"),
 ("2 valutazione", "foto_ridotta",              "cartella del prodotto",     "JPEG 640 px da allegare"),
 ("2 valutazione", "revisiona",                 "client, metadata, cartella","(giudizio validato, errore)"),
 ("2 valutazione", "_primo_oggetto_json",       "risposta grezza",           "JSON bilanciato e parsabile"),
 ("2 valutazione", "valida_risposta",           "JSON del modello",          "blocco vision_review pulito"),
 ("2 valutazione", "main (vision_review.py)",   "catalogo + argomenti",      "metadata.json con vision_review"),
 ("3 vettori",     "pick_representative_image", "lista di foto",             "la foto con più sfondo pulito"),
 ("3 vettori",     "dominant_colors_lab",       "una foto",                  "palette Lab + proporzioni"),
 ("3 vettori",     "needs_review_effettivo",    "metadata.json",             "capo abbinabile sì/no"),
 ("3 vettori",     "formalita_effettiva",       "metadata.json",             "formalità 1-5"),
 ("3 vettori",     "mistura_di_stile",          "metadata.json",             "ripartizione sui 5 registri"),
 ("3 vettori",     "fantasia_effettiva",        "metadata.json",             "pattern del capo"),
 ("3 vettori",     "build_style_vector",        "metadata.json",             "10 colonne: stile+formalità+stagione"),
 ("3 vettori",     "build_feature_table",       "catalogo",                  "features.parquet"),
 ("4 cluster",     "clustering_feature_columns","flag stagione",             "colonne su cui clusterizzare"),
 ("4 cluster",     "run_clustering",            "features.parquet",          "features_clustered.parquet"),
 ("5 punteggi",    "hue_circular_distance",     "due tonalità in gradi",     "distanza 0-180"),
 ("5 punteggi",    "color_harmony_pair",        "due colori Lab",            "armonia 0-1 di quella coppia"),
 ("5 punteggi",    "stono_da_sottotono",        "due colori Lab",            "quanto è il caso nero+blu notte"),
 ("5 punteggi",    "stono_nero_su_blu",         "due colori Lab",            "quanto è nero addosso a blu"),
 ("5 punteggi",    "color_harmony",             "due palette",               "armonia di colore fra due capi"),
 ("5 punteggi",    "coseno_pesato",             "due vettori di stile",      "affinità di registro 0-1"),
 ("5 punteggi",    "style_match",               "vettori + formalità",       "affinità di stile 0-1"),
 ("5 punteggi",    "stono_da_fantasie",         "due pattern",               "conflitto di fantasie 0-1"),
 ("5 punteggi",    "score_pair",                "due righe del parquet",     "componenti + score(A,B)"),
 ("6 accoppiam.",  "classify_slot",             "titolo",                    "slot del capo"),
 ("6 accoppiam.",  "classify_sleeve/leg/gender","titolo, percorso",          "manica, gamba, genere"),
 ("6 accoppiam.",  "load_and_prepare",          "features_clustered.parquet","catalogo pronto per la fase 6"),
 ("6 accoppiam.",  "_formality_ok",             "parziale + candidato",      "dispersione ≤ 0,3 sì/no"),
 ("6 accoppiam.",  "_seasonal_coherence_ok",    "parziale + candidato",      "Regola 1 sì/no"),
 ("6 accoppiam.",  "candidates_for_slot",       "slot, ancora, parziale",    "capi ammissibili"),
 ("6 accoppiam.",  "build_outfits",             "ancora",                    "proposte con punteggi di coppia"),
 ("6 accoppiam.",  "valida_composizione",       "outfit già composto",       "(punteggio, motivo del rifiuto)"),
 ("7 pool",        "compatibilita_media",       "capi",                      "versatilità media, per ordinare le ancore"),
 ("7 pool",        "IndiceNovita.e_nuovo",      "firma dell'outfit",         "abbastanza diverso sì/no"),
 ("7 pool",        "build_outfit_id",           "insieme dei capi",          "id stabile"),
 ("7 pool",        "run_outfit_pipeline",       "catalogo",                  "outfits_pool.jsonl"),
]
pd.set_option("display.max_colwidth", 46)
print(pd.DataFrame(riepilogo, columns=["fase", "funzione", "input", "output"]).to_string(index=False))

         fase                           funzione                      input                                    output
       1 sito              slugify / get_session           stringa · niente                  slug · sessione requests
       1 sito                discover_brand_list                   sessione                    elenco marchi del sito
       1 sito                        match_brand                titolo, url                           (marchio, slug)
       1 sito            discover_category_seeds                   sessione                     categorie di partenza
       1 sito                   crawl_categories                   sessione                   url prodotto → cartella
       1 sito                 fetch_product_page              sessione, url                             (DOM, errore)
       1 sito   extract_gallery_images_from_soup                        DOM                       url delle foto vere
       1 sito         extract_metadata_from_soup        